# Notebook 08 — WWF RGB orthomosaics | Musanga V3_1

**Primary objective:** evaluate **native RGB quality and resolution**, and select orthomosaics for **new exhaustive Musanga annotations** across a broad forest regeneration/disturbance gradient. This is **not** a cross-validation experiment with the old UAV sites.

**Recommended workflow:** 1–3 setup / RGB verification → 4 TMF → 5 Hansen → 6 ecological strata → 7 sampling → **9 final PDF report**. Section 8 preserves optional historical pilot work and is not required for the final analysis.

**Cached results:** Sections 4–7 have already written their tables, spatial layers and circle candidates to Drive. In a fresh runtime you do **not** need to rerun them to generate the report. Run **1.B** to authenticate GCS and mount Drive (and 1.A only if packages are missing), then **9.A**. If Google Cloud remains authenticated and Drive is mounted, **9.A alone** is sufficient.

**Workflow:** A Setup → B GCS inventory → C survey map → D RGB native-resolution audit → E small-window visual QC → F diverse annotation site screening.

**Confirmed pilot:** `UTM33S_Plot135_RGB.tif` is EPSG:32733, **10 cm GSD**, 3 bands, 13 × 3 km and **has no overviews**. A 650-pixel preview of the *entire raster* may force huge reads. **Inspect bounded native-pixel crops instead.**

All GCS operations are read-only. No large mosaic downloads, new annotation labels or modifications to existing V3_1 model data.

## 1 — Environment and authentication
Install dependencies and connect Colab to Google Drive / GCS. Earth Engine is optional for the RGB-first objective.

In [ ]:
# 1.A — Install dependencies
%pip -q install google-cloud-storage geopandas shapely pyogrio rasterio folium earthengine-api geemap fsspec gcsfs

In [ ]:
# 1.B — Authenticate GCS, mount Drive and initialize Earth Engine
from google.colab import auth, drive
from pathlib import Path
import os
auth.authenticate_user()
drive.mount('/content/drive', force_remount=False)
# The registered project used in the existing Musanga satellite notebooks.
EE_PROJECT = (os.environ.get('EE_PROJECT') or 'musanga-511110').strip()
import ee
def initialize_earth_engine():
    """Initialize with Musanga cloud project; authenticate interactively if necessary."""
    try:
        ee.Initialize(project=EE_PROJECT)
    except Exception as first_error:
        print('Earth Engine needs authentication:', str(first_error)[:350])
        ee.Authenticate()
        ee.Initialize(project=EE_PROJECT)
    # Lightweight server read verifies the project actually works.
    print('Earth Engine connected:', EE_PROJECT,
          '| server check:', ee.Number(1).getInfo())
    return True
EE_READY=False
try:
    EE_READY=initialize_earth_engine()
except Exception as err:
    print('Earth Engine setup did not complete:',repr(err))
    print('The GCS inventory and RGB tests work without EE. For TMF, fix the project or authenticate before H2.')
MUSANGA_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Musanga')
WORK_DIR = MUSANGA_ROOT/'V3_1'/'NB08_WWF_Lidar'
WORK_DIR.mkdir(parents=True,exist_ok=True)
print('Outputs:',WORK_DIR)

## 2 — WWF inventory and survey geography
Inspect bucket object sizes, filenames and matched per-plot RGB, LiDAR and CHM coverage. Metadata only.

In [ ]:
# 2.A — Inventory WWF GCS products
from google.cloud import storage
import pandas as pd, re
from pathlib import PurePosixPath
BUCKET='carbon-maps'
client=storage.Client()
rows=[]
for b in client.list_blobs(BUCKET):
    if b.name.endswith('/'): continue
    m=re.search(r'(UTM\d{1,2}[NS])[_/-]?(?:Plot|plot)[_-]?(\d+)',b.name,re.I)
    plot_id=f'{m.group(1).upper()}_Plot{int(m.group(2))}' if m else None
    rows.append(dict(path=b.name,extension=PurePosixPath(b.name).suffix.lower(),size_GB=(b.size or 0)/1e9,plot_id=plot_id,updated=str(b.updated)))
df=pd.DataFrame(rows)
df.to_csv(WORK_DIR/'carbon_maps_inventory.csv',index=False)
print('Files:',len(df),'GB:',round(df.size_GB.sum(),1))
display(df.groupby('extension').agg(files=('path','size'),GB=('size_GB','sum')).sort_values('GB',ascending=False).head(20))

In [ ]:
# 2.A.1 — Match RGB, LiDAR and CHM products
patterns={
 'RGB':r'^001_Lidar_Rasters/RGB_mosaic/.*_RGB\.tif$',
 'CHM':r'^001_Lidar_Rasters/Lidar_2meter/CHM/.*\.tif$',
 'DTM':r'^001_Lidar_Rasters/Lidar_2meter/DTM/.*\.tif$',
 'DSM':r'^001_Lidar_Rasters/Lidar_2meter/DSM/.*\.tif$',
 'LAS':r'^00-raw-data/Plots/.*\.las$',
 'ECW':r'^00-raw-data/Plots/.*\.ecw$'}
products=pd.DataFrame({'plot_id':sorted(df.plot_id.dropna().unique())})
for kind,pattern in patterns.items():
    counts=df.loc[df.path.str.contains(pattern,case=False,regex=True,na=False)].groupby('plot_id').size()
    products[kind+'_n']=products.plot_id.map(counts).fillna(0).astype(int)
display(products.head(20))
print('Plots with RGB:',(products.RGB_n>0).sum())
products.to_csv(WORK_DIR/'wwf_plot_product_inventory.csv',index=False)

### 2.B — WWF survey footprints
Load the 216 georeferenced plot footprints and display a reliable offline-first survey map. Esri satellite is optional; OSM 403 errors are avoided.

In [ ]:
# 2.B.1 — Load georeferenced WWF survey polygons
import tempfile
import geopandas as gpd
from pathlib import Path
prefix='003_GIS_Layers/Lidar_Plots/'
local=Path(tempfile.mkdtemp(prefix='wwf_gis_'))
for b in client.list_blobs(BUCKET,prefix=prefix):
    if b.name.lower().endswith(('.shp','.shx','.dbf','.prj','.cpg','.qpj')):
        b.download_to_filename(str(local/Path(b.name).name))
shps=list(local.glob('*.shp'))
assert shps, 'No plot shapefile found'
plots=gpd.read_file(shps[0]).to_crs('EPSG:4326')
plots=plots[plots.geometry.notna() & ~plots.geometry.is_empty].copy()
plots['geometry']=plots.geometry.make_valid()
plots['plot_id']=plots.Name.astype(str).str.strip()
plots['utm_zone']=plots.plot_id.str.extract(r'^(UTM\d{1,2}[NS])',expand=False)
plots=plots.merge(products,on='plot_id',how='left',validate='one_to_one')
print('Features:',len(plots),'Unique plots:',plots.plot_id.nunique())
display(plots.groupby('utm_zone').agg(plots=('plot_id','size'),rgb=('RGB_n',lambda x:(x>0).sum())))
plots.to_file(WORK_DIR/'wwf_lidar_plots.gpkg',driver='GPKG')

In [ ]:
# 2.B.2 — Map all WWF strips
# The previous OSM basemap returned HTTP 403 within Colab.
# Esri satellite is optional; the default offline map always works.
import folium
from folium.plugins import Fullscreen, MiniMap
from IPython.display import display
from html import escape
import pandas as pd

# tiles=None ensures ZERO requests to restricted OSM tile servers.
m = folium.Map(location=[-2.5, 23.5], zoom_start=6, tiles=None,
               control_scale=True, prefer_canvas=True)

# Reliable fallback: blank geographic background (no external tile dependency).
folium.TileLayer(
    tiles='data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mP8/x8AAusB9Y9UxQAAAABJRU5ErkJggg==',
    attr='Local blank geographic background', name='Offline (always works)',
    overlay=False, show=True, control=True, max_zoom=19
).add_to(m)

# Optional imagery; switch it on only if your Colab browser can access Esri.
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles © Esri', name='Esri World Imagery (optional)',
    overlay=False, show=False, control=True, max_zoom=19
).add_to(m)

colors={'UTM33S':'#9467bd','UTM34S':'#e3771c','UTM34N':'#2ca02c',
        'UTM35S':'#d62728','UTM35N':'#1f77b4','UTM36N':'#8c564b'}

# Plot geometry gives survey context even if ALL basemap servers fail.
foot=folium.FeatureGroup(name='Survey footprints',show=True)
folium.GeoJson(
    plots[['plot_id','geometry']].to_json(),
    style_function=lambda f: {'color':'#555555','weight':1.5,'fillOpacity':0.08},
    tooltip=folium.GeoJsonTooltip(fields=['plot_id'])
).add_to(foot)
foot.add_to(m)

for zone,group in plots.groupby('utm_zone'):
    layer=folium.FeatureGroup(name=f'{zone}: {len(group)} plots',show=True)
    for _,r in group.iterrows():
        pt=r.geometry.representative_point()
        counts={k:int(r[k+'_n']) if pd.notna(r[k+'_n']) else 0
                for k in ('RGB','LAS','CHM','DTM','DSM')}
        label=escape(str(r.plot_id))
        html='<b>'+label+'</b><br>'+'<br>'.join(f'{k}: {v}' for k,v in counts.items())
        col=colors.get(zone,'gray')
        folium.CircleMarker(
            [pt.y,pt.x],radius=5,color=col,weight=2,fill=True,
            fill_color=col if counts['RGB'] else '#ffffff',fill_opacity=.9,
            tooltip=label,popup=folium.Popup(html,max_width=300)
        ).add_to(layer)
    layer.add_to(m)

# Geographic graticule: geographic reference even with the offline background.
graticule=folium.FeatureGroup(name='Latitude / longitude grid',show=True)
for latitude in range(-12,7,2):
    folium.PolyLine([[latitude,12],[latitude,32]],color='#a0a0a0',
                    weight=.6,opacity=.4,tooltip=f'{latitude}° latitude').add_to(graticule)
for longitude in range(12,33,2):
    folium.PolyLine([[-13,longitude],[7,longitude]],color='#a0a0a0',
                    weight=.6,opacity=.4,tooltip=f'{longitude}° longitude').add_to(graticule)
graticule.add_to(m)

w,s,e,n=plots.total_bounds
m.fit_bounds([[s,w],[n,e]])
Fullscreen().add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

output=WORK_DIR/'wwf_lidar_survey_map.html'
m.save(str(output))
print('Offline-capable map:',output)
print('To see satellite imagery: select Esri World Imagery in the layer control.')
display(m)

#### 2.C — Optional UAV ROI context
Their spatial overlap is **not** required. Run this cell only if you want to see the original UAV sites on the map.

In [ ]:
# 2.C.1 — Overlay earlier UAV ROIs (optional)
# Source of truth: V3_1 audited manifest and its ROI.shp (no guessed coordinates).
from pathlib import Path
from shapely.geometry import box
import geopandas as gpd
import rasterio
import folium
import pandas as pd
from IPython.display import display
import json
from html import escape

drive_root=Path('/content/drive/MyDrive')
final_root=next((q for q in [
    drive_root/'Colab Notebooks/Musanga/Final',
    drive_root/'Musanga/Final'] if q.is_dir()), None)
assert final_root, 'Musanga/Final not found. Mount Drive and check V3_1 paths.'
manifest_file=final_root/'Outputs/V3_1/00_input_audit/00_input_manifest.json'
assert manifest_file.exists(), (
    'Approved V3_1 input manifest missing: '+str(manifest_file)
    +'. Run V3_1 Notebook 00 / Block 10 first.')
manifest=json.loads(manifest_file.read_text())
roi_file=Path(manifest['roi'])
assert roi_file.is_file(), f'ROI shapefile missing: {roi_file}'
uav_roi=gpd.read_file(roi_file)
assert uav_roi.crs is not None, 'ROI.shp has no CRS; refusing to guess.'
uav_roi=uav_roi[uav_roi.geometry.notna() & ~uav_roi.geometry.is_empty].copy()
uav_roi.geometry=uav_roi.geometry.make_valid()
uav_roi=uav_roi[uav_roi.geometry.notna() & ~uav_roi.geometry.is_empty].copy()
assert len(uav_roi), 'No valid UAV ROI geometries.'

# Assign ROI polygons to a site by physical overlap with real mosaic footprints.
# Intersections are measured in the ROI's native metric CRS, not lat/lon degrees.
mosaic_records=[]
for entry in manifest['mosaics']:
    image_file=Path(entry['path'])
    if not image_file.exists():
        print('Mosaic missing; cannot assign this footprint:',image_file)
        continue
    with rasterio.open(image_file) as ds:
        assert ds.crs is not None, f'No CRS: {image_file}'
        g=gpd.GeoSeries([box(*ds.bounds)],crs=ds.crs).to_crs(uav_roi.crs).iloc[0]
        mosaic_records.append(dict(site=str(entry['site']),file=str(entry['file']),geometry=g))
assert mosaic_records, 'No readable UAV mosaics for ROI attribution.'

# Aggregate overlapping mosaics by forest site, avoiding duplicate MOS geometry.
from shapely.ops import unary_union
site_footprints={
    site:unary_union([x['geometry'] for x in mosaic_records if x['site']==site])
    for site in sorted(set(x['site'] for x in mosaic_records))}
roi_rows=[]
for index,row in uav_roi.iterrows():
    g=row.geometry
    overlaps={site:float(g.intersection(foot).area)
              for site,foot in site_footprints.items()}
    best=max(overlaps,key=overlaps.get)
    if overlaps[best]<=0:
        best='Unassigned'
    roi_rows.append(best)
uav_roi['site']=roi_rows
uav_roi['roi_row']=[int(i) for i in range(len(uav_roi))]
print('UAV ROI features:',len(uav_roi))
print('Site assignment:',uav_roi.site.value_counts().to_dict())
if (uav_roi.site=='Unassigned').any():
    print('WARNING: Unassigned ROI polygons are shown separately, not silently removed.')

roi_wgs=uav_roi.to_crs('EPSG:4326')
roi_colors={'Baego':'#ef4444','MOS':'#f59e0b','Yangambi':'#10b981',
            'Yoko':'#ec4899','Unassigned':'#111827'}
for site, group in roi_wgs.groupby('site'):
    layer=folium.FeatureGroup(name=f'UAV ROI — {site} ({len(group)})',show=True)
    color=roi_colors.get(site,'#111827')
    folium.GeoJson(
        group[['site','roi_row','geometry']].to_json(),
        style_function=lambda feat,c=color:{
            'color':c,'weight':3.5,'fillColor':c,'fillOpacity':.16},
        highlight_function=lambda feat:{'weight':5,'fillOpacity':.3},
        tooltip=folium.GeoJsonTooltip(fields=['site','roi_row'],
            aliases=['Musanga site','ROI row'])
    ).add_to(layer)
    # Distinct labels at true ROI interiors
    for _,rr in group.iterrows():
        pt=rr.geometry.representative_point()
        folium.Marker([pt.y,pt.x],
            icon=folium.DivIcon(html=(
                '<div style="font: bold 11px sans-serif;white-space:nowrap;'
                'text-shadow:1px 1px 2px white,-1px -1px 2px white;color:'
                +color+';">'+escape(site)+'</div>'))
        ).add_to(layer)
    layer.add_to(m)

# Drop old layer controls before adding the revised one.
for key, child in list(m._children.items()):
    if isinstance(child,folium.map.LayerControl):
        del m._children[key]
folium.LayerControl(collapsed=False).add_to(m)

# Plot boundaries remain visible against the complete WWF footprint.
allbounds=pd.concat([
    pd.Series([plots.total_bounds[0],roi_wgs.total_bounds[0]]),
    pd.Series([plots.total_bounds[1],roi_wgs.total_bounds[1]]),
    pd.Series([plots.total_bounds[2],roi_wgs.total_bounds[2]]),
    pd.Series([plots.total_bounds[3],roi_wgs.total_bounds[3]])
],axis=1)
m.fit_bounds([[min(plots.total_bounds[1],roi_wgs.total_bounds[1]),
               min(plots.total_bounds[0],roi_wgs.total_bounds[0])],
              [max(plots.total_bounds[3],roi_wgs.total_bounds[3]),
               max(plots.total_bounds[2],roi_wgs.total_bounds[2])]])

uav_roi.to_file(WORK_DIR/'musanga_v3_1_roi_wwf_overlay.gpkg',driver='GPKG')
map_path=WORK_DIR/'wwf_lidar_plus_musanga_uav_roi.html'
m.save(str(map_path))
print('Saved map:',map_path)
print('Saved georeferenced ROI:',WORK_DIR/'musanga_v3_1_roi_wwf_overlay.gpkg')
display(m)

## 3 — RGB image quality and feasibility
Measure pixel size **from raster transforms**, not nominal filenames. Report footprint dimensions, band count, CRS, overviews, block layout and color interpretation. Avoid full-resolution reads.

In [ ]:
# 3.A — Read one RGB mosaic header
import rasterio
import pandas as pd
from rasterio.crs import CRS
from pyproj import Geod
sample_path='001_Lidar_Rasters/RGB_mosaic/UTM33S_Plot135_RGB.tif'
uri='/vsigs/'+BUCKET+'/'+sample_path
with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR', CPL_VSIL_CURL_ALLOWED_EXTENSIONS='.tif'):
    with rasterio.open(uri) as src:
        print('File:',sample_path)
        print('CRS:',src.crs)
        print('Width × height:',src.width,'×',src.height)
        print('Bands:',src.count, 'Dtypes:',src.dtypes, 'Color interpretation:',[v.name for v in src.colorinterp])
        print('Pixel size (native CRS units):',src.res)
        print('GSD in cm if CRS projected in metres:',tuple(round(100*abs(v),2) for v in src.res) if src.crs and src.crs.is_projected and src.crs.linear_units=='metre' else 'requires CRS unit conversion')
        print('Extent:',src.bounds)
        print('Internal overviews:',src.overviews(1))
        print('Storage layout:',src.profile.get('tiled'), 'Block shapes:',src.block_shapes)
        print('Compression:',src.profile.get('compress'),'Nodata:',src.nodata)
        print('Pixel count:',f'{src.width*src.height:,}')
print('NOTE: Missing overviews — NEVER downsample entire full-resolution TIFF for an overview.')

### 3.B — RGB header audit across UTM zones
Start with **6 mosaics from different UTM zones** (modifiable), not all 216. Remote metadata access can still incur GCS requests. This saves a QC table with resolution in centimetres and flags missing overviews.

In [ ]:
# 3.B.1 — Sample native GSD across UTM zones
import math, pandas as pd, rasterio
MAX_RGB_METADATA=6
# One example per UTM zone; choose more later when working
rgb_paths=(df[df.path.str.fullmatch(r'001_Lidar_Rasters/RGB_mosaic/.*_RGB\.tif',case=False,na=False)]
           .copy())
rgb_paths['utm']=rgb_paths.plot_id.str.extract(r'^(UTM\d{1,2}[NS])',expand=False)
pick=rgb_paths.groupby('utm',sort=True).head(1).head(MAX_RGB_METADATA)
metadata_rows=[]
for rec in pick.itertuples(index=False):
    row=dict(plot_id=rec.plot_id,path=rec.path,size_GB=rec.size_GB)
    try:
        with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
            with rasterio.open('/vsigs/'+BUCKET+'/'+rec.path) as s:
                row.update(crs=str(s.crs),width=s.width,height=s.height,
                           bands=s.count,gsd_x_native=s.res[0],gsd_y_native=s.res[1],
                           overviews=len(s.overviews(1)),tiled=s.profile.get('tiled'),
                           compression=s.profile.get('compress'),
                           block_x=s.block_shapes[0][1],block_y=s.block_shapes[0][0])
                if s.crs and s.crs.is_projected and s.crs.linear_units=='metre':
                    row.update(gsd_cm_x=100*abs(s.res[0]),gsd_cm_y=100*abs(s.res[1]))
                else:
                    row.update(gsd_cm_x=float('nan'),gsd_cm_y=float('nan'))
    except Exception as e:
        row['error']=str(e)[:250]
    metadata_rows.append(row)
meta=pd.DataFrame(metadata_rows)
meta.to_csv(WORK_DIR/'wwf_rgb_metadata_audit.csv',index=False)
display(meta)
print('Saved:',WORK_DIR/'wwf_rgb_metadata_audit.csv')

### 3.C — Native-resolution RGB image inspection
The main annotation decision requires individual tree crown visibility. **Do not use `src.read(out_shape=(3,650,650))` on an entire 13 km mosaic without overviews.** Instead, read **small native-pixel windows** (40 × 40 m at 10 cm = 400 × 400 pixels). Inspect a few different locations within each mosaic; the default is 4 small windows on the first pilot mosaic.

Note: a large striped TIFF can cause some extra data transfer even for a small window; inspect block shapes and monitor network usage.

In [ ]:
# 3.C.1 — Inspect RGB pixel quality
# Important: zero-valued margins are common in orthomosaics, even without nodata metadata.
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.windows import Window
from pathlib import Path
from rasterio.plot import reshape_as_image

QC_PLOT_ID = 'UTM33S_Plot135'
QC_WINDOW_METRES = 40
QC_MAX_WINDOWS = 4
QC_MAX_PROBES = 30  # bounded window reads only; no full-mosaic scan
QC_VALID_THRESHOLD = 0.80  # require at least 80% nonblack pixels
# RGB/BGR ambiguity is diagnosed by displaying both on the first selected patch.
source_rows = df[df.path.str.fullmatch(
    r'001_Lidar_Rasters/RGB_mosaic/' + QC_PLOT_ID + r'_RGB\.tif',
    case=False, na=False)]
assert len(source_rows) == 1, f'Expected exactly 1 RGB mosaic, got {len(source_rows)}'
qc_path = source_rows.iloc[0].path
preview_dir = WORK_DIR / 'RGB_QC_native_windows_v2'
preview_dir.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(8108)

with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
    with rasterio.open('/vsigs/' + BUCKET + '/' + qc_path) as src:
        assert src.crs and src.crs.is_projected and src.crs.linear_units == 'metre'
        assert src.count >= 3
        W = min(src.width, max(1, round(QC_WINDOW_METRES / abs(src.res[0]))))
        H = min(src.height, max(1, round(QC_WINDOW_METRES / abs(src.res[1]))))
        print('Raster:',qc_path)
        print('Native GSD (cm):',100*abs(src.res[0]),100*abs(src.res[1]))
        print('Band colors:',[c.name for c in src.colorinterp])
        print('Bands dtype:',src.dtypes,'Nodata:',src.nodata)
        print('Block shapes:',src.block_shapes,'Overviews:',src.overviews(1))
        print('Scanning up to',QC_MAX_PROBES,'small windows for nonblack coverage...')
        candidates = [(0.5,0.5),(0.35,0.5),(0.65,0.5),
                      (0.5,0.35),(0.5,0.65),
                      (0.25,0.25),(0.75,0.25),(0.25,0.75),(0.75,0.75)]
        candidates += list(map(tuple,rng.uniform(0.1,0.9,(QC_MAX_PROBES,2))))
        keep = []
        checked = []
        for fx, fy in candidates[:QC_MAX_PROBES]:
            col = int(np.clip(round(fx*src.width-W/2),0,src.width-W))
            row = int(np.clip(round(fy*src.height-H/2),0,src.height-H))
            win = Window(col,row,W,H)
            a = src.read(indexes=[1,2,3],window=win,masked=True)
            # A pixel is usable only if all channels are valid AND not all zeros.
            data = np.ma.filled(a,0)
            valid_mask = (~np.ma.getmaskarray(a).any(axis=0)) & np.any(data != 0,axis=0)
            good = float(valid_mask.mean())
            checked.append((col,row,round(good,3)))
            if good >= QC_VALID_THRESHOLD:
                keep.append((win,data,valid_mask,good))
                if len(keep) >= QC_MAX_WINDOWS: break
        print('Window diagnostics (col,row,valid fraction):',checked)
        if not keep:
            print('No windows meet minimum coverage. Could be sparse flight footprint or nodata encoding.')
        for i,(win,data,valid,good) in enumerate(keep,1):
            # Native integer channels: retain relative R/G/B levels (no separate 2-98% per-band stretch).
            img = np.moveaxis(data.astype('float32'),0,-1)
            vals = img[valid]
            if np.issubdtype(np.dtype(src.dtypes[0]),np.integer):
                scale=float(np.iinfo(np.dtype(src.dtypes[0])).max)
                natural=np.clip(img/scale,0,1)
            else:
                # Shared stretch across all channels; preserves relative band colours.
                hi=np.percentile(vals,99) if vals.size else 1
                natural=np.clip(img/max(hi,1e-6),0,1)
            natural[~valid] = 0
            bgr=natural[:,:,::-1]
            if i==1:
                fig,axs=plt.subplots(1,2,figsize=(12,6))
                axs[0].imshow(natural,interpolation='nearest')
                axs[0].set_title('Original band order (1,2,3)')
                axs[1].imshow(bgr,interpolation='nearest')
                axs[1].set_title('Reversed order (3,2,1) — diagnostic only')
                for ax in axs:ax.axis('off')
                plt.show();plt.close(fig)
            fig,ax=plt.subplots(figsize=(7,7))
            ax.imshow(natural,interpolation='nearest')
            ax.set_title(f'{QC_PLOT_ID} | native {100*abs(src.res[0]):g} cm | valid {good:.1%} | window {i}')
            ax.axis('off')
            output=preview_dir/f'{QC_PLOT_ID}_window{i}.png'
            fig.savefig(output,dpi=160,bbox_inches='tight')
            plt.show();plt.close(fig)
            print('Saved:',output,'pixel window:',win)
print('Judge crown delineation ONLY using valid imagery and verified band order.')


### 3.D — Manual visual quality score
Record whether crowns can be delineated, not merely nominal pixel size. Evaluate blur, shadows, mosaicking seams, color balance and the visibility of small pioneer crowns. Save a simple screening table for each inspected sample.

In [ ]:
# 3.D.1 — Prepare manual quality scores
qc_form=pd.DataFrame([{
    'plot_id':'UTM33S_Plot135','gsd_cm':10.0,
    'crown_visibility_1to5':None,'sharpness_1to5':None,
    'shadow_problem_1to5':None,'seam_problem_1to5':None,
    'possible_musanga_visible':'UNCERTAIN',
    'forest_structure':'UNREVIEWED',
    'exhaustive_annotation_feasible':'UNREVIEWED',
    'notes':'Manual review required; no species inference from GSD alone'
}])
qc_form.to_csv(WORK_DIR/'wwf_rgb_visual_qc_template.csv',index=False)
display(qc_form)
print('QC template:',WORK_DIR/'wwf_rgb_visual_qc_template.csv')

### 3.E — Six-strip RGB feasibility atlas
Test a deliberately small sample across UTM zones. For each mosaic, inspect up to three **35 × 35 m native-resolution crops** (with at most 12 spatial probes). Export both original band order and reversed order, because the blue/yellow colours might reflect BGR interpretation. Results include a contact sheet, individual PNGs and a CSV containing exact raster-window bounds. This is a **visual screening step**, not a validated Musanga identification or forest age label. Some mosaics may have no valid crop among the few sampled positions: record that explicitly, do not label them poor-quality. The first run makes at most 72 small window reads; large striped TIFFs may still incur extra I/O.

In [ ]:
# 3.E.1 — Create native RGB feasibility atlas
# Output: 6 plots × 3 examples, individual PNGs, contact sheet, QC table.
# Zero full-mosaic reads and zero full TIFF downloads.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, rasterio
from rasterio.windows import Window, bounds as window_bounds
from pathlib import Path
from PIL import Image
from IPython.display import display
import warnings

ATLAS_PLOTS = [
    'UTM33S_Plot135','UTM34N_Plot105','UTM34S_Plot60',
    'UTM35N_Plot100','UTM35S_Plot79','UTM36N_Plot104'
]
ATLAS_CROP_M = 35                # 350 px if GSD is 10 cm
ATLAS_EXAMPLES_PER_PLOT = 3
ATLAS_MAX_PROBES_PER_PLOT = 12  # cap remote reads if flight footprint has holes
ATLAS_MIN_VALID = 0.70
# NOTE: plot-wide 1-99% stretches are not used; the 8-bit original bands are retained.
atlas_dir = WORK_DIR/'RGB_annotation_feasibility_atlas'
atlas_dir.mkdir(parents=True,exist_ok=True)

def make_display(raw, mask, dtype):
    """Both original and reversed RGB channels; identical scaling in both."""
    array = np.moveaxis(raw.astype(np.float32), 0, -1)
    good = array[mask]
    if np.issubdtype(np.dtype(dtype), np.integer):
        max_value=float(np.iinfo(np.dtype(dtype)).max)
        original=np.clip(array/max_value,0,1)
    else:
        hi=float(np.nanpercentile(good,99.5)) if good.size else 1.
        original=np.clip(array/max(hi,1e-8),0,1)
    original[~mask]=0
    return original, original[:,:,::-1]

rows=[]
images=[]
for plot_id in ATLAS_PLOTS:
    selection=df[df.path.eq(f'001_Lidar_Rasters/RGB_mosaic/{plot_id}_RGB.tif')]
    if len(selection)!=1:
        print('SKIP: missing mosaic',plot_id)
        continue
    uri='/vsigs/'+BUCKET+'/'+selection.iloc[0].path
    print('\nOPEN',plot_id,flush=True)
    try:
        with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
            with rasterio.open(uri) as ds:
                assert ds.crs and ds.crs.is_projected and ds.crs.linear_units=='metre'
                assert ds.count>=3
                nx=min(ds.width,max(1,int(round(ATLAS_CROP_M/abs(ds.res[0])))))
                ny=min(ds.height,max(1,int(round(ATLAS_CROP_M/abs(ds.res[1])))))
                print('GSD (cm):',tuple(round(abs(r)*100,2) for r in ds.res),
                      '| bands:',[c.name for c in ds.colorinterp],
                      '| overviews:',ds.overviews(1),'| blocks:',ds.block_shapes[0],flush=True)
                # Deterministic staggered probes, including central transect;
                # avoids sampling only the corners of a narrow flight swath.
                locations=[(.5,.5),(.25,.5),(.75,.5),(.5,.25),(.5,.75),
                           (.35,.35),(.65,.65),(.35,.65),(.65,.35),
                           (.15,.5),(.85,.5),(.5,.85)]
                n_saved=0
                for k,(fx,fy) in enumerate(locations[:ATLAS_MAX_PROBES_PER_PLOT]):
                    left=int(np.clip(round(fx*ds.width-nx/2),0,ds.width-nx))
                    top=int(np.clip(round(fy*ds.height-ny/2),0,ds.height-ny))
                    win=Window(left,top,nx,ny)
                    ma=ds.read(indexes=[1,2,3],window=win,masked=True)
                    raw=np.ma.filled(ma,0)
                    good=(~np.ma.getmaskarray(ma).any(axis=0)) & np.any(raw!=0,axis=0)
                    fraction=float(good.mean())
                    if fraction<ATLAS_MIN_VALID:
                        continue
                    original,reversed_rgb=make_display(raw,good,ds.dtypes[0])
                    n_saved+=1
                    rgb=(original*255).round().astype('uint8')
                    bgr=(reversed_rgb*255).round().astype('uint8')
                    stem=f'{plot_id}_crop{n_saved:02d}'
                    Image.fromarray(rgb).save(atlas_dir/f'{stem}_bands123.png')
                    Image.fromarray(bgr).save(atlas_dir/f'{stem}_bands321.png')
                    # Actual window bounding box for relocation; in source CRS
                    x0,y0,x1,y1=window_bounds(win,ds.transform)
                    rows.append(dict(plot_id=plot_id,crop=n_saved,probe=k+1,
                         x_min=x0,y_min=y0,x_max=x1,y_max=y1,
                         crs=str(ds.crs),gsd_cm_x=abs(ds.res[0])*100,
                         gsd_cm_y=abs(ds.res[1])*100,
                         valid_fraction=fraction,channel_order='UNVERIFIED',
                         crown_delineation='UNREVIEWED',musanga_identifiable='UNREVIEWED',
                         forest_environment='UNREVIEWED',notes=''))
                    images.append((stem,original,reversed_rgb,fraction))
                    print(f'  crop {n_saved}: probe {k+1}, valid {fraction:.1%}',flush=True)
                    if n_saved>=ATLAS_EXAMPLES_PER_PLOT:break
                if n_saved==0:
                    print('  WARNING: no valid crop among probed positions; not proof of bad imagery')
    except Exception as exc:
        print('ERROR inspecting',plot_id,type(exc).__name__,str(exc)[:400],flush=True)

if rows:
    table=pd.DataFrame(rows)
    table.to_csv(atlas_dir/'atlas_annotation_QC.csv',index=False)
    display(table[['plot_id','crop','gsd_cm_x','valid_fraction','channel_order',
                   'crown_delineation','musanga_identifiable']])
    # Contact sheet at modest display size only; native PNGs are retained.
    fig,axes=plt.subplots(len(images),2,figsize=(9,3.8*len(images)),squeeze=False)
    for j,(stem,a,b,valid) in enumerate(images):
        for col,img in enumerate((a,b)):
            axes[j,col].imshow(img,interpolation='nearest')
            axes[j,col].set_title(f'{stem} | '+('bands 1/2/3' if col==0 else 'bands 3/2/1'),fontsize=9)
            axes[j,col].axis('off')
    fig.suptitle('WWF RGB — native crops, identical channel scaling (colour order unverified)',fontsize=13)
    fig.tight_layout()
    fig.savefig(atlas_dir/'atlas_contact_sheet.jpg',dpi=110,bbox_inches='tight')
    display(fig);plt.close(fig)
    print('Saved contact sheet, individual native crops and QC:',atlas_dir)
else:
    print('No valid crop produced. Check nodata encoding and file layout; do NOT infer that RGB is bad.')


### 3.F — Geographic screening of RGB mosaics
Choose spatially dispersed candidate RGB mosaics as a starting list. **Spatial dispersion is not a proxy for forest age.** Classification into young secondary forest, regrowth, disturbed forest, mature canopy or open land requires image review and external land-cover history.

In [ ]:
# 3.F.1 — Screen candidate mosaics
import numpy as np, pandas as pd
from pathlib import PurePosixPath
rgb=df[df.path.str.fullmatch(r'001_Lidar_Rasters/RGB_mosaic/.*_RGB\.tif',case=False,na=False)].copy()
geo=plots.copy()
cent=geo.geometry.representative_point()
geo['longitude']=cent.x;geo['latitude']=cent.y
catalog=geo[['plot_id','utm_zone','longitude','latitude','geometry']].drop(columns='geometry').merge(
    rgb[['plot_id','path','size_GB']].rename(columns={'path':'rgb_gcs_path','size_GB':'rgb_GB'}),
    how='inner',on='plot_id',validate='one_to_one')
catalog['visual_forest_class']='UNREVIEWED'
catalog['musanga_presence']='UNREVIEWED'
catalog['annotation_suitability']='UNREVIEWED'
catalog['notes']=''
catalog.to_csv(WORK_DIR/'wwf_rgb_annotation_catalog.csv',index=False)
print('RGB candidates:',len(catalog))
display(catalog.groupby('utm_zone').agg(n=('plot_id','size'),GB=('rgb_GB','sum')))
# Screen a geographically dispersed subset across all UTM zones.
# Farthest-point sampling on geographic coordinates within zones; not interpreted as age.
from pyproj import Geod
geod=Geod(ellps='WGS84')
chosen=[]
for zone, group in catalog.groupby('utm_zone'):
    target=min(5,len(group))
    pool=group.reset_index(drop=True)
    selected=[0]
    while len(selected)<target:
        distances=[]
        for j,rr in pool.iterrows():
            if j in selected: distances.append(-1);continue
            mind=min(geod.inv(rr.longitude,rr.latitude,
                              pool.iloc[k].longitude,pool.iloc[k].latitude)[2]
                     for k in selected)
            distances.append(mind)
        selected.append(int(np.argmax(distances)))
    chosen.append(pool.iloc[selected])
screen=pd.concat(chosen,ignore_index=True)
screen['review_priority']='INITIAL_SCREEN'
screen.to_csv(WORK_DIR/'wwf_rgb_initial_screen.csv',index=False)
print('Initial screening set:',len(screen),'(spatially dispersed, NOT preclassified)')
display(screen[['plot_id','utm_zone','latitude','longitude','rgb_GB']])

## 4 — TMF forest-history characterization
Compare native GSD and observed RGB quality across zones; shortlist mosaics with visually annotatable crowns and a balanced gradient of forest environments, including Musanga-free exhaustive negatives. Expand the screening set only after reviewing the pilot true-resolution crops.

### 4.A — TMF status across all 216 survey strips

**Purpose:** characterize the **complete 216-strip survey**, before selecting further 125 m annotation circles. This uses the actual WWF survey footprints, not the six pilot strips, and is fully independent of UAV sites. The JRC TMF 30 m annual classification supplies *landscape-scale* evidence: proportions of undisturbed, degraded, deforested, regrowing, water, and other land cover in 2014 and 2015; fraction of the 2014–2015 landscape that changed class; and a conservative indicator of clearing already present in 2014 following forest in 2011/2012. The acquisition year is only known as 2014–2015, so no later-year conditions enter sampling descriptors.

**Interpretation:** class fractions describe historical TMF conditions **within the mapped transect polygon**, which is not necessarily the exact valid 10 cm RGB coverage. Deforested ≠ bare clearing, and regrowth ≠ known Musanga. A simple diversity index helps flag mixed landscapes; it is not a biological diversity measurement. All output values are computed server-side on Earth Engine, downloaded as small aggregated tables, then ranked and plotted. No large WWF imagery files are downloaded.

This cell can run after **C / cell 04** (once `plots` and `WORK_DIR` exist). It does **not** require H.1, H.2 or annotation circles.

In [ ]:
# 4.A.1 — Calculate TMF 2014–2015 distributions
# 2014/2015 composition, recent clearing, class-change, landscape diversity
import ee, geopandas as gpd, pandas as pd, numpy as np, matplotlib.pyplot as plt
from shapely.geometry import mapping
from pathlib import Path
from IPython.display import display
import math, time

EE_PROJECT = (globals().get('EE_PROJECT') or 'musanga-511110').strip()
try:
    ee.Initialize(project=EE_PROJECT)
    ee.Number(1).getInfo()
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print('Earth Engine:', EE_PROJECT)

assert 'plots' in globals(), 'Run Section C / cell 04 to load WWF footprints'
assert 'plot_id' in plots.columns and plots.plot_id.is_unique
assert plots.crs is not None

TMF_STRIP_DIR = WORK_DIR / 'TMF_strip_descriptors'
TMF_STRIP_DIR.mkdir(parents=True,exist_ok=True)
BATCH_SIZE = 12    # small requests; keeps Colab responsive and allows resume
RECOMPUTE = False  # false: reuse completed batch CSVs on rerun
EE_SCALE_M = 30
LABELS = ['undisturbed','degraded','deforested','regrowth','water','other']
TMF = ee.ImageCollection('projects/JRC/TMF/v1_2025/AnnualChanges').mosaic()
a11, a12, a14, a15 = [TMF.select('Dec'+str(y)) for y in [2011,2012,2014,2015]]
area_ha = ee.Image.pixelArea().divide(10000)
bands = [area_ha.rename('footprint_ha')]
for year, img in [(2014,a14),(2015,a15)]:
    for value, name in enumerate(LABELS, 1):
        bands.append(area_ha.multiply(img.eq(value).unmask(0)).rename(
            f'ha_{year}_{name}'))
# Describes pixels already deforested by 2014 that were forest in 2011 or 2012.
forest_pre = (a11.eq(1).Or(a11.eq(2))
              .Or(a12.eq(1)).Or(a12.eq(2)))
recent_clear = a14.eq(3).And(forest_pre).And(a15.eq(3))
bands += [
    area_ha.multiply(recent_clear.unmask(0)).rename('ha_recent_clear_pre2014'),
    area_ha.multiply(a14.neq(a15).And(a14.mask()).And(a15.mask()).unmask(0))
       .rename('ha_changed_class_2014_2015'),
]
descriptor_image = ee.Image.cat(bands).toFloat()

geo = plots[['plot_id','utm_zone','geometry']].copy().to_crs('EPSG:4326')
geo = geo.loc[geo.geometry.notna() & ~geo.geometry.is_empty].sort_values('plot_id')
# Use compact coordinates; spatial analysis is carried out using the original polygons.
print('Strips:',len(geo), '| EE scale:',EE_SCALE_M,'m')
partial_files = []
for start in range(0,len(geo),BATCH_SIZE):
    batch=geo.iloc[start:start+BATCH_SIZE]
    outfile=TMF_STRIP_DIR/f'tmf_strip_batch_{start:03d}_{start+len(batch):03d}.csv'
    partial_files.append(outfile)
    if outfile.exists() and not RECOMPUTE:
        print('Cached:',outfile.name)
        continue
    features=[]
    for rec in batch.itertuples(index=False):
        features.append(ee.Feature(
            ee.Geometry(mapping(rec.geometry),proj='EPSG:4326',geodesic=False),
            {'plot_id':str(rec.plot_id),'utm_zone':str(rec.utm_zone)}))
    collection=ee.FeatureCollection(features)
    result=descriptor_image.reduceRegions(
        collection=collection,
        reducer=ee.Reducer.sum(),
        scale=EE_SCALE_M,
        tileScale=4
    )
    raw=result.getInfo()['features']
    records=[{k:v for k,v in f['properties'].items()} for f in raw]
    out=pd.DataFrame(records)
    assert len(out)==len(batch),f'Unexpected row count: {len(out)} vs {len(batch)}'
    out.to_csv(outfile,index=False)
    print(f'Calculated {start+1}–{start+len(batch)} / {len(geo)}')

desc=pd.concat([pd.read_csv(p) for p in partial_files],ignore_index=True)
assert len(desc)==len(geo) and desc.plot_id.nunique()==len(geo), (
    'Missing or duplicate WWF strip summaries')
numeric=[c for c in desc if c.startswith('ha_') or c=='footprint_ha']
desc[numeric]=desc[numeric].apply(pd.to_numeric,errors='coerce')
for year in (2014,2015):
    for name in LABELS:
        desc[f'pct_{year}_{name}']=100*desc[f'ha_{year}_{name}']/desc.footprint_ha
    desc[f'pct_{year}_TMF_forest']=(
        desc[f'pct_{year}_undisturbed']+
        desc[f'pct_{year}_degraded']+
        desc[f'pct_{year}_regrowth'])
    desc[f'pct_{year}_disturbed_or_regrowth']=(
        desc[f'pct_{year}_degraded']+
        desc[f'pct_{year}_regrowth'])
desc['pct_recent_clear_pre2014']=100*desc.ha_recent_clear_pre2014/desc.footprint_ha
desc['pct_changed_class_2014_2015']=100*desc.ha_changed_class_2014_2015/desc.footprint_ha
# Normalized Shannon diversity of the 6-class landscape (not biological diversity).
p=desc[[f'pct_2014_{x}' for x in LABELS]].fillna(0).to_numpy()/100.
p=np.clip(p,0,1)
desc['landscape_diversity_2014']=-(np.where(p>0,p*np.log(np.maximum(p,1e-12)),0)
                                   .sum(axis=1))/np.log(len(LABELS))
desc['max_class_share_2014_pct']=100*p.max(axis=1)
# Ranking emphasizes disturbed / regrowing contexts, but retains entire 6-class landscape.
desc['gradient_priority_score']=(
    desc.pct_2014_degraded*0.25+
    desc.pct_2014_regrowth*0.40+
    desc.pct_recent_clear_pre2014*0.20+
    desc.landscape_diversity_2014*100*0.15
)
desc=desc.sort_values('gradient_priority_score',ascending=False).reset_index(drop=True)
desc['tmf_priority_rank']=np.arange(1,len(desc)+1)
out_csv=TMF_STRIP_DIR/'WWF_all_strips_TMF_2014_2015_descriptors.csv'
desc.to_csv(out_csv,index=False)
joined=geo.merge(desc,on=['plot_id','utm_zone'],validate='one_to_one')
out_gpkg=TMF_STRIP_DIR/'WWF_all_strips_TMF_2014_2015_descriptors.gpkg'
joined.to_file(out_gpkg,driver='GPKG')
print('Saved:',out_csv)
print('Saved:',out_gpkg)
important=['plot_id','utm_zone','footprint_ha','pct_2014_undisturbed',
           'pct_2014_degraded','pct_2014_deforested','pct_2014_regrowth',
           'pct_2014_other','pct_recent_clear_pre2014',
           'pct_changed_class_2014_2015','landscape_diversity_2014',
           'tmf_priority_rank']
display(desc[important].head(25).round(2))
print('\nOverall across-strip variability, percentages (not area weighted):')
display(desc[[c for c in important if c.startswith('pct_')]].describe(
    percentiles=[.1,.25,.5,.75,.9]).round(1))
print('\nMean composition by WWF UTM zone:')
display(desc.groupby('utm_zone')[[f'pct_2014_{x}' for x in LABELS]].mean().round(1))
# Readable stacked composition of the top 30 diversified/disturbed strips.
top=desc.head(30).iloc[::-1]
fig,axs=plt.subplots(1,2,figsize=(16,11),gridspec_kw={'width_ratios':[3,2]})
left=np.zeros(len(top))
palette=['#215b45','#c5a642','#de6b56','#80b779','#4d86b0','#bbb9b4']
for label,color in zip(LABELS,palette):
    v=top[f'pct_2014_{label}'].fillna(0).to_numpy()
    axs[0].barh(top.plot_id,v,left=left,label=label,color=color)
    left+=v
axs[0].set_xlim(0,100)
axs[0].set_xlabel('TMF class area within strip (%) — 2014')
axs[0].set_title('Top 30 strips ranked by recovery / disturbance diversity')
axs[0].legend(ncol=3,fontsize=9,loc='lower center',bbox_to_anchor=(.5,-.14))
axs[1].scatter(desc.pct_2014_undisturbed,desc.pct_2014_regrowth,
               s=20+desc.pct_recent_clear_pre2014*2,
               c=desc.pct_2014_degraded,cmap='viridis',vmin=0,
               vmax=max(20,float(desc.pct_2014_degraded.quantile(.95))),
               alpha=.75,edgecolors='none')
axs[1].set(xlabel='Undisturbed forest (%) — 2014',
           ylabel='Regrowth (%) — 2014',
           title='All 216 strips (size = recent clearing)')
axs[1].grid(alpha=.15)
fig.tight_layout()
png=TMF_STRIP_DIR/'WWF_strip_TMF_gradient_overview.png'
fig.savefig(png,dpi=160,bbox_inches='tight')
plt.show()
print('Saved:',png)
print('Note: these are TMF landscape descriptors, not Musanga prevalence or RGB annotation QC.')

### 4.B — Historical degradation and recovery descriptors through 2014

G.1 quantified the six annual classes for all 216 WWF strips, but **forest-age and disturbance history** are more informative for targeting Musanga pioneer stands.

The JRC provides dedicated **first degradation year** and **first deforestation year** layers, as well as a **2025 endpoint transition map**, intensity and duration. **Do not interpret 2025 endpoint transition subtypes/intensity/duration as conditions visible in WWF imagery acquired in 2014–2015.** This block reconstructs **pre-2015** transitions using annual classes through **December 2014**, plus first-event years with a year ≤2014. We characterize:
- share ever first-degraded / first-deforested by 2014; proportions with first events in **2005–2009** and **2010–2014** (and earlier histories);
- consecutive years in the **degraded (class 2)** and **regrowth (class 4)** states as of end-2014, grouped by **1–3, 4–7 and ≥8 years**;
- historical percentages undisturbed, degraded, deforested and regrowth in **1990, 2000, 2005, 2010 and 2014**;
- **annual TMF class transitions** (1990–2014), distinguished from the number of physical disturbance episodes, and a conservative **recovered from deforested** area measure.

The original G.1 CSV is reused; batch output is cached separately. This is a historical *sampling audit* (not a ground-truth age determination): first-event years are NOT necessarily the latest disturbance date; consecutive-state age is NOT necessarily the exact stand age. TMF uses 30 m pixels, and valid RGB footprint/annotation feasibility must still be checked.

Official TMF sources: <https://forobs.jrc.ec.europa.eu/TMF/data> and <https://forobs.jrc.ec.europa.eu/TMF/resources/tutorial/gee>.

In [ ]:
# 4.B.1 — Calculate historical TMF descriptors
# Runs after G.1; reuses its CSV, writes separately cached batches.
import ee, geopandas as gpd, pandas as pd, numpy as np, matplotlib.pyplot as plt
from shapely.geometry import mapping
from pathlib import Path
from IPython.display import display

EE_PROJECT=(globals().get('EE_PROJECT') or 'musanga-511110').strip()
try:
    ee.Initialize(project=EE_PROJECT)
    ee.Number(1).getInfo()
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

ROOT_TMFD=WORK_DIR/'TMF_strip_descriptors'
BASE_CSV=ROOT_TMFD/'WWF_all_strips_TMF_2014_2015_descriptors.csv'
assert BASE_CSV.exists(), 'First run G.1, which computes the initial 216-strip TMF composition.'
base=pd.read_csv(BASE_CSV)
assert len(base)==216 and base.plot_id.is_unique
assert 'plots' in globals(), 'Run Section C to restore WWF strip geometries.'

OUT_DIR=ROOT_TMFD/'historical_2014'
OUT_DIR.mkdir(parents=True,exist_ok=True)
BATCH_SIZE_HIST=8
RECOMPUTE_HIST=False
SCALE_HIST_M=30
JRC='projects/JRC/TMF/v1_2025/'
annual=ee.ImageCollection(JRC+'AnnualChanges').mosaic()
first_deg=ee.ImageCollection(JRC+'DegradationYear').mosaic().select(0)
first_def=ee.ImageCollection(JRC+'DeforestationYear').mosaic().select(0)
def year_img(y):
    return annual.select('Dec'+str(y))
a2014=year_img(2014)
area=ee.Image.pixelArea().divide(10000)
metrics=[]
def area_band(mask,name):
    metrics.append(area.multiply(mask.unmask(0)).rename('ha_'+name))
# First event: exclude events AFTER the WWF acquisition.
for layer,short in [(first_deg,'first_degradation'),(first_def,'first_deforestation')]:
    valid=layer.gte(1982).And(layer.lte(2014))
    area_band(valid,short+'_by2014')
    area_band(valid.And(layer.lte(2004)),short+'_through2004')
    area_band(layer.gte(2005).And(layer.lte(2009)),short+'_2005_2009')
    area_band(layer.gte(2010).And(layer.lte(2014)),short+'_2010_2014')
    # Area-weighted mean first-event YEAR (not latest event).
    metrics.append(area.multiply(layer).updateMask(valid).unmask(0)
                   .rename('yeararea_'+short+'_through2014'))
# End-of-2014 current-state consecutive run-lengths: *age in TMF state*, not stand age.
age_reg=ee.Image.constant(0).toInt16()
age_deg=ee.Image.constant(0).toInt16()
transitions=ee.Image.constant(0).toInt16()
prior=year_img(1990)
last_def_pre2014=ee.Image.constant(0).toByte()
for yr in range(1990,2015):
    now=year_img(yr)
    age_reg=age_reg.add(1).multiply(now.eq(4)).toInt16()
    age_deg=age_deg.add(1).multiply(now.eq(2)).toInt16()
    if yr>1990:
        # A change of annual categorical state is not necessarily a distinct disturbance.
        transitions=transitions.add(now.neq(prior)).toInt16()
        last_def_pre2014=last_def_pre2014.Or(prior.eq(3))
    prior=now
for current_age,current_state,short in [(age_reg,4,'regrowth'),(age_deg,2,'degraded')]:
    for lo,hi,label in [(1,3,'1_3y'),(4,7,'4_7y'),(8,25,'8plus_y')]:
        area_band(a2014.eq(current_state).And(current_age.gte(lo))
                  .And(current_age.lte(hi)),short+'_state_age_'+label)
# Pixels in regrowth in 2014 that had at least one year classified deforested before 2014.
area_band(a2014.eq(4).And(last_def_pre2014),
          'regrowth2014_previously_deforested')
for yr in [1990,2000,2005,2010,2014]:
    img=year_img(yr)
    for cls,label in [(1,'undisturbed'),(2,'degraded'),(3,'deforested'),(4,'regrowth')]:
        area_band(img.eq(cls),f'{label}_{yr}')
# Number of 1990–2014 annual state transitions (each categorical transition counts once).
metrics.append(area.multiply(transitions).rename('area_times_transition_count'))
area_band(transitions.gte(1),'any_transition_1990_2014')
area_band(transitions.gte(2),'multiple_transitions_1990_2014')
metric_img=ee.Image.cat(metrics).toFloat()

geo=plots[['plot_id','geometry']].copy().to_crs(4326)
geo=geo[geo.plot_id.isin(base.plot_id)].sort_values('plot_id')
assert len(geo)==216 and geo.plot_id.is_unique
files=[]
for start in range(0,len(geo),BATCH_SIZE_HIST):
    batch=geo.iloc[start:start+BATCH_SIZE_HIST]
    fname=OUT_DIR/f'history_{start:03d}_{start+len(batch):03d}.csv'
    files.append(fname)
    if fname.exists() and not RECOMPUTE_HIST:
        print('Cached',fname.name)
        continue
    features=[ee.Feature(
        ee.Geometry(mapping(rec.geometry),proj='EPSG:4326',geodesic=False),
        {'plot_id':str(rec.plot_id)}) for rec in batch.itertuples(index=False)]
    stats=metric_img.reduceRegions(collection=ee.FeatureCollection(features),
        reducer=ee.Reducer.sum(),scale=SCALE_HIST_M,tileScale=4)
    raw=[f['properties'] for f in stats.getInfo()['features']]
    result=pd.DataFrame(raw)
    assert len(result)==len(batch) and result.plot_id.is_unique
    result.to_csv(fname,index=False)
    print(f'Historical TMF: {start+1}–{start+len(batch)} / {len(geo)}')
hist=pd.concat([pd.read_csv(f) for f in files],ignore_index=True)
assert len(hist)==216 and hist.plot_id.is_unique
merged=base.merge(hist,on='plot_id',how='left',validate='one_to_one')
assert len(merged)==216 and merged['ha_first_deforestation_by2014'].notna().all()
# Convert area metrics to fraction of full survey footprint, not fraction of forest.
for key in hist.columns:
    if key.startswith('ha_'):
        merged['pct_'+key[3:]]=100*pd.to_numeric(merged[key])/merged.footprint_ha
for label in ['first_degradation','first_deforestation']:
    merged['mean_'+label+'_year_by2014']=np.where(
        merged['ha_'+label+'_by2014']>0,
        merged['yeararea_'+label+'_through2014']/merged['ha_'+label+'_by2014'],
        np.nan)
merged['mean_class_transitions_1990_2014']=(
    merged.area_times_transition_count/merged.footprint_ha)
# Conditional proportions within existing degraded or regrowth 2014 classes:
for cat in ['degraded','regrowth']:
    denom=merged[f'ha_{cat}_2014'].replace(0,np.nan)
    for age in ['1_3y','4_7y','8plus_y']:
        key=f'ha_{cat}_state_age_{age}'
        merged[f'pct_within_{cat}_{age}']=100*merged[key]/denom

merged.to_csv(OUT_DIR/'WWF_all_216_strips_TMF_history_asof2014.csv',index=False)
geo_out=geo.merge(merged,on='plot_id',validate='one_to_one')
geo_out.to_file(OUT_DIR/'WWF_all_216_strips_TMF_history_asof2014.gpkg',driver='GPKG')
cols=['plot_id','utm_zone','pct_2014_undisturbed',
      'pct_2014_degraded','pct_2014_regrowth',
      'pct_first_degradation_2010_2014','pct_first_deforestation_2010_2014',
      'pct_regrowth_state_age_1_3y','pct_regrowth_state_age_4_7y',
      'pct_regrowth_state_age_8plus_y','pct_multiple_transitions_1990_2014']
print('Most first-degraded strips in 2010–2014:')
display(merged.sort_values('pct_first_degradation_2010_2014',
                           ascending=False)[cols].head(20).round(2))
print('Most recently deforested strips (first event 2010–2014):')
display(merged.sort_values('pct_first_deforestation_2010_2014',
                           ascending=False)[cols].head(20).round(2))
print('Strips richest in 2014 recent-state regrowth (1–3 yr):')
display(merged.sort_values('pct_regrowth_state_age_1_3y',
                           ascending=False)[cols].head(20).round(2))
print('Distribution across the 216 WWF strips:')
display(merged[[c for c in cols if c.startswith('pct_')]].describe(
    percentiles=[.1,.25,.5,.75,.9]).round(2))

fig,axes=plt.subplots(1,2,figsize=(14,5),constrained_layout=True)
axes[0].scatter(merged.pct_first_degradation_2010_2014,
                merged.pct_first_deforestation_2010_2014,
                s=20+3*merged.pct_2014_regrowth,
                alpha=.7)
axes[0].set(xlabel='First degradation in 2010–2014 (% of strip)',
            ylabel='First deforestation in 2010–2014 (% of strip)',
            title='Historical first-event gradient')
axes[1].scatter(merged.pct_2014_regrowth,
                merged.pct_regrowth_state_age_1_3y,
                s=25,alpha=.7)
axes[1].set(xlabel='All regrowth in 2014 (% of strip)',
            ylabel='Regrowth in 1–3 year consecutive state (% of strip)',
            title='Young TMF regrowth vs total regrowth')
for ax in axes:ax.grid(alpha=.15)
fig_path=OUT_DIR/'WWF_TM F_historical_gradient_2014.png'.replace(' ','')
fig.savefig(fig_path,dpi=160)
plt.show()
print('Saved history to',OUT_DIR)
print('Caution: 2025 TransitionMap_Subtypes / duration / intensity are not 2014 classes.')


### 4.C — Full TMF pixel distributions within potential RGB footprints

This is the spatial audit **before selecting any circles**. For each WWF RGB GeoTIFF, read **only its header**, intersect its georeferenced raster bounds with the corresponding LiDAR survey polygon, and summarize **every TMF 30 m pixel** within that intersection (not a random sample). There are **216** available RGB mosaics, rather than 226.

**Footprint caveat:** an intersection between the raster rectangle and survey polygon is only the *potential* RGB-covered footprint. No-data/black swaths remain possible. Computing an *exact* native RGB-valid mask for all mosaics requires reading many GB of native RGB pixels because the GeoTIFFs lack overviews. Before final annotation-site selection, check true image validity in promising locations.

**TMF outputs:** full per-strip **frequency histograms** for 2014/2015 status, first degradation/deforestation year up through 2014, end-2014 consecutive degraded/regrowth-state age, and number of annual class transitions through 2014. Plus annual 1990/2000/2005/2010 composition. A **separately labeled 2025 endpoint** histogram covers the JRC transition subtype map: useful retrospectively, **not** a 2014 forest condition. Percentage profiles, CSVs and GeoPackage are created, plus a map with *actual 30 m TMF pixels* (not just coloured strip aggregates). The raster layers are shown **only inside the candidate RGB footprints**.

**No circles are generated here.** Use the resulting spatial map to choose a sampling frame, then confirm native RGB coverage.

In [ ]:
# 4.C.1 — Audit pixel-level TMF in RGB footprints
# Run after Sections A–C / 'plots' + 'BUCKET' + 'WORK_DIR' exist.
import re, json, rasterio, numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from pathlib import Path
from shapely.geometry import box, mapping
from rasterio.warp import transform_bounds
from IPython.display import display
import ee

EE_PROJECT=(globals().get('EE_PROJECT') or 'musanga-511110').strip()
try:
    ee.Initialize(project=EE_PROJECT)
    ee.Number(1).getInfo()
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

G3_DIR=WORK_DIR/'TMF_strip_descriptors'/'rgb_footprint_pixels'
G3_DIR.mkdir(parents=True,exist_ok=True)
REBUILD_RGB_FOOTPRINTS=False
RECOMPUTE_TMF_HIST=False
TMF_BATCH=8
TMF_SCALE=30
ADD_2025_ENDPOINT=True  # separate retrospective descriptor: NEVER a 2014 label

# -------------------------------------------------------------------------
# 1. Header-only actual TIFF bounds intersected with each LiDAR plot polygon
# -------------------------------------------------------------------------
foot_gpkg=G3_DIR/'WWF_RGB_potential_footprints.gpkg'
if foot_gpkg.exists() and not REBUILD_RGB_FOOTPRINTS:
    rgb_foot=gpd.read_file(foot_gpkg).to_crs(4326)
    print('Reused cached TIFF bounds × plot polygons:',len(rgb_foot))
else:
    assert 'plots' in globals() and 'BUCKET' in globals()
    source=plots.to_crs(4326).copy()
    assert source.plot_id.is_unique
    records=[]
    for k,row in enumerate(source.itertuples(index=False),1):
        plot_id=str(row.plot_id)
        uri=f'/vsigs/{BUCKET}/001_Lidar_Rasters/RGB_mosaic/{plot_id}_RGB.tif'
        try:
            with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
                with rasterio.open(uri) as src:
                    bb=transform_bounds(src.crs,'EPSG:4326',*src.bounds,densify_pts=21)
                    nominal_gsd_cm=100*abs(src.res[0])
            overlap=row.geometry.intersection(box(*bb))
            if overlap.is_empty or overlap.area==0:
                print('NO OVERLAP:',plot_id)
                continue
            records.append(dict(plot_id=plot_id,utm_zone=str(row.utm_zone),
                native_gsd_cm=nominal_gsd_cm,footprint_method='survey_polygon_intersect_rgb_bounds',
                valid_rgb_checked=False,geometry=overlap))
        except Exception as e:
            print('FAILED RGB HEADER:',plot_id,str(e)[:200])
        if k%24==0:print('RGB header checked',k,'/',len(source))
    assert records,'No RGB footprint intersections found.'
    rgb_foot=gpd.GeoDataFrame(records,geometry='geometry',crs='EPSG:4326')
    rgb_foot=rgb_foot[rgb_foot.geometry.is_valid & ~rgb_foot.geometry.is_empty]
    rgb_foot.to_file(foot_gpkg,driver='GPKG')
assert rgb_foot.plot_id.is_unique
print('Potential RGB-mosaic footprints:',len(rgb_foot),'/ 216; only these will be processed')
if len(rgb_foot)!=216:print('WARNING: some RGB mosaics lack usable header/footprint intersections')

# -------------------------------------------------------------------------
# 2. Historical TMF pixel bands; no data after 2014 used in historical fields.
# -------------------------------------------------------------------------
jrc='projects/JRC/TMF/v1_2025/'
annual=ee.ImageCollection(jrc+'AnnualChanges').mosaic()
def annual_y(y):return annual.select('Dec'+str(y))
a14=annual_y(2014)
first_deg=ee.ImageCollection(jrc+'DegradationYear').mosaic().select(0)
first_def=ee.ImageCollection(jrc+'DeforestationYear').mosaic().select(0)
# 0 = first event not observed by 2014; also masks any later first events.
def history_year(im):
    return im.where(im.gt(2014),0).unmask(0).toInt16()
age_reg=ee.Image.constant(0).toInt16()
age_deg=ee.Image.constant(0).toInt16()
n_trans=ee.Image.constant(0).toInt16()
prev=None
for year in range(1990,2015):
    curr=annual_y(year)
    age_reg=age_reg.add(1).multiply(curr.eq(4)).toInt16()
    age_deg=age_deg.add(1).multiply(curr.eq(2)).toInt16()
    if prev is not None:
        n_trans=n_trans.add(curr.neq(prev)).toInt16()
    prev=curr
bands=[annual_y(y).unmask(0).toInt16().rename('status_'+str(y))
       for y in [1990,2000,2005,2010,2014,2015]]
bands.extend([
    history_year(first_deg).rename('first_degradation_by2014'),
    history_year(first_def).rename('first_deforestation_by2014'),
    age_reg.unmask(0).rename('regrowth_run_age2014'),
    age_deg.unmask(0).rename('degraded_run_age2014'),
    n_trans.unmask(0).rename('annual_class_transitions_1990_2014'),
])
if ADD_2025_ENDPOINT:
    endpoint=ee.ImageCollection(jrc+'TransitionMap_Subtypes').mosaic().select(0)
    bands.append(endpoint.unmask(0).toInt16().rename('transition_subtype_ENDPOINT_2025'))
tmf_hist=ee.Image.cat(bands)

# -------------------------------------------------------------------------
# 3. Count all 30 m pixel categories in each intersected polygon (not samples)
# -------------------------------------------------------------------------
rows=[]
ordered=rgb_foot.sort_values('plot_id').reset_index(drop=True)
expected=list(tmf_hist.bandNames().getInfo())
for start in range(0,len(ordered),TMF_BATCH):
    chunk=ordered.iloc[start:start+TMF_BATCH]
    fn=G3_DIR/f'pixel_hist_{start:03d}_{start+len(chunk):03d}.json'
    if fn.exists() and not RECOMPUTE_TMF_HIST:
        with open(fn) as f:partial=json.load(f)
        rows.extend(partial)
        print('Cached pixel histories:',fn.name)
        continue
    features=[ee.Feature(ee.Geometry(mapping(x.geometry),geodesic=False),
                         {'plot_id':str(x.plot_id)})
              for x in chunk.itertuples(index=False)]
    reduced=tmf_hist.reduceRegions(
        collection=ee.FeatureCollection(features),
        reducer=ee.Reducer.frequencyHistogram(),
        scale=TMF_SCALE,tileScale=4)
    response=reduced.getInfo()['features']
    partial=[]
    for feat in response:
        p=feat['properties']
        rec={'plot_id':p['plot_id']}
        for band in expected:
            hist=p.get(band) or {}
            rec[band]={str(k):int(round(v)) for k,v in hist.items()}
        partial.append(rec)
    assert len(partial)==len(chunk),'Unexpected EE results for batch'
    with open(fn,'w') as f:json.dump(partial,f)
    rows.extend(partial)
    print('TMF all-pixel histograms:',start+1,'–',start+len(chunk),'/',len(ordered))
assert len(rows)==len(ordered) and len({r['plot_id'] for r in rows})==len(ordered)
with open(G3_DIR/'WWF_all_RGB_footprints_full_TMF_pixel_histograms.json','w') as f:
    json.dump(rows,f,indent=1)

# -------------------------------------------------------------------------
# 4. Summaries and time/age bins; all derived from the per-pixel histograms
# -------------------------------------------------------------------------
def count_if(hist,predicate):
    return sum(int(count) for key,count in hist.items() if predicate(int(key)))
tab=[]
for r in rows:
    d={'plot_id':r['plot_id']}
    for b in expected:
        h=r[b]
        d['n_valid_'+b]=count_if(h,lambda x:x!=0)
        d['n_all_'+b]=sum(h.values())
    n=d['n_all_status_2014']
    d['tmf_pixels_2014']=n
    for year in [2014,2015]:
        h=r['status_'+str(year)]
        for c,label in enumerate(['undisturbed','degraded','deforested',
                                   'regrowth','water','other'],1):
            d[f'pct_{year}_{label}']=100*h.get(str(c),0)/max(n,1)
    for key,out in [('first_degradation_by2014','degradation'),
                    ('first_deforestation_by2014','deforestation')]:
        h=r[key]
        for lo,hi,label in [(1982,1999,'before2000'),(2000,2009,'2000_2009'),
                             (2010,2014,'2010_2014')]:
            d[f'pct_first_{out}_{label}']=100*count_if(h,lambda y:lo<=y<=hi)/max(n,1)
        d[f'pct_first_{out}_by2014']=100*count_if(h,lambda y:1982<=y<=2014)/max(n,1)
    for key,label in [('regrowth_run_age2014','regrowth'),
                      ('degraded_run_age2014','degraded')]:
        h=r[key]
        for lo,hi,cls in [(1,3,'1_3y'),(4,7,'4_7y'),(8,100,'8plus_y')]:
            d[f'pct_{label}_state_age_{cls}']=100*count_if(h,lambda a:lo<=a<=hi)/max(n,1)
    h=r['annual_class_transitions_1990_2014']
    d['pct_2plus_annual_transitions']=100*count_if(h,lambda x:x>=2)/max(n,1)
    if ADD_2025_ENDPOINT:
        d['n_2025_transition_subtypes']=count_if(
            r['transition_subtype_ENDPOINT_2025'],lambda x:x!=0)
    tab.append(d)
summary=pd.DataFrame(tab).merge(ordered[['plot_id','utm_zone','native_gsd_cm']],
                                on='plot_id',validate='one_to_one')
summary['potential_RGB_footprint_only']=True
summary['TMF_pixel_scale_m']=TMF_SCALE
outcsv=G3_DIR/'WWF_all_RGB_footprints_TMF_pixel_descriptors.csv'
summary.to_csv(outcsv,index=False)
merged=ordered.merge(summary.drop(columns=['utm_zone','native_gsd_cm']),
                     on='plot_id',validate='one_to_one')
outgpkg=G3_DIR/'WWF_all_RGB_footprints_TMF_pixel_descriptors.gpkg'
merged.to_file(outgpkg,driver='GPKG')
print('Saved full categorical pixel histograms:',G3_DIR/'WWF_all_RGB_footprints_full_TMF_pixel_histograms.json')
print('Saved descriptor table:',outcsv)
print('Saved spatial table:',outgpkg)
display(summary.sort_values('pct_regrowth_state_age_1_3y',ascending=False).head(20).round(2))
print('Total 2014 TMF pixel observations:',summary.tmf_pixels_2014.sum())

# Map lives in G.3B below: rerun ONLY G.3B if summary CSV/GPKG already exists.
print('Run G.3B for interactive TMF maps. No extra histogram processing required.')


### 4.D — Interactive TMF pixel map (Earth Engine tiles)

This **standalone** visualization reads the saved G.3 GeoPackage and uses native **Folium + Earth Engine map tiles**. It does **not** reprocess TIFF headers, rerun 4.84 million-pixel histograms or depend on `geemap.foliumap` / `xyz_to_folium`.

Switch layers to inspect geographic patterns of 2014 status, young regrowth, degradation and deforestation years, and 1990–2014 class-transition counts. The coloured TMF pixels are clipped to **survey polygon ∩ georeferenced RGB TIFF bounds**, not verified RGB-valid pixel masks. In historical maps all time-dependent layers stop at 2014.

In [ ]:
# 4.D.1 — Interactive TMF raster map
# Standalone: mount Drive and reconstruct saved-result directory if this is a new runtime.
from pathlib import Path
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive',force_remount=False)
WORK_DIR=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/V3_1/NB08_WWF_Lidar')
assert WORK_DIR.exists(), f'WWF NB08 output folder not found: {WORK_DIR}'
# Only needs saved G.3 GeoPackage, WORK_DIR and Google Earth Engine access.
import ee, folium, geopandas as gpd
from pathlib import Path
from shapely.geometry import mapping
from IPython.display import display
from folium.plugins import Fullscreen

EE_PROJECT=(globals().get('EE_PROJECT') or 'musanga-511110').strip()
try:
    ee.Initialize(project=EE_PROJECT)
    ee.Number(1).getInfo()
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

G3_DIR=WORK_DIR/'TMF_strip_descriptors'/'rgb_footprint_pixels'
saved=G3_DIR/'WWF_all_RGB_footprints_TMF_pixel_descriptors.gpkg'
assert saved.exists(), f'G.3 spatial results missing: {saved}'
g3_map_data=gpd.read_file(saved).to_crs('EPSG:4326')
assert len(g3_map_data)==216 and g3_map_data.plot_id.is_unique
print('Loaded saved RGB-potential footprints:',len(g3_map_data))

# Server-side clip to collection of all candidate georeferenced RGB footprints.
fc=ee.FeatureCollection([
    ee.Feature(ee.Geometry(mapping(row.geometry),geodesic=False))
    for row in g3_map_data.itertuples(index=False)
])
JRC='projects/JRC/TMF/v1_2025/'
annual=ee.ImageCollection(JRC+'AnnualChanges').mosaic()
state14=annual.select('Dec2014')
firstdeg=ee.ImageCollection(JRC+'DegradationYear').mosaic().select(0)
firstdef=ee.ImageCollection(JRC+'DeforestationYear').mosaic().select(0)
age_reg=ee.Image.constant(0).toInt16()
nchanges=ee.Image.constant(0).toInt16()
prev=None
for year in range(1990,2015):
    current=annual.select('Dec'+str(year))
    age_reg=age_reg.add(1).multiply(current.eq(4)).toInt16()
    if prev is not None:
        nchanges=nchanges.add(current.neq(prev)).toInt16()
    prev=current

def add_ee_layer(folium_map,ee_image,vis,label,show=False):
    # Earth Engine returns a standard HTTPS XYZ tile URL for Folium.
    tile=ee_image.clipToCollection(fc).getMapId(vis)['tile_fetcher'].url_format
    folium.TileLayer(tiles=tile,attr='Google Earth Engine / JRC TMF',
                     name=label,overlay=True,control=True,show=show,
                     max_zoom=20,opacity=0.85).add_to(folium_map)

m=folium.Map(location=[-2.0,23.5],zoom_start=6,tiles=None,
             control_scale=True,prefer_canvas=True)
# Reliable background independent of public OSM / Carto tile availability.
folium.TileLayer(
    tiles='data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mP8/x8AAusB9Y9UxQAAAABJRU5ErkJggg==',
    attr='Offline background',name='Blank (offline)',overlay=False,
    show=True,control=True,max_zoom=20).add_to(m)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri imagery',name='Esri satellite (optional)',overlay=False,
    show=False,control=True,max_zoom=20).add_to(m)
add_ee_layer(m,state14,{'min':1,'max':6,
    'palette':['005a25','8cba44','e97649','a0db4d','0089c6','cccccc']},
    'TMF 2014 forest classes',True)
add_ee_layer(m,age_reg.updateMask(state14.eq(4)),
    {'min':1,'max':12,'palette':['f6e799','a4cf50','368f49']},
    'Regrowth state age in 2014 (years)')
add_ee_layer(m,firstdeg.updateMask(firstdeg.gte(2010).And(firstdeg.lte(2014))),
    {'min':2010,'max':2014,'palette':['ffda75','ffb04b','ff6b42','c52d36']},
    'First degradation year 2010–2014')
add_ee_layer(m,firstdef.updateMask(firstdef.gte(2010).And(firstdef.lte(2014))),
    {'min':2010,'max':2014,'palette':['ffe2a8','ffa556','f7603a','a9152d']},
    'First deforestation year 2010–2014')
add_ee_layer(m,nchanges.updateMask(nchanges.gt(0)),
    {'min':1,'max':5,'palette':['fff5b6','fcb45a','e4573a','8d2052']},
    'Number of annual TMF status changes (1990–2014)')

outline_cols=['plot_id','pct_2014_undisturbed',
              'pct_2014_degraded','pct_2014_regrowth',
              'pct_first_deforestation_2010_2014','geometry']
outline=g3_map_data[outline_cols].copy()
for col in outline_cols[1:-1]:
    outline[col]=outline[col].round(2)
folium.GeoJson(
    outline.to_json(),
    style_function=lambda f:{'color':'#222222','weight':1.0,
                             'fillColor':'#000000','fillOpacity':0.0},
    tooltip=folium.GeoJsonTooltip(
        fields=outline_cols[:-1],
        aliases=['WWF plot','Undisturbed (%)','Degraded (%)',
                 'Regrowth (%)','First deforestation 2010–14 (%)'],
        localize=True),
    name='WWF potential RGB footprints'
).add_to(m)
bounds=g3_map_data.total_bounds
m.fit_bounds([[bounds[1],bounds[0]],[bounds[3],bounds[2]]])
Fullscreen().add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
out=G3_DIR/'WWF_all_216_RGB_TMF_pixel_map.html'
m.save(str(out))
print('Interactive TMF map saved:',out)
display(m)
print('NOTE: RGB no-data/black margins are still unmasked. No annotation sites selected.')


### 4.E — TMF distributions, tables and figures

This is a **pure local statistics and visualization block**: it loads the saved G.3 CSV and complete JSON pixel histograms. **No Earth Engine calls or TIFF downloads.** Two complementary perspectives are reported:

1. **Across-strip distributions:** descriptive statistics (min, P10, P25, median, P75, P90, P95, max, standard deviation and fraction of strips with zero) for all `pct_*` TMF features; histograms of key gradients and a per-strip composition heatmap. **Each strip has equal weight** here.
2. **Pooled TMF pixel distributions:** sum of the complete pixel histograms across all 216 potential RGB footprints. For each annual status year, degradation/deforestation *first-event* year, regrowth-state age, degraded-state age, historical class-transition count and **separately labeled 2025 transition subtypes**, display the full categorical distribution (counts and percentages). **Each mapped 30 m pixel observation has equal weight**, subject to the server's raster pixel grid and possible overlap between strips.

These are distributions **within survey polygon ∩ georeferenced RGB raster bounds**, **not** masks of genuine RGB-valid pixels. Numbers are historical TMF classifications, **not** Musanga occurrence and not unbiased landscape-wide prevalence.

In [ ]:
# 4.E.1 — Statistical TMF overview
# Standalone: mount Drive and reconstruct saved-result directory if this is a new runtime.
from pathlib import Path
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive',force_remount=False)
WORK_DIR=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/V3_1/NB08_WWF_Lidar')
assert WORK_DIR.exists(), f'WWF NB08 output folder not found: {WORK_DIR}'
import json, os, shutil, tempfile, pandas as pd, numpy as np, matplotlib
matplotlib.use('Agg')  # avoid slow interactive backend while rendering multiple figures
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

G3_DIR=WORK_DIR/'TMF_strip_descriptors'/'rgb_footprint_pixels'
csv=G3_DIR/'WWF_all_RGB_footprints_TMF_pixel_descriptors.csv'
histfile=G3_DIR/'WWF_all_RGB_footprints_full_TMF_pixel_histograms.json'
assert csv.exists() and histfile.exists(), 'Run G.3 once; the saved CSV and histogram JSON are required.'
strips=pd.read_csv(csv)
with open(histfile) as handle: pixel_rows=json.load(handle)
assert len(strips)==216 and strips.plot_id.nunique()==216
assert len(pixel_rows)==216 and len({r['plot_id'] for r in pixel_rows})==216
REPORT=G3_DIR/'distribution_report'
REPORT.mkdir(parents=True,exist_ok=True)
LOCAL_PLOTS=Path(tempfile.mkdtemp(prefix='wwf_tmf_figs_'))
def save_report_figure(fig, filename):
    # Avoid rendering directly onto mounted Drive; export modest PNG to local disk first.
    local=LOCAL_PLOTS/filename
    fig.savefig(local,dpi=110,bbox_inches='tight')
    plt.close(fig)
    final=REPORT/filename
    shutil.copyfile(local,final)
    print('Saved',final)
    from IPython.display import Image,display
    display(Image(filename=str(local),width=950))
    return final
print('Loaded',len(strips),'WWF strips and',len(pixel_rows),'complete TMF pixel histograms')

# -------------------------------------------------------------
# 1 — Equal-weight strip-level distributions for EVERY percent feature
# -------------------------------------------------------------
features=sorted(c for c in strips.columns if c.startswith('pct_'))
assert features,'Missing percent descriptors from G.3'
stat=[]
for name in features:
    values=pd.to_numeric(strips[name],errors='coerce').replace([np.inf,-np.inf],np.nan)
    valid=values.dropna()
    if len(valid)==0:continue
    q=valid.quantile([.10,.25,.50,.75,.90,.95])
    stat.append(dict(descriptor=name,n=len(valid),mean=valid.mean(),
         sd=valid.std(),min=valid.min(),p10=q.loc[.10],p25=q.loc[.25],
         median=q.loc[.50],p75=q.loc[.75],p90=q.loc[.90],p95=q.loc[.95],
         max=valid.max(),pct_strips_zero=100*(valid==0).mean()))
stats=pd.DataFrame(stat)
stats.to_csv(REPORT/'WWF_TMF_all_descriptors_across_216_strips.csv',index=False)
print('Across-strip distribution of ALL',len(stats),'TMF percentage descriptors:')
display(stats.round(2))

# -------------------------------------------------------------
# 2 — Aggregate the FULL categorical TMF histograms from G.3
# -------------------------------------------------------------
bands=sorted({k for row in pixel_rows for k in row if k!='plot_id'})
long=[]
for band in bands:
    counter={}
    for row in pixel_rows:
        for k,v in row.get(band,{}).items():
            key=int(k)
            counter[key]=counter.get(key,0)+int(v)
    total=sum(counter.values())
    nonzero=sum(v for k,v in counter.items() if k!=0)
    for key,value in sorted(counter.items()):
        long.append(dict(band=band,category_value=key,count=value,
                         total_pixels_for_band=total,
                         percent_of_all_pixels=100*value/max(total,1),
                         percent_of_nonzero_pixels=(100*value/nonzero
                            if key!=0 and nonzero else 0)))
    print(f'{band}: {len(counter)} distinct values | total={total:,} | '
          f'nonzero={nonzero:,}')
pooled=pd.DataFrame(long)
pooled.to_csv(REPORT/'WWF_TMF_complete_pooled_pixel_distributions.csv',index=False)
# Keep explicit status codes; 0 denotes masked/unclassified and must NOT be a forest class.
status_codes={0:'Masked / unclassified',1:'Undisturbed',2:'Degraded',
              3:'Deforested',4:'Regrowth',5:'Water',6:'Other'}
status=pooled[pooled.band.isin(['status_2014','status_2015'])].copy()
status['class']=status.category_value.map(status_codes).fillna('Other code')
display(status[['band','category_value','class','count','percent_of_all_pixels']].round(3))
print('Pooled-pixel statistics are area-like, whereas across-strip statistics give equal weight to each WWF strip.')

# -------------------------------------------------------------
# 3 — Figures: composition by 2014 class, strip distributions
# -------------------------------------------------------------
main_classes=['undisturbed','degraded','deforested','regrowth','water','other']
fig,axes=plt.subplots(2,3,figsize=(15,8),constrained_layout=True)
for ax,cl in zip(axes.flat,main_classes):
    x=strips[f'pct_2014_{cl}'].dropna()
    ax.hist(x,bins=np.arange(0,101,5),edgecolor='white')
    ax.axvline(x.median(),ls='--',lw=1.4)
    ax.set(title=f'{cl} (median {x.median():.2f}%)',xlabel='% of potential RGB footprint',
           ylabel='Number of strips',xlim=(0,100))
p1=REPORT/'2014_TMF_class_percentages_all_strips.png'
p1=save_report_figure(fig,p1.name)

# The long-tailed, rare regrowth fractions deserve a zoomed view.
rare=['pct_2014_regrowth','pct_regrowth_state_age_1_3y',
      'pct_regrowth_state_age_4_7y','pct_regrowth_state_age_8plus_y',
      'pct_first_degradation_2010_2014',
      'pct_first_deforestation_2010_2014','pct_2plus_annual_transitions']
rare=[x for x in rare if x in strips]
fig,axes=plt.subplots(2,4,figsize=(18,8),constrained_layout=True)
for ax,key in zip(axes.flat,rare):
    x=pd.to_numeric(strips[key],errors='coerce').dropna()
    ax.hist(x,bins=30,edgecolor='white')
    ax.axvline(x.median(),ls='--',lw=1.4)
    ax.set(title=key.replace('pct_','').replace('_',' '),xlabel='% of strip',
           ylabel='WWF strips')
for ax in list(axes.flat)[len(rare):]:ax.axis('off')
p2=REPORT/'2014_TMF_rare_recovery_disturbance_distributions.png'
p2=save_report_figure(fig,p2.name)

# -------------------------------------------------------------
# 4 — Complete year-by-year / age histograms (not just summary bins)
# -------------------------------------------------------------
requested_bands=[
    'first_degradation_by2014','first_deforestation_by2014',
    'regrowth_run_age2014','degraded_run_age2014',
    'annual_class_transitions_1990_2014',
    'transition_subtype_ENDPOINT_2025'
]
requested_bands=[x for x in requested_bands if x in bands]
fig,axes=plt.subplots(len(requested_bands),1,
    figsize=(16,max(4,2.7*len(requested_bands))),constrained_layout=True)
axes=np.atleast_1d(axes)
for ax,band in zip(axes,requested_bands):
    sub=pooled[(pooled.band==band)&(pooled.category_value!=0)]
    ax.bar(sub['category_value'].to_numpy(),sub['count'].to_numpy())
    ax.set(xlabel='Year, years-in-state, or categorical code',
           ylabel='30 m pixel observations',title=band)
    ax.tick_params(axis='x',labelsize=8)
p3=REPORT/'TMF_full_year_age_transition_histograms.png'
p3=save_report_figure(fig,p3.name)
print('Saved full year / age / transition distributions, including 2025 endpoint as separate series.')

# -------------------------------------------------------------
# 5 — Heatmap: how 2014 TMF composition varies among all strips
# -------------------------------------------------------------
heat=strips.sort_values('pct_2014_regrowth',ascending=False)
arr=heat[[f'pct_2014_{c}' for c in main_classes]].fillna(0).to_numpy()
fig,ax=plt.subplots(figsize=(9,13))
im=ax.imshow(arr,aspect='auto',interpolation='nearest',vmin=0,vmax=100,cmap='viridis')
ax.set_xticks(range(len(main_classes)),main_classes,rotation=40,ha='right')
ax.set_yticks(np.arange(0,len(heat),10))
ax.set_yticklabels(heat.plot_id.iloc[::10].tolist(),fontsize=7)
ax.set_title('All 216 WWF RGB footprints — 2014 class composition\n'
             '(sorted by regrowth proportion)')
fig.colorbar(im,ax=ax,label='TMF pixels in class (%)',fraction=.03,pad=.02)
p4=REPORT/'TMF_2014_composition_heatmap_all_216_strips.png'
p4=save_report_figure(fig,p4.name)

print('\nReport folder:',REPORT)
for file in [p1,p2,p3,p4]:
    print('Figure:',file.name)
print('Tables: WWF_TMF_all_descriptors_across_216_strips.csv; '
      'WWF_TMF_complete_pooled_pixel_distributions.csv')
print('NOTE: first-event years are not the most recent disturbance year;'
      ' 2025 endpoint subtypes are retrospective only.')


### 4.F — Offline HTML atlas of strip-level TMF statistics

Make one **genuinely self-contained HTML document**, embedding simplified WWF polygons and the saved TMF attributes. The HTML uses **native SVG + JavaScript** rather than Folium/Leaflet CDN, GEE tokens, external basemaps or external files; it can be emailed, placed on Google Drive or hosted as an ordinary static page. Colleagues can open the downloaded HTML locally and use the selector to switch among all available TMF percentage descriptors, hover a strip for its values, click for details, and zoom/pan. The background is a blank geographic coordinate plane (no satellite imagery); the survey geometry and statistics are embedded. The full TMF pixel rasters are **not embedded**: this is a per-strip thematic atlas, not the exact 30 m pixel map. It uses the existing G.3 results only and does not call Earth Engine.

**Sharing:** Download the resulting HTML and email/share as a file (some Google Drive previews block scripts, so collaborators should download and open it in a browser). It will remain usable without Earth Engine accounts and without expiring GEE tile URLs.

In [ ]:
# 4.F.1 — Durable strip-level HTML atlas
# Embedded geographic polygons + all TMF percentages; no Earth Engine or CDN.
from pathlib import Path
from google.colab import drive
import json, html
import pandas as pd, numpy as np, geopandas as gpd
from shapely.geometry import mapping

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive',force_remount=False)
WORK_DIR=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/V3_1/NB08_WWF_Lidar')
folder=WORK_DIR/'TMF_strip_descriptors'/'rgb_footprint_pixels'
source=folder/'WWF_all_RGB_footprints_TMF_pixel_descriptors.gpkg'
assert source.exists(), f'Run G.3 first: {source}'
g=gpd.read_file(source).to_crs('EPSG:4326')
assert len(g)==216 and g.plot_id.is_unique
# Keep a small HTML file: simplify only the display geometry in geographic degrees.
g['geometry']=g.geometry.simplify(.002,preserve_topology=True)
cols=sorted(k for k in g.columns if k.startswith('pct_'))
assert cols
records=[]
for row in g.itertuples(index=False):
    rec={'id':str(row.plot_id),'zone':str(row.utm_zone),
         'geometry':mapping(row.geometry)}
    for col in cols:
        val=getattr(row,col)
        rec[col]=None if pd.isna(val) else round(float(val),3)
    records.append(rec)
payload=json.dumps({'features':records,'metrics':cols},separators=(',',':'))
# A real single-file SVG explorer: no fetch(), external tiles, JS libraries or CDN.
page=r"""<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>WWF RGB footprints | TMF forest gradient 2014–2015</title>
<style>
:root{font-family:system-ui,-apple-system,Segoe UI,sans-serif;color:#203235}
body{margin:0;background:#f5f7f6}
header{padding:16px 20px;background:#123a36;color:white}
header h1{font-size:20px;margin:0 0 5px}
header small{color:#cfe5db}
#tools{display:flex;align-items:center;flex-wrap:wrap;gap:10px;padding:12px 20px;background:white;border-bottom:1px solid #ddd}
select,button{font:inherit;padding:7px 10px;border:1px solid #bdc9c5;background:white;border-radius:5px}
main{display:grid;grid-template-columns:minmax(0,1fr) 290px;gap:14px;padding:14px}
#mapwrap{background:#edf3ee;border:1px solid #cbd8ce;border-radius:8px;position:relative;min-height:490px;overflow:hidden}
svg{display:block;width:100%;height:72vh;min-height:490px;cursor:grab;touch-action:none}
path.strip{stroke:#435f5a;stroke-width:0.9;vector-effect:non-scaling-stroke;cursor:pointer}
path.strip:hover{stroke:#131d1b;stroke-width:2;vector-effect:non-scaling-stroke}
#info{padding:15px;background:white;border:1px solid #dce4df;border-radius:8px;font-size:13px}
#info h2{font-size:16px;margin:0 0 9px}
#info table{border-collapse:collapse;width:100%}
#info td{padding:4px 3px;border-bottom:1px solid #eee}
#info td:last-child{text-align:right;font-variant-numeric:tabular-nums}
#tip{position:absolute;background:#173c38;color:white;padding:5px 8px;font-size:12px;border-radius:5px;pointer-events:none;display:none;z-index:4}
#legend{display:flex;align-items:center;gap:4px;font-size:11px;color:#394c48}
#legend i{width:130px;height:10px;display:inline-block;background:linear-gradient(90deg,#edf1e6,#d5df9b,#a5bd71,#569b71,#176c55)}
#note{font-size:11px;color:#53615b;margin-top:10px;line-height:1.5}
@media(max-width:800px){main{grid-template-columns:1fr}svg{height:58vh}}
</style></head><body>
<header><h1>WWF RGB forest-gradient atlas</h1>
<small>216 potential RGB footprints | JRC TMF 30 m | 2014–2015 survey period | shareable, no Earth Engine or external resources</small></header>
<section id="tools"><label for="metric">TMF descriptor:</label>
<select id="metric"></select><span id="legend"><span>0%</span><i></i><span id="upper">100%</span></span>
<button id="zoomIn">Zoom +</button><button id="zoomOut">Zoom −</button><button id="reset">Reset</button>
<span style="font-size:11px">Hover or click a strip · Wheel to zoom · Drag to pan</span></section>
<main><div id="mapwrap"><svg id="map" viewBox="0 0 1200 690" aria-label="WWF TMF survey geographic distribution">
<rect width="1200" height="690" fill="#eef2ee"/><g id="mapgroup"></g></svg>
<div id="tip"></div></div><aside id="info"><h2>Select a WWF strip</h2>
<div id="detail">Click any strip to see its TMF profile.</div>
<div id="note">TMF fractions refer to the intersection of the WWF survey polygon and RGB raster bounds, <strong>not validated nonblack RGB pixels</strong>. No actual WWF RGB photographs or 30 m TMF rasters are embedded. Historical categories are not Musanga annotations.</div></aside></main>
<script>
const DATA=__PAYLOAD__;
const features=DATA.features, metrics=DATA.metrics;
const WIDTH=1200,HEIGHT=690,PAD=38;
const sel=document.getElementById('metric'),root=document.getElementById('mapgroup');
const svg=document.getElementById('map'),tip=document.getElementById('tip');
metrics.forEach(m=>{const op=document.createElement('option');op.value=m;op.textContent=m.replace(/^pct_/,'').replaceAll('_',' ');sel.appendChild(op)});
sel.value='pct_2014_regrowth';
let bounds=[Infinity,Infinity,-Infinity,-Infinity];
function coordsWalk(c){if(typeof c[0]==='number'){bounds[0]=Math.min(bounds[0],c[0]);bounds[1]=Math.min(bounds[1],c[1]);bounds[2]=Math.max(bounds[2],c[0]);bounds[3]=Math.max(bounds[3],c[1]);}else c.forEach(coordsWalk)}
features.forEach(f=>coordsWalk(f.geometry.coordinates));
const midlat=(bounds[1]+bounds[3])/2,cos=Math.cos(midlat*Math.PI/180);
const extX=(bounds[2]-bounds[0])*cos,extY=bounds[3]-bounds[1];
const scale=Math.min((WIDTH-2*PAD)/extX,(HEIGHT-2*PAD)/extY);
function toXY(pt){return [WIDTH/2+((pt[0]-(bounds[0]+bounds[2])/2)*cos)*scale,
                    HEIGHT/2-((pt[1]-(bounds[1]+bounds[3])/2))*scale]}
function pathD(geo){const polys=geo.type==='Polygon'?[geo.coordinates]:geo.type==='MultiPolygon'?geo.coordinates:[];
 return polys.flatMap(poly=>poly.map(ring=>ring.map((p,i)=>{const xy=toXY(p);return (i?'L':'M')+xy[0].toFixed(2)+' '+xy[1].toFixed(2)}).join('')+'Z')).join(' ')}
const NS='http://www.w3.org/2000/svg';
features.forEach(f=>{
 const p=document.createElementNS(NS,'path');p.setAttribute('d',pathD(f.geometry));
 p.setAttribute('class','strip');p.setAttribute('fill-rule','evenodd');
 p.addEventListener('mouseenter',e=>{tip.style.display='block';tip.textContent=f.id+' · '+f.zone+' · '+(f[sel.value]??'NA')+'%';});
 p.addEventListener('mousemove',e=>{const b=document.getElementById('mapwrap').getBoundingClientRect();tip.style.left=(e.clientX-b.left+12)+'px';tip.style.top=(e.clientY-b.top+12)+'px';});
 p.addEventListener('mouseleave',()=>tip.style.display='none');
 p.addEventListener('click',()=>showDetails(f));
 root.appendChild(p);f.node=p;
});
let view={x:0,y:0,k:1};
function applyView(){root.setAttribute('transform',`translate(${view.x} ${view.y}) scale(${view.k})`)}
function zoom(factor,cx=600,cy=345){const beforeX=(cx-view.x)/view.k,beforeY=(cy-view.y)/view.k;
view.k=Math.min(35,Math.max(1,view.k*factor));view.x=cx-beforeX*view.k;view.y=cy-beforeY*view.k;applyView()}
document.getElementById('zoomIn').onclick=()=>zoom(1.5);
document.getElementById('zoomOut').onclick=()=>zoom(1/1.5);
document.getElementById('reset').onclick=()=>{view={x:0,y:0,k:1};applyView()};
let drag=null;
svg.addEventListener('pointerdown',e=>{drag={x:e.clientX,y:e.clientY,startX:view.x,startY:view.y};svg.setPointerCapture(e.pointerId)});
svg.addEventListener('pointermove',e=>{if(!drag)return;const b=svg.getBoundingClientRect();view.x=drag.startX+(e.clientX-drag.x)*1200/b.width;view.y=drag.startY+(e.clientY-drag.y)*690/b.height;applyView()});
svg.addEventListener('pointerup',()=>drag=null);
svg.addEventListener('wheel',e=>{e.preventDefault();const b=svg.getBoundingClientRect();
zoom(e.deltaY<0?1.25:0.8,(e.clientX-b.left)*1200/b.width,(e.clientY-b.top)*690/b.height)},{passive:false});
function palette(v,max){if(v===null||v===undefined)return '#c9ccca';let t=Math.max(0,Math.min(1,v/Math.max(max,1e-9)));return `hsl(${(75-72*t).toFixed(0)} 42% ${(92-54*t).toFixed(0)}%)`}
function paint(){const m=sel.value;const vals=features.map(f=>f[m]).filter(v=>v!==null&&Number.isFinite(v));
const sorted=vals.slice().sort((a,b)=>a-b);const q95=sorted[Math.floor(.95*(sorted.length-1))]||1;
features.forEach(f=>f.node.setAttribute('fill',palette(f[m],q95)));
document.getElementById('upper').textContent=q95.toFixed(2)+'% (P95)';
}
sel.onchange=paint;
function showDetails(f){let h='<h2>'+f.id+'</h2><p>UTM zone: '+f.zone+'</p><table>';
for(const m of metrics){const v=f[m];h+='<tr><td>'+m.replace(/^pct_/,'').replaceAll('_',' ')+'</td><td>'+(v===null?'NA':v.toFixed(2)+'%')+'</td></tr>'}
document.getElementById('detail').innerHTML=h+'</table>'}
paint();
</script></body></html>"""
page=page.replace('__PAYLOAD__',payload.replace('</','<\\/'))
out=folder/'WWF_TMF_216_strips_DURABLE_offline_atlas.html'
out.write_text(page,encoding='utf-8')
print('Saved permanent standalone map:',out)
print('Size:',round(out.stat().st_size/1024/1024,2),'MB')
print('Contains',len(records),'embedded polygons and',len(cols),'TMF descriptors.')
print('Download the HTML and open locally; external Earth Engine/Leaflet/CDN NOT required.')
from IPython.display import HTML,display
display(HTML(f'<a href="/content/drive/MyDrive/Colab%20Notebooks/Musanga/V3_1/NB08_WWF_Lidar/TMF_strip_descriptors/rgb_footprint_pixels/{out.name}" target="_blank">HTML file saved to Drive (download to view)</a>'))


### 4.G — Offline 30 m TMF pixel atlas

**This is the pixel-map counterpart to G.3D.** It exports seven **historical** TMF bands directly from Earth Engine as small, clipped 30 m GeoTIFF windows, converts them to paletted transparent PNGs, and embeds the PNGs together with the WWF polygon boundaries and G.3 statistics in **one HTML file**. Earth Engine is needed **only to build it**; colleagues can open the resulting HTML offline with no expiring links, external libraries, or hosted resources. **Nothing is uploaded into Earth Engine.**

Layers: **2014 status, 2015 status, first degradation year up to 2014, first deforestation year up to 2014, regrowth consecutive-state age in 2014, degraded consecutive-state age in 2014, annual class-transition count 1990–2014**. Dates beyond 2014 are explicitly masked for first-event dates. Each plot's downloaded GeoTIFF and compressed tile PNGs are cached, so reruns skip completed strips. Starts with a **six-strip preview** to validate that the cartography works, then set `PREVIEW_ONLY=False` for all 216. A simple geographic SVG displays the tiles (small local geographic reprojection approximation for rendering only); the original TMF grid statistics are unchanged.

**Caveats:** this maps *potential* RGB footprint = LiDAR survey polygon intersected with GeoTIFF rectangular bounds, **not confirmed valid RGB pixels**; no 10 cm imagery is downloaded. TMF first-event years are not latest-disturbance years. This is intended for exploratory spatial sampling, not precise pixel-scale positional measurement or Musanga species labels.

In [ ]:
# 4.G.1 — Durable 30 m TMF HTML atlas
# One Earth Engine 7-band download PER WWF plot; no WWF RGB data downloaded.
import json, base64, io, time, shutil, requests
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from PIL import Image
from rasterio.features import geometry_mask
from shapely.geometry import mapping
from IPython.display import display, HTML
import ee

from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive',force_remount=False)
WORK_DIR=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/V3_1/NB08_WWF_Lidar')
G3_DIR=WORK_DIR/'TMF_strip_descriptors'/'rgb_footprint_pixels'
assert (G3_DIR/'WWF_all_RGB_footprints_TMF_pixel_descriptors.gpkg').exists(), 'Run G.3 first'
foot=gpd.read_file(G3_DIR/'WWF_all_RGB_footprints_TMF_pixel_descriptors.gpkg').to_crs(4326)
assert len(foot)==216 and foot.plot_id.is_unique

EE_PROJECT=(globals().get('EE_PROJECT') or 'musanga-511110').strip()
try:
    ee.Initialize(project=EE_PROJECT)
    ee.Number(1).getInfo()
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

PREVIEW_ONLY=True     # Run 6 first; then False for ALL 216
MAX_PLOTS_PREVIEW=6
REFRESH_CACHED=False
MAX_FAILURES=8
ROOT=G3_DIR/'durable_30m_atlas'
RAW=ROOT/'source_geotiff_cache'
PNG=ROOT/'transparent_png_tiles'
for f in [ROOT,RAW,PNG]: f.mkdir(parents=True,exist_ok=True)

# 30 m source pixels, via GEE. First event dates after acquisition are excluded.
jrc='projects/JRC/TMF/v1_2025/'
annual=ee.ImageCollection(jrc+'AnnualChanges').mosaic()
firstdeg=ee.ImageCollection(jrc+'DegradationYear').mosaic().select(0)
firstdef=ee.ImageCollection(jrc+'DeforestationYear').mosaic().select(0)
status14=annual.select('Dec2014')
status15=annual.select('Dec2015')
reg_age=ee.Image(0).toInt16()
deg_age=ee.Image(0).toInt16()
nchange=ee.Image(0).toInt16()
prev=None
for year in range(1990,2015):
    current=annual.select('Dec'+str(year))
    reg_age=reg_age.add(1).multiply(current.eq(4)).toInt16()
    deg_age=deg_age.add(1).multiply(current.eq(2)).toInt16()
    if prev is not None:nchange=nchange.add(current.neq(prev)).toInt16()
    prev=current
img=ee.Image.cat([
    status14.rename('status2014'),
    status15.rename('status2015'),
    firstdeg.where(firstdeg.gt(2014),0).rename('firstdeg'),
    firstdef.where(firstdef.gt(2014),0).rename('firstdef'),
    reg_age.rename('regrowth_age'),
    deg_age.rename('degraded_age'),
    nchange.rename('transitions')
]).unmask(0).toInt16()
layer_names=['status2014','status2015','firstdeg','firstdef',
             'regrowth_age','degraded_age','transitions']

# Fixed 256-entry colour tables; index 0 is transparent in all layers.
def grad_palette(start,end,maximum=50):
    a=np.array(start,float);b=np.array(end,float)
    pal=[(0,0,0)]
    for i in range(1,256):
        t=min(1,max(0,(i-1)/max(1,maximum-1)))
        pal.append(tuple(np.round(a+(b-a)*t).astype('uint8')))
    return pal
palette={}
palette['status2014']=[(0,0,0),(25,104,53),(203,156,57),(225,95,60),
                       (107,186,77),(54,144,193),(185,189,188)]+[(185,189,188)]*249
palette['status2015']=palette['status2014']
palette['firstdeg']=grad_palette((253,225,150),(155,29,68),33)
palette['firstdef']=grad_palette((255,223,163),(126,18,70),33)
palette['regrowth_age']=grad_palette((245,223,102),(24,121,73),25)
palette['degraded_age']=grad_palette((250,222,123),(153,52,53),25)
palette['transitions']=grad_palette((255,218,114),(135,22,90),5)

def encode_index(band,values):
    v=values.astype(np.int32)
    if band.startswith('status'): out=np.where((v>=1)&(v<=6),v,0)
    elif band in ('firstdeg','firstdef'): out=np.where((v>=1982)&(v<=2014),v-1981,0)
    else: out=np.clip(v,0,255)
    return out.astype('uint8')

# Preview takes first geographically diverse plots; all-plot run uses complete frame.
if PREVIEW_ONLY:
    selection=foot.sort_values('plot_id').groupby('utm_zone',sort=True).head(1).head(MAX_PLOTS_PREVIEW)
else:
    selection=foot.sort_values('plot_id')
print('Processing',len(selection),'RGB potential footprints; first run may need Google EE reads.')

def cached_tile_info(plot_id):
    meta=PNG/(plot_id+'.json')
    if not meta.exists(): return None
    try:
        item=json.loads(meta.read_text())
        if set(item['tiles'])==set(layer_names) and all((PNG/x).exists() for x in item['tiles'].values()):
            return item
    except Exception: pass
    return None

failed=[]
for i,row in enumerate(selection.itertuples(index=False),1):
    pid=str(row.plot_id)
    if cached_tile_info(pid) and not REFRESH_CACHED:
        if i%12==0 or PREVIEW_ONLY: print('Cached',i,'/',len(selection),pid)
        continue
    source_tif=RAW/(pid+'.tif')
    try:
        if REFRESH_CACHED or not source_tif.exists():
            polygon=ee.Geometry(mapping(row.geometry),proj='EPSG:4326',geodesic=False)
            clipped=img.clip(polygon)
            url=clipped.getDownloadURL({
                'name':'wwf_tmf_'+pid,
                'region':polygon,
                'scale':30,
                'crs':'EPSG:4326',
                'format':'GEO_TIFF',
                'filePerBand':False
            })
            # The GEE download URL is temporary, but output TIFF/PNG files are permanent.
            response=requests.get(url,timeout=180)
            response.raise_for_status()
            source_tif.write_bytes(response.content)
        with rasterio.open(source_tif) as ds:
            assert ds.count==7, f'{pid}: expected 7 bands, got {ds.count}'
            assert ds.crs.to_epsg()==4326, f'{pid}: unexpected output CRS {ds.crs}'
            arr=ds.read()
            inside=geometry_mask([mapping(row.geometry)],out_shape=(ds.height,ds.width),
                                 transform=ds.transform,invert=True)
            bounds=list(ds.bounds)
        tiles={}
        for band_index,band in enumerate(layer_names):
            q=encode_index(band,arr[band_index])
            q[~inside]=0
            pic=Image.fromarray(q,mode='P')
            flat=np.array(palette[band],dtype=np.uint8).ravel().tolist()
            pic.putpalette(flat)
            pngname=pid+'_'+band+'.png'
            pic.save(PNG/pngname,optimize=True,transparency=0)
            tiles[band]=pngname
        item={'plot_id':pid,'bounds':bounds,'tiles':tiles}
        (PNG/(pid+'.json')).write_text(json.dumps(item))
        if i%12==0 or PREVIEW_ONLY:print('Prepared',i,'/',len(selection),pid)
    except Exception as err:
        failed.append((pid,str(err)[:250]))
        print('FAIL',pid,str(err)[:180])
        if len(failed)>=MAX_FAILURES:
            print('Stopping after repeated errors; previous strips remain cached.')
            break
print('Failed plot IDs:',failed[:MAX_FAILURES])

# Embed one 30 m image per plot per selectable layer, plus all TMF strip descriptors.
records=[]
for row in selection.itertuples(index=False):
    pid=str(row.plot_id)
    item=cached_tile_info(pid)
    if item is None:continue
    props={name:round(float(getattr(row,name)),3)
           for name in foot.columns if name.startswith('pct_')
           and pd.notna(getattr(row,name))}
    records.append(dict(id=pid,zone=str(row.utm_zone),
       bounds=item['bounds'],geometry=mapping(row.geometry),props=props,
       tiles={k:'data:image/png;base64,'+
              base64.b64encode((PNG/v).read_bytes()).decode('ascii')
              for k,v in item['tiles'].items()}))
assert records, 'No completed image tiles found; inspect failures above.'
print('Packing HTML using',len(records),'plots and',len(layer_names),'layers')

# SVG drawing: pixels and polygons in the same approximate lon-lat projection.
# All pixel positioning comes from georeferenced GeoTIFF bounds; SVG is for exploration.
html_template=r"""<!doctype html><html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>WWF 30 m TMF pixel atlas — 2014–2015</title>
<style>
body{margin:0;background:#f4f7f5;font-family:system-ui,Arial;color:#243d37}
header{background:#153e34;color:#fff;padding:14px 18px}h1{font-size:19px;margin:0 0 4px}
small{opacity:.8}section{background:white;padding:9px 16px;display:flex;align-items:center;gap:12px;flex-wrap:wrap}
select,button{font:inherit;padding:5px;border:1px solid #ccd5cf;background:white;border-radius:5px}
main{display:grid;grid-template-columns:minmax(0,1fr) 280px;gap:12px;padding:12px}
#canvas{position:relative;border:1px solid #d4ddd5;border-radius:8px;background:#edf1e9}
svg{width:100%;height:75vh;min-height:450px;display:block;touch-action:none;cursor:grab}
#panel{background:white;padding:13px;border:1px solid #d5ded7;border-radius:8px;font-size:12px;overflow:auto;max-height:75vh}
#panel h3{margin-top:0}#panel table{border-collapse:collapse;width:100%}
td{padding:4px;border-bottom:1px solid #eee}td:last-child{text-align:right}
#tip{display:none;position:absolute;background:#153e34;color:white;padding:5px;border-radius:4px;font-size:12px;pointer-events:none}
@media(max-width:800px){main{grid-template-columns:1fr}svg{height:55vh}}
</style></head><body>
<header><h1>WWF RGB survey — 30 m TMF historical pixel atlas</h1><small id="subtitle">2014–2015 imagery context · offline HTML · historical forest classes and disturbance histories</small></header>
<section><label>Raster layer: <select id="layer"></select></label><button id="plus">Zoom +</button><button id="minus">Zoom −</button><button id="reset">Reset</button><small>Scroll to zoom; drag to pan; click a strip to inspect</small></section>
<main><div id="canvas"><svg id="svg" viewBox="0 0 1200 700"><rect width="1200" height="700" fill="#e9eee9"/><g id="scene"><g id="tiles"></g><g id="outlines"></g></g></svg><div id="tip"></div></div>
<aside id="panel"><h3>WWF strip statistics</h3><div id="stats">Click a strip to inspect its 2014 forest composition and historical disturbance descriptors.</div>
<p><b>Legend:</b> 2014/15 classes: dark green undisturbed, ochre degraded, orange deforested, light green regrowth, blue water, grey other. Year and age layers: pale yellow older/shorter to dark red/green newer/longer. Transparent = not applicable or outside footprint.</p>
<p><b>Note:</b> source 30 m TMF pixels clipped to potential WWF RGB footprints, not confirmed valid RGB. Geographical placement is approximate for SVG visualization. First-event year ≠ latest event. No WWF photos or species identifications included.</p></aside></main>
<script>
const DATA=__DATA__;
const LAYERS=['status2014','status2015','firstdeg','firstdef','regrowth_age','degraded_age','transitions'];
const TITLES=['TMF status 2014','TMF status 2015','First degradation year by 2014','First deforestation year by 2014','Consecutive regrowth-state years in 2014','Consecutive degraded-state years in 2014','Annual class transitions 1990–2014'];
const select=document.getElementById('layer'),tileG=document.getElementById('tiles'),outlineG=document.getElementById('outlines');
const root=document.getElementById('scene'),svg=document.getElementById('svg'),tip=document.getElementById('tip');
LAYERS.forEach((x,i)=>{const o=document.createElement('option');o.value=x;o.textContent=TITLES[i];select.appendChild(o)});
const BB=[Infinity,Infinity,-Infinity,-Infinity];
for(const f of DATA){const b=f.bounds;BB[0]=Math.min(BB[0],b[0]);BB[1]=Math.min(BB[1],b[1]);BB[2]=Math.max(BB[2],b[2]);BB[3]=Math.max(BB[3],b[3]);}
const midlat=(BB[1]+BB[3])/2,C=Math.cos(midlat*Math.PI/180);
const scale=Math.min(1100/((BB[2]-BB[0])*C),620/(BB[3]-BB[1]));
function xy(lon,lat){return [600+(lon-(BB[0]+BB[2])/2)*C*scale,350-(lat-(BB[1]+BB[3])/2)*scale]}
const NS='http://www.w3.org/2000/svg';
function outline(geo){const polys=geo.type==='Polygon'?[geo.coordinates]:(geo.coordinates||[]);
return polys.flatMap(poly=>poly.map(ring=>ring.map((p,i)=>{
 const q=xy(p[0],p[1]);return (i?'L':'M')+q[0].toFixed(1)+','+q[1].toFixed(1)}).join('')+'Z')).join(' ')}
for(const f of DATA){
  const b=f.bounds,p0=xy(b[0],b[3]),p1=xy(b[2],b[1]);
  const im=document.createElementNS(NS,'image');
  im.setAttribute('x',p0[0]);im.setAttribute('y',p0[1]);
  im.setAttribute('width',p1[0]-p0[0]);im.setAttribute('height',p1[1]-p0[1]);
  im.setAttribute('preserveAspectRatio','none');im.setAttribute('href',f.tiles.status2014);
  tileG.appendChild(im);f.image=im;
  const p=document.createElementNS(NS,'path');p.setAttribute('d',outline(f.geometry));
  p.setAttribute('fill','none');p.setAttribute('stroke','#263f39');
  p.setAttribute('stroke-width','.8');p.setAttribute('vector-effect','non-scaling-stroke');
  p.setAttribute('cursor','pointer');outlineG.appendChild(p);
  p.addEventListener('mouseenter',()=>{tip.textContent=f.id;tip.style.display='block'});
  p.addEventListener('mousemove',e=>{const b=document.getElementById('canvas').getBoundingClientRect();tip.style.left=(e.clientX-b.left+10)+'px';tip.style.top=(e.clientY-b.top+10)+'px'});
  p.addEventListener('mouseleave',()=>tip.style.display='none');
  p.addEventListener('click',()=>{document.getElementById('stats').innerHTML='<h3>'+f.id+'</h3><p>Zone '+f.zone+'</p><table>'+
    Object.entries(f.props).map(([k,v])=>'<tr><td>'+k.replace('pct_','').replaceAll('_',' ')+'</td><td>'+v.toFixed(2)+'%</td></tr>').join('')+'</table>'});
}
select.onchange=()=>{const layer=select.value;DATA.forEach(f=>f.image.setAttribute('href',f.tiles[layer]))};
let t={x:0,y:0,z:1};function redraw(){root.setAttribute('transform','translate('+t.x+' '+t.y+') scale('+t.z+')')}
function zoom(amount,x=600,y=350){const old=t.z;t.z=Math.max(1,Math.min(55,t.z*amount));t.x=x-(x-t.x)*t.z/old;t.y=y-(y-t.y)*t.z/old;redraw()}
document.getElementById('plus').onclick=()=>zoom(1.5);
document.getElementById('minus').onclick=()=>zoom(1/1.5);
document.getElementById('reset').onclick=()=>{t={x:0,y:0,z:1};redraw()};
let drag=null;
svg.addEventListener('pointerdown',e=>{drag={x:e.clientX,y:e.clientY,ox:t.x,oy:t.y};svg.setPointerCapture(e.pointerId)});
svg.addEventListener('pointermove',e=>{if(!drag)return;const b=svg.getBoundingClientRect();t.x=drag.ox+(e.clientX-drag.x)*1200/b.width;t.y=drag.oy+(e.clientY-drag.y)*700/b.height;redraw()});
svg.addEventListener('pointerup',()=>drag=null);
svg.addEventListener('wheel',e=>{e.preventDefault();const b=svg.getBoundingClientRect();zoom(e.deltaY<0?1.3:1/1.3,(e.clientX-b.left)*1200/b.width,(e.clientY-b.top)*700/b.height)},{passive:false});
document.getElementById('subtitle').textContent=DATA.length+' WWF RGB footprints · original 30 m TMF pixels · offline, shareable HTML';
</script></body></html>"""
file=ROOT/('WWF_TMF_30m_PILOT_OFFLINE.html' if PREVIEW_ONLY else 'WWF_TMF_30m_ALL216_OFFLINE.html')
page=html_template.replace('__DATA__',json.dumps(records,separators=(',',':')).replace('</','<\\/'))
file.write_text(page,encoding='utf-8')
print('Durable offline pixel atlas:',file)
print('Size:',round(file.stat().st_size/1024/1024,2),'MB')
print('Share by downloading this HTML; opening it requires NO Earth Engine account.')


## 5 — Hansen × TMF forest-history integration

**Purpose:** answer whether the WWF RGB survey adequately represents forest structural histories and potential Musanga recovery environments, at both **strip level** (equal-weight comparison) and **survey-wide pixel level** (pooled mapped-pixel composition). This is a descriptive sampling-frame audit, **not a validation of Musanga occurrence** or a claim of unbiased forest-area estimates.

Uses **Hansen Global Forest Change v1.13** (2000 canopy cover, loss years 2001–2014, gain 2000–2012) alongside JRC TMF 2014 forest class. For the 2014–2015 UAV survey, **2014 is the conservative reference year**. The 2015-sensitive table is deliberately NOT produced as exact flight dates are unknown; future loss (>2014) is ignored, avoiding temporal leakage.

**Categorical design:** Hansen 2000 canopy cover bins 0–20, 20–40, 40–60, 60–80, 80–100%; first/recorded stand-replacing forest loss **2001–05, 2006–10, 2011–14, or no loss detected through 2014**; Hansen 2000–12 gain; 2014 TMF categories 1–6; and **TMF class × Hansen loss cohort** (24 joint categories) sampled **on the same 30 m analysis grid**. No Hansen forest-loss year is treated as the age of a living regrowing stand: actual state recovery is assessed using the separate TMF regrowth-state age distributions saved by G.3.

**Outputs:** cached per-batch GEE histograms; per-strip comparison CSV and GeoPackage, pooled pixel cross-tab of TMF class × Hansen loss cohort, pooled forest-history summary, forest-canopy distributions, statistics across strips and comparison plots. All samples are from **LiDAR polygon ∩ TIFF bounding rectangle**, not yet a verified RGB valid-data mask. A Hansen loss pixel with no recent loss can have a loss *after* 2014; our category means no **detected loss by 2014**. Native Hansen and TMF grids differ, so the pixel-level cross-tab uses a common 30 m EE sampling scale; minor discrepancies with original G.3 native-class frequencies are expected.

Official GEE Hansen dataset: `UMD/hansen/global_forest_change_2025_v1_13`. Hansen forest loss is *stand-replacement loss*, not selective degradation or Musanga occurrence. Annual detection algorithms changed around the Landsat 8 introduction: compare historical cohorts with care.

In [ ]:
# 5.A — Calculate Hansen × TMF joint statistics
# Runs independently of G.3/G.3C as long as their saved footprint GeoPackage exists.
from pathlib import Path
import json, tempfile, shutil
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display
from shapely.geometry import mapping
from google.colab import drive
import ee

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive', force_remount=False)
WORK_DIR=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/V3_1/NB08_WWF_Lidar')
G3_DIR=WORK_DIR/'TMF_strip_descriptors'/'rgb_footprint_pixels'
G4_DIR=WORK_DIR/'TMF_strip_descriptors'/'hansen_tmf_2014'
G4_DIR.mkdir(parents=True, exist_ok=True)
src=G3_DIR/'WWF_all_RGB_footprints_TMF_pixel_descriptors.gpkg'
assert src.exists(), 'Missing G.3 RGB footprint GeoPackage. Run G.3 first.'
strips=gpd.read_file(src).to_crs(4326).sort_values('plot_id').reset_index(drop=True)
assert len(strips)==216 and strips.plot_id.is_unique

EE_PROJECT=(globals().get('EE_PROJECT') or 'musanga-511110').strip()
try:
    ee.Initialize(project=EE_PROJECT)
    ee.Number(1).getInfo()
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

G4_BATCH=6
G4_FORCE_RECOMPUTE=False
G4_MAX_STRIPS=None    # None => all 216; set 12 for a short test run
REF_YEAR=2014         # conservative: WWF RGB flights in 2014/15, dates unknown
SCALE=30
assert REF_YEAR==2014, 'G.4 definitions explicitly refer to 2014.'
hansen=ee.Image('UMD/hansen/global_forest_change_2025_v1_13')
tmf=ee.ImageCollection('projects/JRC/TMF/v1_2025/AnnualChanges').mosaic().select('Dec2014')
land=hansen.select('datamask').eq(1)
status=tmf.toInt16()
treecover=hansen.select('treecover2000')
lossyear=hansen.select('lossyear').toInt16()
gain=hansen.select('gain').toInt16()
# Hansen 'lossyear' is 1 for 2001, ... 14 for 2014, 25 for 2025.
# 0 = no loss through 2014 (including pixels first lost AFTER 2014).
loss_cohort=(ee.Image(0).where(lossyear.gte(1).And(lossyear.lte(5)),1)
    .where(lossyear.gte(6).And(lossyear.lte(10)),2)
    .where(lossyear.gte(11).And(lossyear.lte(14)),3)).toInt16()
# Store annual 2001...2014 loss category; zero denotes no detected loss through 2014.
loss_year_asof2014=lossyear.where(lossyear.gt(14),0).toInt16()
# 2000 tree canopy bins: 1=0–19,2=20–39,3=40–59,4=60–79,5=80–100.
cover_bin=(treecover.divide(20).floor().add(1).min(5)).toInt16()
# Joint category: TMF class x10 + Hansen loss cohort; e.g. 43 = regrowth, loss 2011–14.
joint=status.multiply(10).add(loss_cohort).toInt16()
# Exclude water/no data and missing TMF classes from comparable denominators.
valid=land.And(status.gte(1)).And(status.lte(6))
cat=ee.Image.cat([
    status.rename('tmf_status'),
    cover_bin.rename('canopy2000_bin'),
    loss_cohort.rename('hansen_loss_cohort'),
    loss_year_asof2014.rename('hansen_loss_year'),
    gain.rename('hansen_gain_2000_2012'),
    joint.rename('tmf_x_hansen')
]).updateMask(valid)
cat_bands=['tmf_status','canopy2000_bin','hansen_loss_cohort',
           'hansen_loss_year','hansen_gain_2000_2012','tmf_x_hansen']
coverage=treecover.updateMask(valid).rename('canopy2000_mean')

# Cache batches so a Colab interruption does not lose completed work.
subset=strips if G4_MAX_STRIPS is None else strips.head(G4_MAX_STRIPS)
records=[]
for start in range(0,len(subset),G4_BATCH):
    chunk=subset.iloc[start:start+G4_BATCH]
    cache=G4_DIR/f'batch_{start:03d}_{start+len(chunk):03d}.json'
    if cache.exists() and not G4_FORCE_RECOMPUTE:
        part=json.loads(cache.read_text())
        records.extend(part)
        print('Reused cached batch',cache.name)
        continue
    feats=ee.FeatureCollection([
        ee.Feature(ee.Geometry(mapping(row.geometry),geodesic=False),
                   {'plot_id':str(row.plot_id)})
        for row in chunk.itertuples(index=False)
    ])
    histfc=cat.reduceRegions(collection=feats,reducer=ee.Reducer.frequencyHistogram(),
                             scale=SCALE,tileScale=4)
    meanfc=coverage.reduceRegions(collection=feats,reducer=ee.Reducer.mean(),
                                  scale=SCALE,tileScale=4)
    hrows=histfc.getInfo()['features']
    mrows=meanfc.getInfo()['features']
    means={f['properties']['plot_id']:f['properties'].get('mean') for f in mrows}
    part=[]
    for f in hrows:
        pr=f['properties'];pid=pr['plot_id']
        d={'plot_id':pid,'canopy2000_mean':means.get(pid)}
        for band in cat_bands:
            h=pr.get(band) or {}
            d[band]={str(k):int(round(v)) for k,v in h.items()}
        part.append(d)
    assert len(part)==len(chunk),'Earth Engine returned incomplete batch'
    cache.write_text(json.dumps(part))
    records.extend(part)
    print('Processed TMF × Hansen strips',start+1,'to',start+len(chunk),'/',len(subset))

assert len(records)==len(subset) and len({d['plot_id'] for d in records})==len(records)
(G4_DIR/'WWF_TMF_Hansen_all_histograms_2014.json').write_text(json.dumps(records,indent=1))
loss_labels={0:'No loss detected through 2014',1:'Loss 2001–2005',
             2:'Loss 2006–2010',3:'Loss 2011–2014'}
tmf_labels={1:'Undisturbed',2:'Degraded',3:'Deforested',
            4:'Regrowth',5:'Water',6:'Other'}
def count(h,k): return int(h.get(str(k),0))
def pct(n,d): return 100*n/d if d else float('nan')

per=[]
pooled={b:{} for b in cat_bands}
for rec in records:
    p={'plot_id':rec['plot_id'],'hansen_treecover2000_mean':rec['canopy2000_mean']}
    for b in cat_bands:
        hist=rec[b]
        for k,n in hist.items():
            pooled[b][int(k)]=pooled[b].get(int(k),0)+int(n)
    total=sum(rec['tmf_status'].values())
    p['n_common_valid_tmf_hansen_pixels']=total
    for k,label in tmf_labels.items():
        p['pct_tmf2014_'+label.lower()]=pct(count(rec['tmf_status'],k),total)
    for k in range(1,6):
        p[f'pct_hansen_canopy2000_bin{k}']=pct(count(rec['canopy2000_bin'],k),total)
    for k,label in loss_labels.items():
        p[f'pct_hansen_loss_cohort{k}']=pct(count(rec['hansen_loss_cohort'],k),total)
    for k in range(0,15):
        p[f'pct_hansen_lossyear_code{k:02d}']=pct(count(rec['hansen_loss_year'],k),total)
    p['pct_hansen_gain_2000_2012']=pct(count(rec['hansen_gain_2000_2012'],1),total)
    for cls in [1,2,3,4]:
        class_total=count(rec['tmf_status'],cls)
        for coh in range(4):
            nn=count(rec['tmf_x_hansen'],cls*10+coh)
            p[f'pct_joint_tmf{cls}_loss{coh}_of_footprint']=pct(nn,total)
            p[f'pct_hansen_loss{coh}_within_tmf{cls}']=pct(nn,class_total)
    # Regrowth–loss overlap: a diagnostic for recovery histories, not a stand-age estimator.
    p['pct_regrowth_with_loss_by2014_of_footprint']=sum(
        count(rec['tmf_x_hansen'],40+k) for k in [1,2,3])*100/max(total,1)
    per.append(p)

per=pd.DataFrame(per)
strips_merged=strips.merge(per,on='plot_id',validate='one_to_one')
out_csv=G4_DIR/'WWF_all_strips_TMF_Hansen_2014_descriptors.csv'
out_gpkg=G4_DIR/'WWF_all_strips_TMF_Hansen_2014_descriptors.gpkg'
strips_merged.drop(columns='geometry').to_csv(out_csv,index=False)
strips_merged.to_file(out_gpkg,driver='GPKG')
print('Saved',out_csv)
print('Saved',out_gpkg)

# Pool HISTOGRAM COUNTS, not percentages: every common valid TMF/Hansen observation
# has equal weight in these tables. Avoid summing strip %s.
pooled_long=[]
for band,values in pooled.items():
    denominator=sum(values.values())
    for key,n in sorted(values.items()):
        pooled_long.append(dict(band=band,value=key,count=n,
                                pct=100*n/denominator if denominator else np.nan,
                                total=denominator))
pooled_df=pd.DataFrame(pooled_long)
pooled_df.to_csv(G4_DIR/'WWF_TMF_Hansen_2014_pooled_histograms.csv',index=False)

cross=pd.DataFrame(0,dtype=int,index=[tmf_labels[i] for i in range(1,7)],
                   columns=[loss_labels[j] for j in range(4)])
for k,n in pooled['tmf_x_hansen'].items():
    cls,coh=divmod(k,10)
    if cls in tmf_labels and coh in loss_labels:
        cross.loc[tmf_labels[cls],loss_labels[coh]]+=n
cross.to_csv(G4_DIR/'WWF_TMF_Hansen_2014_joint_counts.csv')
cross_pct=100*cross/cross.values.sum()
cross_pct.to_csv(G4_DIR/'WWF_TMF_Hansen_2014_joint_pct_all_pixels.csv')
rowpct=cross.div(cross.sum(axis=1).replace(0,np.nan),axis=0)*100
rowpct.to_csv(G4_DIR/'WWF_TMF_Hansen_2014_joint_pct_within_tmf_class.csv')

# Across-strip descriptive statistics; equal strip weighting, *not pooled pixels*.
metrics=[c for c in per.columns if c.startswith('pct_') or c=='hansen_treecover2000_mean']
summ=[]
for c in metrics:
    x=pd.to_numeric(per[c],errors='coerce').dropna()
    if not len(x): continue
    q=x.quantile([.10,.25,.50,.75,.90,.95])
    summ.append(dict(metric=c,n=len(x),mean=x.mean(),std=x.std(),
                     min=x.min(),p10=q.loc[.1],p25=q.loc[.25],median=q.loc[.5],
                     p75=q.loc[.75],p90=q.loc[.9],p95=q.loc[.95],max=x.max(),
                     zero_pct=100*(x==0).mean()))
summ=pd.DataFrame(summ)
summ.to_csv(G4_DIR/'WWF_TMF_Hansen_2014_across_strip_stats.csv',index=False)

# Dedicated readable global summary (including conditional regrowth overlap).
global_n=int(cross.values.sum())
global_stats=[]
for name,nums in [
    ('Common valid 30 m pixel observations',global_n),
    ('TMF regrowth pixels',int(cross.loc['Regrowth'].sum())),
    ('TMF regrowth with Hansen loss 2001–2014',int(cross.loc['Regrowth'].iloc[1:].sum())),
    ('TMF regrowth with Hansen loss 2011–2014',int(cross.loc['Regrowth'].iloc[3])),
    ('Hansen loss 2011–2014 in any TMF status',int(cross.iloc[:,3].sum())),
    ('Hansen loss 2001–2014 in any TMF status',int(cross.iloc[:,1:].values.sum())),
]:
    global_stats.append(dict(description=name,pixels=nums,
                             pct_all=100*nums/global_n if global_n else np.nan))
global_frame=pd.DataFrame(global_stats)
global_frame.to_csv(G4_DIR/'WWF_TMF_Hansen_2014_global_summary.csv',index=False)

print('\n=== GLOBAL pooled Hansen × TMF pixels, 2014 reference ===')
display(global_frame.round(2))
display(cross)
print('\n=== PERCENT OF EACH TMF CLASS BY HANSEN LOSS COHORT ===')
display(rowpct.round(2))
print('\n=== BETWEEN-STRIP variability (selected indicators) ===')
keys=['pct_hansen_loss_cohort3','pct_hansen_loss_cohort2',
      'pct_hansen_gain_2000_2012','pct_tmf2014_regrowth',
      'pct_regrowth_with_loss_by2014_of_footprint',
      'hansen_treecover2000_mean']
display(summ[summ.metric.isin(keys)].round(2))
print('\nTop WWF strips with overlap of regrowth and pre-2015 Hansen forest loss')
display(per.sort_values('pct_regrowth_with_loss_by2014_of_footprint',
                        ascending=False)[['plot_id','pct_tmf2014_regrowth',
                        'pct_regrowth_with_loss_by2014_of_footprint',
                        'pct_hansen_loss_cohort3','pct_hansen_loss_cohort2',
                        'pct_hansen_gain_2000_2012']].head(20).round(2))

# Plots: write to temporary LOCAL filesystem before copying to mounted Drive.
local=Path(tempfile.mkdtemp(prefix='wwf_hansen_tmf_'))
def publish(fig,filename):
    destination=G4_DIR/filename
    source=local/filename
    fig.savefig(source,dpi=110,bbox_inches='tight')
    plt.close(fig)
    shutil.copyfile(source,destination)
    print('Saved figure:',destination)
    display(IPImage(filename=str(source),width=1000))
fig,axs=plt.subplots(1,2,figsize=(13,5))
cross_pct.plot(kind='bar',stacked=True,ax=axs[0],width=.75)
axs[0].set(title='TMF 2014 classes by Hansen loss cohort (pooled)',
           xlabel='TMF 2014 class',ylabel='Percent of all joint pixels')
axs[0].tick_params(axis='x',rotation=40)
rowpct.plot(kind='bar',stacked=True,ax=axs[1],width=.75)
axs[1].set(title='Within each TMF class: Hansen loss cohorts',
           xlabel='TMF 2014 class',ylabel='% of TMF class')
axs[1].tick_params(axis='x',rotation=40)
fig.tight_layout();publish(fig,'TMF_Hansen_cross_class_composition.png')

fig,axs=plt.subplots(2,2,figsize=(13,9))
plot_keys=['pct_hansen_loss_cohort3','pct_tmf2014_regrowth',
           'pct_regrowth_with_loss_by2014_of_footprint',
           'hansen_treecover2000_mean']
for ax,key in zip(axs.flat,plot_keys):
    values=pd.to_numeric(per[key],errors='coerce').dropna()
    ax.hist(values,bins=30,edgecolor='white')
    ax.axvline(values.median(),ls='--',color='black')
    ax.set(title=key.replace('_',' '),xlabel='Percent (or 2000 canopy-cover %)',
           ylabel='Number of WWF strips')
fig.tight_layout();publish(fig,'TMF_Hansen_recovery_gradient_strip_distributions.png')

fig,ax=plt.subplots(figsize=(11,5))
loss_values=pooled['hansen_loss_year']
years=np.arange(2001,2015)
counts=np.array([loss_values.get(int(y-2000),0) for y in years])
ax.bar(years,counts)
ax.set(title='Hansen annual stand-replacing forest loss before WWF RGB acquisition',
       xlabel='Loss year (2001–2014)',ylabel='30 m mapped pixel observations')
ax.set_xticks(years)
ax.tick_params(axis='x',labelrotation=45)
fig.tight_layout();publish(fig,'Hansen_annual_loss_2001_2014.png')

fig,ax=plt.subplots(figsize=(10,5))
cover_counts=[pooled['canopy2000_bin'].get(i,0) for i in range(1,6)]
ax.bar(['0–19%','20–39%','40–59%','60–79%','80–100%'],cover_counts)
ax.set(title='Hansen year-2000 canopy cover across WWF RGB footprints',
       ylabel='Common valid TMF/Hansen 30 m pixel observations',xlabel='Canopy cover 2000')
fig.tight_layout();publish(fig,'Hansen_canopy2000_global_distribution.png')

print('\nG.4 DONE. Output directory:',G4_DIR)
print('IMPORTANT: this is a description of WWF sampling opportunities, NOT a')
print('population-unbiased forest area estimate and NOT Musanga occurrence evidence.')
print('TMF/Hansen pixels need native WWF RGB inspection before choosing circles.')


## 6 — Integrated ecological assessment

**Scientific goal:** evaluate which disturbance/recovery environments are represented in the WWF RGB survey, their global abundance, distribution across **independent strips**, and potential usefulness for Musanga annotations. G.4 offers underlying paired TMF/Hansen data; G.5 assigns an **explicit, mutually exclusive ecological sampling stratum to each 30 m pixel**, then compares strips. A second, nonexclusive **regrowth age × Hansen loss-cohort** cross-tabulation preserves important historical information.

**Sampling-stratum rules (ordered precedence):**
1. **Young TMF regrowth** (1–3 consecutive years in TMF regrowth state as of 2014), whether Hansen detects forest loss or not.
2. **Intermediate regrowth** (4–7 years).
3. **Longer TMF regrowth** (8+ years).
4. **TMF degraded without Hansen loss through 2014** (possible canopy disturbance not detected as stand-replacing loss).
5. **TMF degraded with Hansen loss 2001–2014** (stronger/mixed disturbance evidence).
6. **Recent Hansen loss 2011–2014**, excluding already-assigned regrowth/degraded areas (not assumed to have recovered).
7. **Historical Hansen loss 2001–2010**, excluding already-assigned classes (not assumed to be secondary forest).
8. **Stable high-canopy reference** (TMF undisturbed, no Hansen loss through 2014, Hansen 2000 canopy cover ≥80%).
9. **Stable lower-canopy reference** (TMF undisturbed, no Hansen loss through 2014, Hansen 2000 canopy cover <80%).
10. **Other / ambiguous TMF–Hansen combinations**, including open/other/water and anomalies.

**Interpretation safeguards:** These are **operational sampling environments**, **not measured tree ages**. TMF regrowth-state duration is not stand age. Hansen 2000 canopy cover is a historical baseline and *not* 2014 canopy closure. Hansen 'no loss' means no *detected stand-replacing loss before 2015* (not proof of no disturbance). The G.5 classification is applied to valid Hansen land pixels and valid TMF status; others are excluded.

**Statistics and outputs:** pooled counts and fractions by stratum, equal-weight strip statistics (mean, SD, quantiles, share of strips with zero), number of strips exceeding **0.5%, 1%, 2%, 5%, 10%** of a stratum, top strips for every environment, cross-tabulation of regrowth-state duration × Hansen loss cohort, CSV/GPKG and comparative figures. A separate **G.5B** checks 125 m circle-neighbourhood composition (>=70% target stratum), respecting interior survey boundaries: this measures *candidate centre pixels*, not independent annotation circles. Processing uses saved WWF footprint polygons, GEE batch caching, and no WWF RGB downloads.

In [ ]:
# 6.A — Classify integrated ecological strata
import json, tempfile, shutil
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display
from shapely.geometry import mapping
from google.colab import drive
import ee

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive',force_remount=False)
WORK_DIR=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/V3_1/NB08_WWF_Lidar')
G3_DIR=WORK_DIR/'TMF_strip_descriptors'/'rgb_footprint_pixels'
G5_DIR=WORK_DIR/'TMF_strip_descriptors'/'integrated_ecological_gradient_2014'
G5_DIR.mkdir(parents=True,exist_ok=True)
source=G3_DIR/'WWF_all_RGB_footprints_TMF_pixel_descriptors.gpkg'
assert source.exists(), 'G.3 potential RGB footprint GeoPackage missing'
foot=gpd.read_file(source).to_crs(4326).sort_values('plot_id').reset_index(drop=True)
assert len(foot)==216 and foot.plot_id.is_unique
EE_PROJECT=(globals().get('EE_PROJECT') or 'musanga-511110').strip()
try:
    ee.Initialize(project=EE_PROJECT)
    ee.Number(1).getInfo()
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

G5_BATCH=6
G5_FORCE=False
G5_TEST_N=None   # e.g. 12 for short testing; None means all 216
G5_SCALE=30
# Distinguish useful ecological environments rather than claiming stand ages.
classes={
    1:'Young TMF regrowth (1–3 yr state)',
    2:'Intermediate TMF regrowth (4–7 yr state)',
    3:'Longer TMF regrowth (8+ yr state)',
    4:'TMF degraded / no Hansen loss through 2014',
    5:'TMF degraded / Hansen loss by 2014',
    6:'Recent Hansen loss (2011–14), other TMF states',
    7:'Historical Hansen loss (2001–10), other TMF states',
    8:'Undisturbed / high canopy in 2000 / no Hansen loss',
    9:'Undisturbed / lower canopy in 2000 / no Hansen loss',
    10:'Other or ambiguous combinations'
}
# Easy-to-read immutable codebook for downstream app/sampling.
(G5_DIR/'G5_ecological_class_codebook.json').write_text(json.dumps(classes,indent=2))
h=ee.Image('UMD/hansen/global_forest_change_2025_v1_13')
tmfstack=ee.ImageCollection('projects/JRC/TMF/v1_2025/AnnualChanges').mosaic()
st=tmfstack.select('Dec2014').toInt16()
ly=h.select('lossyear').toInt16()
c00=h.select('treecover2000').toInt16()
valid=h.select('datamask').eq(1).And(st.gte(1)).And(st.lte(6))
loss=(ee.Image(0).where(ly.gte(1).And(ly.lte(5)),1)
    .where(ly.gte(6).And(ly.lte(10)),2)
    .where(ly.gte(11).And(ly.lte(14)),3)).toInt16()
age=ee.Image.constant(0).toInt16()
for y in range(1990,2015):
    cur=tmfstack.select('Dec'+str(y))
    age=age.add(1).multiply(cur.eq(4)).toInt16()
reg=st.eq(4)
# Precedence: specific regrowth age -> degraded -> loss events -> stable references.
eco=ee.Image(10).toInt16()
eco=eco.where(st.eq(1).And(loss.eq(0)).And(c00.gte(80)),8)
eco=eco.where(st.eq(1).And(loss.eq(0)).And(c00.lt(80)),9)
eco=eco.where(loss.gte(1).And(loss.lte(2)),7)
eco=eco.where(loss.eq(3),6)
eco=eco.where(st.eq(2).And(loss.eq(0)),4)
eco=eco.where(st.eq(2).And(loss.gt(0)),5)
eco=eco.where(reg.And(age.gte(1)).And(age.lte(3)),1)
eco=eco.where(reg.And(age.gte(4)).And(age.lte(7)),2)
eco=eco.where(reg.And(age.gte(8)),3)
eco=eco.updateMask(valid).rename('ecological_stratum')
# Nonexclusive diagnostics: all combinations of age group and Hansen loss cohort.
agegrp=(ee.Image(0).where(reg.And(age.gte(1)).And(age.lte(3)),1)
       .where(reg.And(age.gte(4)).And(age.lte(7)),2)
       .where(reg.And(age.gte(8)),3)).toInt16()
reg_cross=(agegrp.multiply(10).add(loss)).updateMask(reg.And(valid)).rename('regrowth_age_x_loss')
all_bands=ee.Image.cat(eco,reg_cross)
selection=foot if G5_TEST_N is None else foot.head(G5_TEST_N)
data=[]
for begin in range(0,len(selection),G5_BATCH):
    part=selection.iloc[begin:begin+G5_BATCH]
    cache=G5_DIR/f'g5_hist_{begin:03d}_{begin+len(part):03d}.json'
    if cache.exists() and not G5_FORCE:
        rows=json.loads(cache.read_text())
        data.extend(rows)
        print('Cached',cache.name)
        continue
    fc=ee.FeatureCollection([
        ee.Feature(ee.Geometry(mapping(row.geometry),geodesic=False),
                   {'plot_id':str(row.plot_id)}) for row in part.itertuples(index=False)])
    response=all_bands.reduceRegions(collection=fc,
        reducer=ee.Reducer.frequencyHistogram(),scale=G5_SCALE,
        tileScale=4).getInfo()['features']
    rows=[]
    for f in response:
        p=f['properties']
        rows.append(dict(plot_id=p['plot_id'],
            strata={str(k):int(round(v)) for k,v in
                (p.get('ecological_stratum') or {}).items()},
            regrowth_age_x_loss={str(k):int(round(v)) for k,v in
                (p.get('regrowth_age_x_loss') or {}).items()}))
    assert len(rows)==len(part)
    cache.write_text(json.dumps(rows))
    data.extend(rows)
    print('Processed',begin+1,'–',begin+len(part),'/',len(selection))
assert len(data)==len(selection) and len({x['plot_id'] for x in data})==len(data)
(G5_DIR/'WWF_G5_all_strips_pixel_histograms.json').write_text(json.dumps(data,indent=1))
global_count={k:0 for k in classes}
global_age_loss=np.zeros((3,4),dtype=np.int64)
strip_rows=[]
for d in data:
    n=sum(d['strata'].values())
    row=dict(plot_id=d['plot_id'],n_common_valid_pixels=n)
    for k in classes:
        count=int(d['strata'].get(str(k),0))
        global_count[k]+=count
        row['count_stratum_'+str(k)]=count
        row['pct_stratum_'+str(k)]=100*count/n if n else np.nan
    cross=d['regrowth_age_x_loss']
    for a in range(1,4):
        for l in range(4):
            v=int(cross.get(str(a*10+l),0))
            global_age_loss[a-1,l]+=v
            row[f'count_reg_age{a}_hansen_loss{l}']=v
    strip_rows.append(row)
stripdata=pd.DataFrame(strip_rows)
full=foot.merge(stripdata,on='plot_id',validate='one_to_one')
full.drop(columns='geometry').to_csv(G5_DIR/'WWF_G5_integrated_strata_per_strip.csv',index=False)
full.to_file(G5_DIR/'WWF_G5_integrated_strata_per_strip.gpkg',driver='GPKG')
pooled_n=sum(global_count.values())
global_table=pd.DataFrame([dict(code=k,stratum=name,pixels=global_count[k],
    pct_of_valid_pixels=100*global_count[k]/pooled_n if pooled_n else np.nan,
    pixel_area_ha_nominal=global_count[k]*0.09) for k,name in classes.items()])
global_table.to_csv(G5_DIR/'WWF_G5_global_integrated_strata.csv',index=False)
age_matrix=pd.DataFrame(global_age_loss,
    index=['Regrowth state 1–3y','Regrowth state 4–7y','Regrowth state 8+y'],
    columns=['No Hansen loss through 2014','Loss 2001–05','Loss 2006–10','Loss 2011–14'])
age_matrix.to_csv(G5_DIR/'WWF_G5_regrowth_age_x_hansen_loss_counts.csv')
(age_matrix.div(age_matrix.sum(axis=1),axis=0)*100).to_csv(
    G5_DIR/'WWF_G5_regrowth_age_x_hansen_loss_within_age_pct.csv')
stat=[]
for k,name in classes.items():
    z=stripdata['pct_stratum_'+str(k)].dropna()
    q=z.quantile([.1,.25,.5,.75,.9,.95])
    stat.append(dict(code=k,stratum=name,n=len(z),mean=z.mean(),sd=z.std(),
        p10=q.loc[.1],p25=q.loc[.25],median=q.loc[.5],p75=q.loc[.75],
        p90=q.loc[.9],p95=q.loc[.95],maximum=z.max(),pct_strips_zero=100*(z==0).mean(),
        strips_ge_0_5pct=int((z>=.5).sum()),strips_ge_1pct=int((z>=1).sum()),
        strips_ge_2pct=int((z>=2).sum()),strips_ge_5pct=int((z>=5).sum()),
        strips_ge_10pct=int((z>=10).sum())))
stat=pd.DataFrame(stat)
stat.to_csv(G5_DIR/'WWF_G5_stratum_replication_across_strips.csv',index=False)
ranked=[]
for k,name in classes.items():
    for rank,(idx,rr) in enumerate(stripdata.sort_values('pct_stratum_'+str(k),
                                                ascending=False).head(15).iterrows(),1):
        ranked.append(dict(code=k,stratum=name,rank=rank,plot_id=rr.plot_id,
                           share_pct=rr['pct_stratum_'+str(k)],
                           n_pixels=rr['count_stratum_'+str(k)]))
pd.DataFrame(ranked).to_csv(G5_DIR/'WWF_G5_top15_strips_each_stratum.csv',index=False)
print('G5 — POOLED GLOBAL DISTRIBUTION')
display(global_table.round(2))
print('G5 — BETWEEN-STRIP REPLICATION')
display(stat.round(2))
print('G5 — REGROWTH STATE AGE × HANSEN LOSS COHORT (PIXEL COUNTS)')
display(age_matrix)
print('Row-normalized % within each TMF regrowth-state age')
display((age_matrix.div(age_matrix.sum(axis=1),axis=0)*100).round(2))

# Compact figures. Render locally before copying to mounted Drive.
tmp=Path(tempfile.mkdtemp(prefix='g5_wwf_'))
def figure_out(fig,name):
    src=tmp/name
    fig.savefig(src,dpi=110,bbox_inches='tight')
    plt.close(fig)
    shutil.copyfile(src,G5_DIR/name)
    display(IPImage(filename=str(src),width=1000))
fig,axes=plt.subplots(1,2,figsize=(15,5))
axes[0].bar(global_table['code'].astype(str),global_table['pct_of_valid_pixels'])
axes[0].set(title='All WWF RGB footprints: share of each ecological stratum',
            xlabel='Ecological stratum code (see table)',ylabel='% of valid TMF×Hansen pixels')
axes[1].bar(stat['code'].astype(str),stat['strips_ge_1pct'])
axes[1].set(title='Representation: number of WWF strips ≥1% in each stratum',
            xlabel='Ecological stratum code',ylabel='Number of strips (of 216)')
fig.tight_layout()
figure_out(fig,'G5_global_composition_and_replication.png')
fig,ax=plt.subplots(figsize=(11,4.5))
im=ax.imshow(age_matrix.to_numpy(),aspect='auto',cmap='viridis')
ax.set_xticks(range(4),age_matrix.columns,rotation=25,ha='right')
ax.set_yticks(range(3),age_matrix.index)
for y in range(3):
    for x in range(4):
        ax.text(x,y,f'{age_matrix.iloc[y,x]:,}',ha='center',va='center',fontsize=9,
                color='white' if age_matrix.iloc[y,x]>(age_matrix.to_numpy().max()*.5) else 'black')
ax.set(title='TMF regrowth-state duration × Hansen forest-loss history')
fig.colorbar(im,ax=ax,label='30 m pixel observations')
fig.tight_layout()
figure_out(fig,'G5_regrowth_age_by_hansen_loss_heatmap.png')
# Distribution of ecological gradients by geography (rank strip rows, no geodata download).
fig,ax=plt.subplots(figsize=(11,7))
heat=stripdata.set_index('plot_id')[[f'pct_stratum_{k}' for k in classes]]
order=heat.sort_values(['pct_stratum_1','pct_stratum_2','pct_stratum_3'],
                       ascending=False).index
arr=heat.loc[order].to_numpy()
im=ax.imshow(arr,aspect='auto',interpolation='nearest',cmap='viridis',
             vmin=0,vmax=np.nanpercentile(arr,98))
ax.set_xticks(range(len(classes)),[str(k) for k in classes])
ax.set_yticks(np.arange(0,len(heat),15),order[::15],fontsize=7)
ax.set(xlabel='Ecological stratum code',title='216 WWF strips: disturbance/recovery gradient (sorted by regrowth)')
fig.colorbar(im,ax=ax,label='% of strip footprint')
fig.tight_layout()
figure_out(fig,'G5_all_strips_ecological_composition_heatmap.png')
print('G.5 finished, data and figures:',G5_DIR)
print('Caution: none of these classes establish true stand age or Musanga occurrence.')
print('G.5B can optionally test whether each stratum occupies coherent 125 m neighbourhoods.')


### 6.B — 125 m neighbourhood feasibility

Run **after G.5** if you want to know whether the ecological strata occupy patches large enough for the existing **125 m diameter** annotation circles. On a 30 m working grid, test **candidate centre pixels** where at least 70% of a 62.5 m radius neighbourhood belongs to the target ecological stratum; the centre must be at least 62.5 m inside the *potential* RGB footprint. The resulting candidate counts are **correlated neighbouring centres, not independent 125 m circles**; they provide a feasibility screen only. Final locations still require valid-RGB/no-data verification and spatial separation.

The class image uses exactly the same 2014 TMF/Hansen rules as G.5. This is a separate (potentially more computationally intensive) Earth Engine query, cached in small batches.

In [ ]:
# 6.B.1 — Test 125 m circular neighbourhood feasibility
# Run directly after G.5 in same runtime: reuses ee image 'eco', footprints 'foot' and 'G5_DIR'.
assert 'eco' in globals() and 'foot' in globals() and 'G5_DIR' in globals(), 'Run G.5 first.'
import json, numpy as np, pandas as pd
from shapely.geometry import mapping
G5B_BATCH=3
G5B_FORCE=False
CIRCLE_RADIUS_M=62.5
TARGET_FRACTION=.70
TARGET_CLASSES=[1,2,3,4,5,6,7,8,9]
# A 125 m disk radius is 62.5 m; running mean across its 30 m TMF samples.
# Because eco is masked beyond valid forest pixels, unmask(0) before neighborhood.
# The threshold is applied against the WHOLE circular neighbourhood, not
# conditional on already classified pixels.
kern=ee.Kernel.circle(radius=CIRCLE_RADIUS_M,units='meters',normalize=False)
coherent=ee.Image.cat([
    eco.eq(k).unmask(0).reduceNeighborhood(
        reducer=ee.Reducer.mean(),kernel=kern,skipMasked=False
    ).gte(TARGET_FRACTION).rename('candidate_'+str(k)).toUint8()
    for k in TARGET_CLASSES
])
records=[]
for start in range(0,len(foot),G5B_BATCH):
    chunk=foot.iloc[start:start+G5B_BATCH]
    fn=G5_DIR/f'circle_centres_{start:03d}_{start+len(chunk):03d}.json'
    if fn.exists() and not G5B_FORCE:
        records.extend(json.loads(fn.read_text()))
        print('Cached',fn.name)
        continue
    feats=[]
    for row in chunk.itertuples(index=False):
        geo=ee.Geometry(mapping(row.geometry),geodesic=False)
        # Avoid counting candidate centres whose 125 m circle crosses survey boundary.
        inner=geo.buffer(-CIRCLE_RADIUS_M,maxError=5)
        feats.append(ee.Feature(inner,{'plot_id':str(row.plot_id)}))
    result=coherent.reduceRegions(
        collection=ee.FeatureCollection(feats),reducer=ee.Reducer.sum(),
        scale=30,tileScale=4).getInfo()['features']
    rows=[]
    for f in result:
        p=f['properties']
        d={'plot_id':p['plot_id']}
        for k in TARGET_CLASSES:
            value=p.get('candidate_'+str(k))
            d['candidate_centres_k'+str(k)]=int(round(value or 0))
        rows.append(d)
    assert len(rows)==len(chunk)
    fn.write_text(json.dumps(rows))
    records.extend(rows)
    print('Processed 125m neighbourhoods',start+1,'–',start+len(chunk),'/',len(foot))
assert len(records)==216
screen=pd.DataFrame(records)
screen.to_csv(G5_DIR/'WWF_G5_125m_circle_candidate_centre_counts.csv',index=False)
availability=[]
for k in TARGET_CLASSES:
    values=screen['candidate_centres_k'+str(k)]
    availability.append(dict(stratum_code=k,environment=classes[k],
       strips_with_any_eligible_centres=int((values>0).sum()),
       total_candidate_centre_pixels=int(values.sum()),
       maximum_centres_in_one_strip=int(values.max())))
display(pd.DataFrame(availability))
print('Saved candidate-centre counts; NOT unique or independent circles.')
print('A selected centre still requires checking true 10 cm RGB validity.')


## 7 — Ecologically stratified annotation-circle selection

From G.5/G.5B to a **reproducible, spatially balanced candidate sampling plan** for the WWF RGB data.

- Start with **rare regrowth classes (1–3)**, protecting geographic replication by limiting them initially to **one centre per stratum per WWF strip**. Then sample degradation and Hansen-loss classes (4–7), followed by preferentially **paired stable-forest controls (8)** in the same strips, and a small number of lower-canopy references (9).
- A **70%-pure 125 m-diameter disk** and a centre at least 62.5 m inside the potential RGB footprint are necessary for consideration (same principle as G.5B). Centres are chosen from stratified Earth Engine samples of eligible 30 m pixels, **not randomly from the strip**.
- Enforce **minimum 250 m distance between all selected centres** (geodesic), even if they belong to different strips/classes. The output is a realistic candidate design, not an inflated count of correlated pixels.
- Use a **deterministic seed** and selection targets; do not force rare targets where actual eligible places are absent. Export individual circle polygons, centres, CSV tables, per-stratum diagnostics, GeoPackage and map. Also save all screened points for revisiting the selection without rerunning Earth Engine.
- **Critical:** TMF/Hansen states are *hypotheses for sampling*, not Musanga labels, and TMF consecutive regrowth-state years are **not tree ages**. The footprint remains *survey polygon ∩ RGB file bounds*; selected circles are **provisional until verified against actual nonblack RGB coverage**. No circles are published to shared assets automatically.

Run **G.6A** (sampling and selection), then optionally **G.6B** (native RGB validity checks). Both have cached outputs; no full RGB mosaic download is required.

In [ ]:
# 7.A — Select spatially separated 125 m circles
# Recreates the G.5 ecological map (standalone in a fresh Colab runtime).
import json, math, hashlib, tempfile, shutil
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd, geopandas as gpd
from shapely.geometry import Point, mapping
from pyproj import Geod
from google.colab import drive
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage
import ee

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive',force_remount=False)
WORK_DIR=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/V3_1/NB08_WWF_Lidar')
G3_DIR=WORK_DIR/'TMF_strip_descriptors'/'rgb_footprint_pixels'
G5_DIR=WORK_DIR/'TMF_strip_descriptors'/'integrated_ecological_gradient_2014'
G6_DIR=WORK_DIR/'TMF_strip_descriptors'/'G6_125m_sampling_design'
G6_DIR.mkdir(parents=True,exist_ok=True)
src=G3_DIR/'WWF_all_RGB_footprints_TMF_pixel_descriptors.gpkg'
assert src.exists(), 'Run G.3 to create WWF potential RGB footprint polygons first.'
foot=gpd.read_file(src).to_crs(4326).sort_values('plot_id').reset_index(drop=True)
assert len(foot)==216 and foot.plot_id.is_unique
assert (G5_DIR/'WWF_G5_125m_circle_candidate_centre_counts.csv').exists(), \
    'Run G.5B first to verify feasible 125m neighbourhoods.'
g5counts=pd.read_csv(G5_DIR/'WWF_G5_125m_circle_candidate_centre_counts.csv').set_index('plot_id')

EE_PROJECT=(globals().get('EE_PROJECT') or 'musanga-511110').strip()
try:
    ee.Initialize(project=EE_PROJECT)
    ee.Number(1).getInfo()
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

# Reproducible sampling controls
G6_SEED=20261028
CIRCLE_DIAMETER_M=125
RADIUS_M=CIRCLE_DIAMETER_M/2
NEIGHBOURHOOD_MIN_PURITY=.70
MIN_CENTRE_SEPARATION_M=250  # circle edges then have 125 m between them
MAX_PER_STRIP_TOTAL=5
MAX_PER_STRIP_EACH_STRATUM=2
MAX_PER_STRIP_RARE=1        # for each of 1,2,3
EE_CANDIDATES_PER_STRATUM_PER_STRIP=45
G6_REBUILD_CANDIDATE_CACHE=False
G6_TEST_N=None              # optional first 12 strips, or None for all 216
# Aspirational numbers; actual feasibility and spatial separation take precedence.
G6_TARGETS={1:7,2:10,3:15,4:16,5:12,6:12,7:12,8:22,9:8}
labels={
    1:'Young regrowth (TMF 1–3 yr state)',
    2:'Intermediate regrowth (TMF 4–7 yr state)',
    3:'Longer regrowth (TMF 8+ yr state)',
    4:'Degraded / no Hansen loss',
    5:'Degraded / Hansen loss',
    6:'Recent Hansen loss (2011–14)',
    7:'Historical Hansen loss (2001–10)',
    8:'Stable high-canopy forest reference',
    9:'Stable lower-canopy forest reference'
}
# Recreate EXACT priority rules of G.5
h=ee.Image('UMD/hansen/global_forest_change_2025_v1_13')
tmf=ee.ImageCollection('projects/JRC/TMF/v1_2025/AnnualChanges').mosaic()
st=tmf.select('Dec2014').toInt16()
ly=h.select('lossyear').toInt16()
c00=h.select('treecover2000')
valid=h.select('datamask').eq(1).And(st.gte(1)).And(st.lte(6))
loss=(ee.Image(0).where(ly.gte(1).And(ly.lte(5)),1)
    .where(ly.gte(6).And(ly.lte(10)),2)
    .where(ly.gte(11).And(ly.lte(14)),3)).toInt16()
age=ee.Image(0).toInt16()
for year in range(1990,2015):
    age=age.add(1).multiply(tmf.select('Dec'+str(year)).eq(4)).toInt16()
eco=ee.Image(10).toInt16()
eco=eco.where(st.eq(1).And(loss.eq(0)).And(c00.gte(80)),8)
eco=eco.where(st.eq(1).And(loss.eq(0)).And(c00.lt(80)),9)
eco=eco.where(loss.gte(1).And(loss.lte(2)),7)
eco=eco.where(loss.eq(3),6)
eco=eco.where(st.eq(2).And(loss.eq(0)),4)
eco=eco.where(st.eq(2).And(loss.gt(0)),5)
eco=eco.where(st.eq(4).And(age.gte(1)).And(age.lte(3)),1)
eco=eco.where(st.eq(4).And(age.gte(4)).And(age.lte(7)),2)
eco=eco.where(st.eq(4).And(age.gte(8)),3)
eco=eco.updateMask(valid)

# Build all 9 local-purity surfaces; precedence is rare-first.
kernel=ee.Kernel.circle(radius=RADIUS_M,units='meters',normalize=False)
eligible_code=ee.Image(0).toInt16()
candidate_purity=ee.Image(0).toInt16()
for k in range(1,10):
    purity=(eco.eq(k).unmask(0).reduceNeighborhood(
        reducer=ee.Reducer.mean(),kernel=kernel,skipMasked=False))
    eligible=purity.gte(NEIGHBOURHOOD_MIN_PURITY).And(eligible_code.eq(0))
    eligible_code=eligible_code.where(eligible,k)
    candidate_purity=candidate_purity.where(eligible,purity.multiply(100).round())
candidate_img=ee.Image.cat(eligible_code.rename('eligible_code'),
                           candidate_purity.rename('purity_pct')).updateMask(eligible_code.gt(0))
sample_foot=foot if G6_TEST_N is None else foot.head(G6_TEST_N)
cache_dir=G6_DIR/'EE_candidate_samples'
cache_dir.mkdir(parents=True,exist_ok=True)
all_candidates=[]
errors=[]
for number,row in enumerate(sample_foot.itertuples(index=False),1):
    pid=str(row.plot_id)
    cache=cache_dir/(pid+'.json')
    if cache.exists() and not G6_REBUILD_CANDIDATE_CACHE:
        samples=json.loads(cache.read_text())
        all_candidates.extend(samples)
        if number%24==0: print('Cached centre samples:',number,'/',len(sample_foot))
        continue
    if not any(g5counts.loc[pid,'candidate_centres_k'+str(k)]>0 for k in range(1,10)):
        cache.write_text('[]')
        continue
    try:
        inner=ee.Geometry(mapping(row.geometry),geodesic=False).buffer(-RADIUS_M,maxError=5)
        fc=candidate_img.stratifiedSample(
            numPoints=0,classBand='eligible_code',classValues=list(range(1,10)),
            classPoints=[EE_CANDIDATES_PER_STRATUM_PER_STRIP]*9,
            region=inner,scale=30,seed=G6_SEED,dropNulls=True,
            geometries=True,tileScale=4)
        feats=fc.getInfo()['features']
        samples=[]
        for f in feats:
            props=f['properties']
            lon,lat=f['geometry']['coordinates']
            k=int(props['eligible_code'])
            samples.append(dict(plot_id=pid,utm_zone=str(row.utm_zone),
                                stratum=k,longitude=float(lon),latitude=float(lat),
                                purity_pct=float(props.get('purity_pct') or 0)))
        cache.write_text(json.dumps(samples))
        all_candidates.extend(samples)
        if number%12==0:print('Sampled eligible points:',number,'/',len(sample_foot))
    except Exception as ex:
        errors.append((pid,str(ex)[:220]))
        print('WARNING: candidate sampling failed:',pid,str(ex)[:170])
if errors:
    (G6_DIR/'EE_sample_errors.json').write_text(json.dumps(errors,indent=2))
    print('WARNING:',len(errors),'strip requests failed; do not treat this as exhaustive.')
assert all_candidates, 'No candidate points obtained. Inspect Earth Engine sampling errors.'
candidates=pd.DataFrame(all_candidates)
candidates.to_csv(G6_DIR/'WWF_G6_screened_candidate_centres.csv',index=False)
print('Total screened candidate centres:',len(candidates),
      'across',candidates.plot_id.nunique(),'WWF strips.')

# Geodesic separation among ALL selected centres, even across strip borders.
geod=Geod(ellps='WGS84')
rng=np.random.default_rng(G6_SEED)
candidates['tiebreak']=rng.random(len(candidates))
chosen=[]
strip_n=Counter()
per_pair_n=Counter()
zones_by_class=defaultdict(Counter)
selected_ids=set()
def far_enough(row):
    for q in chosen:
        # Cheap bounding-box guard: never skip actual nearby points.
        if abs(row.latitude-q['latitude'])>.01 or abs(row.longitude-q['longitude'])>.02:
            continue
        _,_,dist=geod.inv(float(row.longitude),float(row.latitude),
                          float(q['longitude']),float(q['latitude']))
        if dist<MIN_CENTRE_SEPARATION_M:return False
    return True
def add_stratum(k,matched_controls=False):
    opts=candidates[candidates.stratum.eq(k)].copy()
    if matched_controls:
        previous={q['plot_id'] for q in chosen if q['stratum']!=8}
        opts['paired_first']=opts.plot_id.isin(previous).astype(int)
        opts=opts.sort_values(['paired_first','purity_pct','tiebreak'],
                              ascending=[False,False,True])
    else:
        opts=opts.sort_values(['purity_pct','tiebreak'],ascending=[False,True])
    # Spatially distribute across strips/zones in rounds, avoiding a single high-purity strip.
    while sum(q['stratum']==k for q in chosen)<G6_TARGETS[k]:
        possibilities=[]
        for i,row in opts.iterrows():
            if i in selected_ids:continue
            pid=row.plot_id
            if strip_n[pid]>=MAX_PER_STRIP_TOTAL:continue
            limit=MAX_PER_STRIP_RARE if k in [1,2,3] else MAX_PER_STRIP_EACH_STRATUM
            if per_pair_n[(pid,k)]>=limit:continue
            if not far_enough(row):continue
            pfirst=int(pid in {q['plot_id'] for q in chosen if q['stratum'] in range(1,8)}) \
                if matched_controls else 0
            # Coverage first; prioritize unused strips for this stratum, and underused UTM zones.
            key=(-pfirst, per_pair_n[(pid,k)],strip_n[pid],
                 zones_by_class[k][row.utm_zone],-row.purity_pct,row.tiebreak)
            possibilities.append((key,i,row))
        if not possibilities:break
        _,i,row=min(possibilities,key=lambda t:t[0])
        d={col:row[col] for col in ['plot_id','utm_zone','stratum',
                                    'longitude','latitude','purity_pct']}
        d['selection_priority']='paired_reference' if matched_controls else 'ecological_gradient'
        chosen.append(d)
        selected_ids.add(i)
        strip_n[row.plot_id]+=1
        per_pair_n[(row.plot_id,k)]+=1
        zones_by_class[k][row.utm_zone]+=1

# Rare recovery first, then disturbance; matched high-canopy reference AFTER all targets.
for k in [1,2,3,4,5,6,7]:
    add_stratum(k)
add_stratum(8,matched_controls=True)
add_stratum(9)
selection=pd.DataFrame(chosen)
assert len(selection) and selection.stratum.isin(range(1,10)).all()
selection['environment']=selection.stratum.map(labels)
selection.insert(0,'circle_id',['WWF_G6_C'+str(i+1).zfill(3) for i in range(len(selection))])
selection['radius_m']=RADIUS_M
selection['rgb_validity']='NOT_CHECKED'
selection['sampling_status']='PROVISIONAL'
selection.to_csv(G6_DIR/'WWF_G6_provisional_125m_circles.csv',index=False)
# Actual metric-radius polygon circles, in the UTM CRS local to each centre.
centres=gpd.GeoDataFrame(selection.copy(),
    geometry=gpd.points_from_xy(selection.longitude,selection.latitude),crs=4326)
polys=[]
for row in centres.itertuples(index=False):
    geom=Point(float(row.longitude),float(row.latitude))
    local=gpd.GeoSeries([geom],crs='EPSG:4326')
    crs=local.estimate_utm_crs()
    poly=local.to_crs(crs).buffer(RADIUS_M,resolution=32).to_crs(4326).iloc[0]
    polys.append(poly)
circle_gdf=gpd.GeoDataFrame(selection.copy(),geometry=polys,crs=4326)
centres.to_file(G6_DIR/'WWF_G6_provisional_centre_points.gpkg',driver='GPKG')
circle_gdf.to_file(G6_DIR/'WWF_G6_provisional_125m_circles.gpkg',driver='GPKG')
# Also GeoJSON for easy import to QGIS, web maps and Earth Engine.
circle_gdf.to_file(G6_DIR/'WWF_G6_provisional_125m_circles.geojson',driver='GeoJSON')

report=[]
for k in labels:
    this=selection[selection.stratum.eq(k)]
    screened=candidates[candidates.stratum.eq(k)]
    report.append(dict(stratum_code=k,environment=labels[k],target=G6_TARGETS[k],
        n_selected=len(this),n_strips_selected=this.plot_id.nunique(),
        n_utm_zones_selected=this.utm_zone.nunique(),screened_candidate_pixels=len(screened),
        strips_with_sampled_candidate_centres=screened.plot_id.nunique(),
        median_purity_pct=this.purity_pct.median() if len(this) else np.nan,
        target_reached=len(this)>=G6_TARGETS[k]))
summary=pd.DataFrame(report)
summary.to_csv(G6_DIR/'WWF_G6_sampling_balance_and_shortfalls.csv',index=False)
print('\n=== G.6 ECOLOGICAL CIRCLE DESIGN ===')
display(summary.round(2))
print('\nSelected',len(selection),'provisional circles in',
      selection.plot_id.nunique(),'WWF strips; minimum centre spacing',MIN_CENTRE_SEPARATION_M,'m')
print('G.6 output folder:',G6_DIR)
print('NOT YET VALIDATED against actual 10 cm RGB. No training labels inferred.')

# Lightweight QA figures: diversity by stratum/UTM zone and global location overview.
temp=Path(tempfile.mkdtemp(prefix='g6_figs_'))
fig,ax=plt.subplots(figsize=(12,4.5))
xx=np.arange(len(summary))
ax.bar(xx-.2,summary.target,width=.4,label='Aspirational target')
ax.bar(xx+.2,summary.n_selected,width=.4,label='Provisional selected')
ax.set_xticks(xx,[str(k) for k in labels])
ax.set(xlabel='Ecological stratum code',ylabel='125 m circles',
       title='G.6 stratified sample: target vs spatially feasible selection')
ax.legend()
fig.tight_layout()
fig.savefig(temp/'G6_sampling_balance.png',dpi=120)
plt.close(fig)
shutil.copyfile(temp/'G6_sampling_balance.png',G6_DIR/'G6_sampling_balance.png')
display(IPImage(filename=str(temp/'G6_sampling_balance.png'),width=920))
fig,ax=plt.subplots(figsize=(12,6))
foot.boundary.plot(ax=ax,color='#b8b8b8',linewidth=.4)
centres.plot(ax=ax,column='stratum',categorical=True,cmap='tab10',
             legend=True,markersize=13)
ax.set(xlabel='Longitude',ylabel='Latitude',
       title='Geographic replication of provisional 125 m Musanga circles')
fig.tight_layout()
fig.savefig(temp/'G6_geographic_distribution.png',dpi=120)
plt.close(fig)
shutil.copyfile(temp/'G6_geographic_distribution.png',G6_DIR/'G6_geographic_distribution.png')
display(IPImage(filename=str(temp/'G6_geographic_distribution.png'),width=920))


### 7.B — Native WWF RGB screening of selected circles

Checks **only the G.6 shortlisted circles**, not all 264 GB of WWF mosaics. Each 125 m window is read in a small, downsampled rasterio array from the original GCS GeoTIFF using existing Colab access. We check coverage by RGB data values (nodata, black and obvious saturated pixels), and produce per-circle preview/contact sheets. Results remain **screening estimates**, not definitive image-valid masks: black shadow pixels may be real and imagery colour varies; a circle must still be visually reviewed before annotation.

Requires the authenticated Colab/GCS access that worked for NB08 RGB reads. Set `G6_RGB_CHECK=False` to skip this optional step; otherwise the GCS reads will only access bounded windows.

In [ ]:
# 7.B.1 — Screen native RGB image quality
# Important: original WWF RGB GeoTIFFs have no internal overviews.
from pathlib import Path
import numpy as np, pandas as pd, rasterio
from rasterio.windows import from_bounds, Window
from rasterio.enums import Resampling
from shapely.geometry import Point
from pyproj import Transformer
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive',force_remount=False)
WORK_DIR=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/V3_1/NB08_WWF_Lidar')
G6_DIR=WORK_DIR/'TMF_strip_descriptors'/'G6_125m_sampling_design'
input_path=G6_DIR/'WWF_G6_provisional_125m_circles.csv'
assert input_path.exists(),'Run G.6A first'
G6_RGB_CHECK=True
G6_RGB_SAMPLE_SIDE=180    # downsampled read for the circle window
G6_RGB_PASS_FRACTION=0.95
if not G6_RGB_CHECK:
    print('RGB validity check skipped.')
else:
    circles=pd.read_csv(input_path)
    output_path=G6_DIR/'WWF_G6_rgb_coverage_screen.csv'
    old=pd.read_csv(output_path).set_index('circle_id').to_dict('index') if output_path.exists() else {}
    checked=[]
    rgb_root='/vsigs/carbon-maps/001_Lidar_Rasters/RGB_mosaic/'
    for n,c in enumerate(circles.itertuples(index=False),1):
        if c.circle_id in old:
            checked.append({'circle_id':c.circle_id,**old[c.circle_id]})
            continue
        uri=rgb_root+str(c.plot_id)+'_RGB.tif'
        try:
            with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
                with rasterio.open(uri) as ds:
                    tr=Transformer.from_crs('EPSG:4326',ds.crs,always_xy=True)
                    x,y=tr.transform(float(c.longitude),float(c.latitude))
                    x0,y0=x-64,y-64
                    x1,y1=x+64,y+64
                    window=from_bounds(x0,y0,x1,y1,transform=ds.transform)
                    a=ds.read(indexes=[1,2,3],window=window,
                        out_shape=(3,G6_RGB_SAMPLE_SIDE,G6_RGB_SAMPLE_SIDE),
                        resampling=Resampling.nearest,boundless=True,fill_value=0)
            yy,xx=np.mgrid[:G6_RGB_SAMPLE_SIDE,:G6_RGB_SAMPLE_SIDE]
            inside=((xx+0.5-G6_RGB_SAMPLE_SIDE/2)**2+
                    (yy+0.5-G6_RGB_SAMPLE_SIDE/2)**2 <=
                    (G6_RGB_SAMPLE_SIDE*(62.5/128))**2)
            black=np.all(a==0,axis=0)
            saturated=np.all(a>=254,axis=0)
            valid=inside & ~black & ~saturated
            fraction=float(valid.sum()/max(inside.sum(),1))
            result=dict(circle_id=c.circle_id,plot_id=c.plot_id,
                        rgb_valid_pixel_fraction_proxy=fraction,
                        rgb_quality_screen=('PASS' if fraction>=G6_RGB_PASS_FRACTION else 'REVIEW'),
                        rgb_error='')
        except Exception as e:
            result=dict(circle_id=c.circle_id,plot_id=c.plot_id,
                        rgb_valid_pixel_fraction_proxy=np.nan,
                        rgb_quality_screen='ERROR',rgb_error=str(e)[:250])
        checked.append(result)
        if n%12==0:
            pd.DataFrame(checked).to_csv(output_path,index=False)
            print('Screened',n,'/',len(circles),'RGB circle windows')
    rgb=pd.DataFrame(checked)
    rgb.to_csv(output_path,index=False)
    print('Saved',output_path)
    display(rgb.groupby('rgb_quality_screen').size().rename('n_circles').to_frame())
    review=rgb[rgb.rgb_quality_screen.ne('PASS')]
    display(review.head(30))
    print('This is a bounded screen only; inspect RGB visually before accepting circles.')


## 8 — Archived exploratory TMF sampling (optional; superseded by section 7)

**Research aim:** expand Musanga labels to secondary and regenerating forest, degraded canopy, and recently cleared landscapes, plus undisturbed reference forest. The WWF RGB imagery is the annotation source; TMF is a **coarse (30 m), contextual sampling stratifier**, not a Musanga label or a claim about individual 125 m² circles. <https://forobs.jrc.ec.europa.eu/TMF/resources/tutorial/gee>

**Temporal control:** The WWF survey imagery was acquired during **2014–2015** (individual plot years not yet distinguished). Sample only TMF states that remain consistent across *both* 2014 and 2015; recently cleared candidates must already be cleared by 2014. Later TMF classes are never used as contemporary labels. "Recently cleared" means transition from forest to deforested land within the preceding three years, as measured in the relevant TMF annual maps. "Regrowth" is the official TMF regrowth state at the acquisition year.

**Confirmed annotation unit:** 125 m **diameter** (radius 62.5 m; area approximately 1.227 ha). Candidate circles and exports use this diameter by default.

**Workflow:** verify dates → sample TMF strata within actual WWF LiDAR footprints → inspect 250 m context + native RGB validity → export selected circles and small georeferenced RGB clips to Drive → hand annotation in GIS → independent label QC. No automatic Musanga annotations or giant mosaic downloads.

In [ ]:
# 8.A — Historical pilot sampling metadata
import math, pandas as pd
from pathlib import Path
TMF_OUT=WORK_DIR/'TMF_guided_annotation'
TMF_OUT.mkdir(parents=True,exist_ok=True)
PILOT_PLOTS=['UTM33S_Plot135','UTM34N_Plot105','UTM34S_Plot60',
             'UTM35N_Plot100','UTM35S_Plot79','UTM36N_Plot104']
CIRCLE_MODE='diameter_m'  # confirmed: 125 m diameter, not 125 m²
CIRCLE_VALUE=125.0
CIRCLE_RADIUS_M=(math.sqrt(CIRCLE_VALUE/math.pi)
                 if CIRCLE_MODE=='area_m2' else CIRCLE_VALUE/2)
print('Circle mode:',CIRCLE_MODE,'radius (m):',round(CIRCLE_RADIUS_M,2),
      'diameter (m):',round(2*CIRCLE_RADIUS_M,2),
      'area (m²):',round(math.pi*CIRCLE_RADIUS_M**2,1))
# User-confirmed WWF campaign dates: 2014–2015, individual plot years unknown.
# H2 will sample only candidate landscape states stable across BOTH years.
WWF_RGB_ACQUISITION_WINDOW=(2014,2015)
WWF_RGB_ACQUISITION_YEAR={p:None for p in PILOT_PLOTS}
print('Campaign period:',WWF_RGB_ACQUISITION_WINDOW,
      '| individual plot years remain unverified; use stable-class sampling.')
# List potentially useful bucket reports and metadata filenames (metadata requests only)
helpful=df[df.path.str.contains(r'00-raw-data/Reports/|\.xml$|\.pdf$',case=False,regex=True)]
display(helpful[['path','size_GB']].head(55))
print('Note: filename/2016 national products do NOT establish individual RGB flight year.')

### 8.B — Historical acquisition-date metadata
Inspect RGB GeoTIFF tags and list associated metadata/reports. **Do not mistake a file update, TIFF processing timestamp or 2016 map product date for an aerial acquisition date.** This is evidence gathering: candidate years must be verified before H.2.

In [ ]:
# 8.B.1 — Inspect WWF flight-date metadata
import re, json, pandas as pd, rasterio
from pathlib import PurePosixPath
from IPython.display import display
# Recognize acquisition-specific metadata fields; generic TIFF DateTime is often processing time.
date_key=re.compile(r'acqui|capture|flight|survey|sensing|exposure|gps.*date|date.*gps',re.I)
year_re=re.compile(r'(?<!\d)(19\d{2}|20\d{2})(?!\d)')
date_rows=[]
for plot_id in PILOT_PLOTS:
    rgb=f'001_Lidar_Rasters/RGB_mosaic/{plot_id}_RGB.tif'
    hits=[]
    try:
        with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
            with rasterio.open('/vsigs/'+BUCKET+'/'+rgb) as ds:
                namespaces=[''] + [k for k in ds.tag_namespaces() if k]
                for ns in dict.fromkeys(namespaces):
                    try:
                        tags=ds.tags(ns=ns) if ns else ds.tags()
                    except Exception:
                        continue
                    for key,value in tags.items():
                        if date_key.search(key):
                            hits.append(dict(source='geotiff_tag',key=f'{ns}:{key}',value=str(value)[:300]))
                        elif re.search(r'datetime|software',key,re.I):
                            hits.append(dict(source='processing_tag_UNVERIFIED',key=f'{ns}:{key}',value=str(value)[:300]))
    except Exception as exc:
        hits.append(dict(source='ERROR',key='read',value=repr(exc)[:300]))
    # Candidate supporting filenames from source raw plot and report directories
    raw_files=df[df.path.str.contains(re.escape(plot_id.split('_')[0]),regex=True,na=False) &
                 df.path.str.contains(r'\.xml$|\.pdf$|\.txt$|\.csv$',case=False,regex=True,na=False)]
    for filename in raw_files.path.head(10):
        hits.append(dict(source='supporting_filename_ONLY',key='object',value=filename))
    if not hits:
        hits=[dict(source='NO_ACQUISITION_DATE_FOUND',key='',value='')]
    for hit in hits:
        date_rows.append(dict(plot_id=plot_id,**hit))
date_evidence=pd.DataFrame(date_rows)
display(date_evidence)
date_evidence.to_csv(TMF_OUT/'wwf_acquisition_date_evidence.csv',index=False)
# Reports inventory is small; inspect original survey reports if tags don't contain actual flight dates.
reports=df[df.path.str.startswith('00-raw-data/Reports/') & df.path.str.endswith('.pdf')]
print('\nSurvey reports (potential flight dates):')
display(reports[['path','size_GB']].head(70))
print('Evidence exported:',TMF_OUT/'wwf_acquisition_date_evidence.csv')
print('Fill WWF_RGB_ACQUISITION_YEAR in H1 only when a genuine flight acquisition year is established.')


In [ ]:
# 8.C — Superseded exploratory TMF circle sampling
# Only sites with matching TMF annual state in Dec2014 and Dec2015 qualify.
# 'Recently cleared' must already be deforested in 2014 and 2015,
# having been forest in 2011 or 2012. No post-2015 forest conditions used.
import ee, geopandas as gpd, numpy as np, pandas as pd, rasterio
from shapely.geometry import Point, mapping
from shapely.ops import transform as shape_transform
from pyproj import Transformer
assert tuple(WWF_RGB_ACQUISITION_WINDOW)==(2014,2015)
# Self-healing setup: GCS authentication alone does NOT initialize Earth Engine.
EE_PROJECT=(globals().get('EE_PROJECT') or 'musanga-511110').strip()
assert EE_PROJECT, 'EE_PROJECT must not be empty'
print('GEE project:',repr(EE_PROJECT))
try:
    ee.Initialize(project=EE_PROJECT)
    print('Earth Engine connected:',EE_PROJECT,'| test:',ee.Number(1).getInfo())
except Exception as first_error:
    print('Earth Engine setup needed:',str(first_error)[:300])
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
    print('Earth Engine authenticated:',EE_PROJECT,'| test:',ee.Number(1).getInfo())
tmf=ee.ImageCollection('projects/JRC/TMF/v1_2025/AnnualChanges').mosaic()
annual14=tmf.select('Dec2014')
annual15=tmf.select('Dec2015')
forest11=tmf.select('Dec2011').eq(1).Or(tmf.select('Dec2011').eq(2))
forest12=tmf.select('Dec2012').eq(1).Or(tmf.select('Dec2012').eq(2))
# Stable class 1,2,4; class 3 only if recently deforested before image capture.
recent_clear=annual14.eq(3).And(annual15.eq(3)).And(forest11.Or(forest12))
stable=annual14.eq(annual15)
stratum=ee.Image.constant(0).where(stable.And(annual14.eq(1)),1)
stratum=stratum.where(stable.And(annual14.eq(2)),2)
stratum=stratum.where(recent_clear,3)
stratum=stratum.where(stable.And(annual14.eq(4)),4).rename('stratum').toByte()
# One 30 m TMF pixel is 900 m²: candidate forest category is broad spatial context,
# NOT a verified label for the much smaller annotation circle.
SAMPLES_PER_STRATUM_PER_PLOT=5
KEEP_ONLY_IF_WITHIN_PLOT=True
rows=[]
crs_by_plot={}
for plot_id in PILOT_PLOTS:
    plotrow=plots.loc[plots.plot_id==plot_id]
    assert len(plotrow)==1, plot_id
    geom=plotrow.geometry.iloc[0]
    region=ee.Geometry(mapping(geom),proj='EPSG:4326',geodesic=False)
    fc=stratum.updateMask(stratum.gt(0)).stratifiedSample(
        numPoints=SAMPLES_PER_STRATUM_PER_PLOT,classBand='stratum',
        classValues=[1,2,3,4],
        classPoints=[SAMPLES_PER_STRATUM_PER_PLOT]*4,
        region=region,scale=30,geometries=True,seed=8108,
        dropNulls=True,tileScale=4)
    features=fc.getInfo()['features']
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
        with rasterio.open('/vsigs/'+BUCKET+
             f'/001_Lidar_Rasters/RGB_mosaic/{plot_id}_RGB.tif') as src:
            assert src.crs and src.crs.is_projected and src.crs.linear_units=='metre'
            crs_by_plot[plot_id]=src.crs
    for f in features:
        lon,lat=f['geometry']['coordinates']
        cls=int(f['properties']['stratum'])
        rows.append(dict(plot_id=plot_id,TMF_years='2014;2015',
                         time_policy='stable_in_both_2014_and_2015',
                         tmf_stratum=cls,
                         stratum_label={1:'undisturbed',2:'degraded',
                           3:'cleared_by_2014_after_2011_2012_forest',
                           4:'regrowth'}[cls],
                         longitude=float(lon),latitude=float(lat)))
    print(plot_id,':',len(features),'TMF candidate centres')
assert rows,'No eligible TMF candidate centres found'
table=pd.DataFrame(rows)
proposals=[]
for i,r in enumerate(table.itertuples(index=False),1):
    local_crs=crs_by_plot[r.plot_id]
    proj=Transformer.from_crs('EPSG:4326',local_crs,always_xy=True)
    x,y=proj.transform(r.longitude,r.latitude)
    circle=Point(x,y).buffer(CIRCLE_RADIUS_M,resolution=32)
    footprint=gpd.GeoSeries(
        [plots.loc[plots.plot_id==r.plot_id,'geometry'].iloc[0]],
        crs='EPSG:4326').to_crs(local_crs).iloc[0]
    if not footprint.covers(circle):continue
    rev=Transformer.from_crs(local_crs,'EPSG:4326',always_xy=True)
    proposals.append(dict(candidate_id=f'WWF_{i:04d}',plot_id=r.plot_id,
        TMF_years=r.TMF_years,time_policy=r.time_policy,
        tmf_stratum=r.tmf_stratum,stratum_label=r.stratum_label,
        geometry=shape_transform(rev.transform,circle)))
assert proposals,'Candidates exist but circles did not fit in mapped WWF footprints'
gdf=gpd.GeoDataFrame(proposals,geometry='geometry',crs='EPSG:4326')
gdf['rgb_validated']=False
gdf['visually_approved']=False
gdf['musanga_annotation_status']='NOT_ANNOTATED'
gdf.to_file(TMF_OUT/'wwf_tmf_candidate_circles.gpkg',driver='GPKG')
gdf.drop(columns='geometry').to_csv(TMF_OUT/'wwf_tmf_candidate_circles.csv',index=False)
print('Proposed (not visually approved) circles:',len(gdf))
display(gdf.groupby(['plot_id','stratum_label']).size().rename('n').reset_index())
print('TMF classes must be confirmed visually in 2014–2015 RGB.')


In [ ]:
# 8.D — Archived RGB context export
# Select at most one candidate per plot × TMF class initially, no bulk download.
# Images contain full 250 m context; annotation circle stays much smaller.
import rasterio, numpy as np, geopandas as gpd, pandas as pd
from rasterio.windows import from_bounds, Window
from rasterio.windows import transform as window_transform
from rasterio.enums import Resampling
from pathlib import Path
CHIP_CONTEXT_M=250
MAX_CHIPS=12
RUN_CHIP_EXPORT=False  # change to True after inspecting candidate table
CHIP_DIR=TMF_OUT/'rgb_native_context_chips'
CHIP_DIR.mkdir(parents=True,exist_ok=True)
if RUN_CHIP_EXPORT:
    selection=gdf.sort_values(['plot_id','tmf_stratum','candidate_id']).groupby(
        ['plot_id','tmf_stratum']).head(1).head(MAX_CHIPS)
    chip_logs=[]
    for row in selection.itertuples(index=False):
        uri='/vsigs/'+BUCKET+f'/001_Lidar_Rasters/RGB_mosaic/{row.plot_id}_RGB.tif'
        try:
            with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
                with rasterio.open(uri) as src:
                    circle=gpd.GeoSeries([row.geometry],crs='EPSG:4326').to_crs(src.crs).iloc[0]
                    x,y=circle.centroid.x,circle.centroid.y
                    half=CHIP_CONTEXT_M/2
                    win=from_bounds(x-half,y-half,x+half,y+half,src.transform)
                    win=win.round_offsets().round_lengths()
                    full=Window(0,0,src.width,src.height)
                    win=win.intersection(full)
                    a=src.read(indexes=[1,2,3],window=win,masked=True)
                    data=np.ma.filled(a,0)
                    valid=(~np.ma.getmaskarray(a).any(axis=0)) & np.any(data!=0,axis=0)
                    fraction=float(valid.mean())
                    if fraction<.70:
                        print('SKIP low-validity',row.candidate_id,round(fraction,3))
                        continue
                    profile=src.profile.copy()
                    profile.update(driver='GTiff',count=3,width=int(win.width),
                                   height=int(win.height),transform=window_transform(win,src.transform),
                                   tiled=True,compress='deflate',BIGTIFF='IF_SAFER')
                    for key in ['photometric','interleave']:
                        profile.pop(key,None)
                    dst=CHIP_DIR/f'{row.candidate_id}_{row.plot_id}_{row.stratum_label}.tif'
                    with rasterio.open(dst,'w',**profile) as sink:
                        sink.write(data)
                        sink.colorinterp=src.colorinterp[:3]
                    chip_logs.append(dict(candidate_id=row.candidate_id,
                                          plot_id=row.plot_id,context_m=CHIP_CONTEXT_M,
                                          native_gsd_cm=100*abs(src.res[0]),
                                          valid_fraction=fraction,rgb_tif=str(dst)))
                    print('Saved:',dst.name,'| valid',round(fraction,3))
        except Exception as ex:
            print('FAILED:',row.candidate_id,repr(ex))
    pd.DataFrame(chip_logs).to_csv(TMF_OUT/'wwf_context_chip_exports.csv',index=False)
    print('Use GeoPackage circles as separate QGIS overlay. Do NOT confuse TMF stratum with a field label.')
else:
    print('Export OFF. Review H2 candidates, then set RUN_CHIP_EXPORT=True.')
print('Working directory:',TMF_OUT)

### 8.E — Exhaustive annotation quality controls

Load `wwf_tmf_candidate_circles.gpkg` and corresponding **georeferenced** RGB chips into QGIS. First check the imagery acquisition year, validity, and apparent disturbance/regrowth stage; reject mismatched areas. Annotate all visible Musanga crowns inside each approved circle and explicitly record true absence as zero. Keep uncertainty labels for ambiguous young crowns; do **not** treat unknown pixels as negative. Record annotator, date, annotation confidence, tree/crown counts, and image quality. For a robust cross-forest validation exercise, group spatially nearby circles by parent WWF plot to avoid leakage.

Sampling reliability: TMF pixels are ~30 m, coarser than the crown-scale detail within a **125 m diameter** circle. TMF determines the surrounding landscape class, not the circle's actual vegetation. Candidate status is never ground truth. Crop valid coverage and human photo interpretation must confirm suitability before annotation.

**Campaign date 2014–2015:** H2 uses only TMF candidate states stable across these two years. Verify per-plot dates later for finer transition sampling.

### 8.F — Optional legacy Earth Engine circle exports

Publish a deterministic 12-circle pilot (three per TMF category, diverse WWF plots) or all 90 circles into `musanga-511110`. Two **separate** table assets are created: the **125 m diameter exhaustive annotation circles** and the **225 m diameter context boundaries**, representing an additional 50 m buffer. Circles are buffered in their own metric UTM CRS before conversion to WGS84. Assets preserve circle identifiers, site, TMF sampling category, and unreviewed annotation status. This operation **does not upload WWF RGB imagery or create editable annotations**; imagery must be separately ingested as EE image assets (subject to redistribution permissions), and annotations should be saved in a new dedicated vector layer.

The cell starts Earth Engine export tasks. Wait for both tasks to complete before using their IDs in the <https://code.earthengine.google.com/> Code Editor. Share the table assets with collaborators using Earth Engine Assets permissions.

In [ ]:
# 8.F.1 — Publish historical pilot candidates (optional)
# This creates TWO table assets: 125 m diameter cores and 225 m context circles.
# Default: same 12 reproducible examples as H.2B (3 per TMF class).
# To publish all candidate circles, change EXPORT_SCOPE = 'all_90'.
import re, ee, pandas as pd, geopandas as gpd
from shapely.geometry import mapping

EE_PROJECT = (globals().get('EE_PROJECT') or 'musanga-511110').strip()
ee.Initialize(project=EE_PROJECT)

EXPORT_SCOPE = 'pilot_12'     # 'pilot_12' or 'all_90'
N_PER_CLASS = 3
SELECTION_SEED = 42
OUTER_BUFFER_M = 50.0        # additional 50 m outside the 62.5 m core
SUBMIT_EXPORT_TASKS = True   # set False to preview only

if EXPORT_SCOPE == 'pilot_12':
    labels = ['undisturbed','degraded',
              'cleared_by_2014_after_2011_2012_forest','regrowth']
    selections=[]
    for label in labels:
        sub=gdf[gdf['stratum_label'].eq(label)].sample(frac=1,random_state=SELECTION_SEED)
        distinct=sub.drop_duplicates('plot_id').head(N_PER_CLASS)
        remainder=sub.loc[~sub.index.isin(distinct.index)].head(N_PER_CLASS-len(distinct))
        selections.append(pd.concat([distinct,remainder]))
    export_gdf=gpd.GeoDataFrame(pd.concat(selections,ignore_index=True),
                                geometry='geometry',crs=gdf.crs)
elif EXPORT_SCOPE == 'all_90':
    export_gdf=gdf.copy()
else:
    raise ValueError('EXPORT_SCOPE must be pilot_12 or all_90')

assert len(export_gdf)>0
assert export_gdf.candidate_id.is_unique, 'Duplicate candidate IDs'
assert export_gdf.crs.to_epsg()==4326, 'Expected WGS84 original circles'
# Fail if H1 was accidentally run in 125 m² mode.
assert abs(CIRCLE_RADIUS_M-62.5)<0.01, 'Expected 125 m DIAMETER (radius 62.5 m)'

def plot_utm_epsg(plot_id):
    found=re.match(r'^UTM(\d{1,2})([NS])_',str(plot_id))
    if not found:raise ValueError('Unexpected plot UTM zone: '+str(plot_id))
    zone=int(found.group(1))
    return (32600 if found.group(2)=='N' else 32700)+zone

core_features=[]
context_features=[]
for _,row in export_gdf.iterrows():
    # Geometries are made in a local metric CRS (one of several WWF UTM zones).
    utm=plot_utm_epsg(row['plot_id'])
    circle=gpd.GeoSeries([row.geometry],crs='EPSG:4326').to_crs(epsg=utm).iloc[0]
    expanded=circle.buffer(OUTER_BUFFER_M,resolution=48)
    center=circle.centroid
    assert abs(circle.area - 3.141592653589793*62.5**2)<35, (
        'Core does not have the expected 125 m diameter: '+str(row['candidate_id']))
    geom_core=gpd.GeoSeries([circle],crs=f'EPSG:{utm}').to_crs(4326).iloc[0]
    geom_context=gpd.GeoSeries([expanded],crs=f'EPSG:{utm}').to_crs(4326).iloc[0]
    attrs={
        'circle_id':str(row['candidate_id']),
        'plot_id':str(row['plot_id']),
        'tmf_class':str(row['stratum_label']),
        'tmf_code':int(row['tmf_stratum']),
        'image_years':'2014-2015',
        'sampling':'TMF_2014_2015_stable',
        'review':'NOT_REVIEWED',
        'annotation':'NOT_ANNOTATED',
        'core_diam_m':125,
        'context_diam_m':225,
    }
    core_features.append(ee.Feature(
        ee.Geometry(mapping(geom_core),geodesic=False),
        {**attrs,'boundary':'core_125m'}))
    context_features.append(ee.Feature(
        ee.Geometry(mapping(geom_context),geodesic=False),
        {**attrs,'boundary':'context_225m'}))

core_fc=ee.FeatureCollection(core_features)
context_fc=ee.FeatureCollection(context_features)
suffix='pilot12' if EXPORT_SCOPE=='pilot_12' else 'all90'
core_asset=f'projects/{EE_PROJECT}/assets/WWF_TMF_2014_2015_{suffix}_core_125m'
context_asset=f'projects/{EE_PROJECT}/assets/WWF_TMF_2014_2015_{suffix}_context_225m'

print('Circles selected:',len(export_gdf))
display(export_gdf[['candidate_id','plot_id','stratum_label']])
print('Core asset:',core_asset)
print('Context asset:',context_asset)

# Preserve the same candidates locally for annotation logistics.
export_gdf.to_file(
    TMF_OUT/f'WWF_TMF_2014_2015_{suffix}_core125m.gpkg',driver='GPKG')

if SUBMIT_EXPORT_TASKS:
    for title,fc,asset in [
        ('core125m',core_fc,core_asset),
        ('context225m',context_fc,context_asset),
    ]:
        try:
            existing=ee.data.getAsset(asset)
        except ee.EEException as err:
            if 'not found' in str(err).lower() or '404' in str(err):
                existing=None
            else:
                raise
        if existing is not None:
            print('ALREADY EXISTS (not overwritten):',asset)
            continue
        task=ee.batch.Export.table.toAsset(
            collection=fc,
            description=f'WWF_TMF_{suffix}_{title}',
            assetId=asset)
        task.start()
        print('Submitted export:',title,'task ID:',task.id)
    print('Check the Earth Engine Tasks tab. An export is not immediately available.')
else:
    print('Dry run only; set SUBMIT_EXPORT_TASKS=True to create EE assets.')


## 9 — Final sampling report and RGB illustration atlas

**Run 9.A once the section 7.A and 7.B results exist.** A Google Drive mount alone does not authenticate native WWF GCS access. This block obtains a GCS OAuth token (and prompts Colab login if required). It now aborts rather than emitting blank-image reports if any 10 cm RGB preview fails. This is the endpoint of the WWF forest-history sampling workflow. The historical exploratory H cells are preserved for reproducibility but are **not needed** to make the final report.

9.A reads saved results from sections 4, 5, 6 and 7, and makes a landscape PDF with:

- A short objective/data/methods/results summary with actual pooled ecological statistics.
- A table of G.6 target vs **actual selected circles**, number of independent WWF strips, median TMF/Hansen category purity and RGB screen outcomes.
- One full page per ecological class, **codes 1–10**, with up to nine **distinct actual selected 125 m circles** in a 3×3 grid, chosen to maximise different strips. Each native WWF RGB image shows a 125 m red ring, circle ID, plot, class and estimated purity.
- Honest sampling gaps: fewer than nine selected examples means an annotated empty panel. **Class 10 was deliberately not selected in G.6**, so its page explicitly reports zero examples rather than inventing nine.
- All example images and page montage PNGs saved separately for use in presentations.

Images are read **directly from the WWF GCS bucket via bounded `/vsigs/` raster windows**. The report does not download the mosaics, upload to GEE or require a new GCS bucket. It may require reauthenticating Colab as in NB08 A. The PDF itself is portable and shareable, subject to the WWF imagery licence. **RGB automated PASS ≠ human image-quality acceptance**.

**Interpretation:** 114 was the *sum of manually specified aspirational target counts*, not an independently optimized sample size. G.6 uses greedy rare-first stratified selection; it enforces 70% target-neighbourhood purity, 62.5 m interior clearance, 250 m centre spacing, max five circles per strip and limited per-stratum replication. Actual achieved class counts and number of strips must be read from G.6 output.

In [ ]:
# 9.A — Generate final PDF report with 3×3 WWF RGB examples
# Standalone (new Colab runtime OK after mounting Drive and authenticating GCS).
# One PDF: scientific summary + actual sampling balance + 10 class-specific pages.
import json, io, math, hashlib, textwrap
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
from pyproj import Transformer
from PIL import Image, ImageDraw, ImageFont, ImageOps
# Install only the lightweight PDF writer if Colab does not already have it.
import importlib.util, subprocess, sys
if importlib.util.find_spec('reportlab') is None:
    subprocess.check_call([sys.executable,'-m','pip','-q','install','reportlab'])
from reportlab.pdfgen import canvas as pdfcanvas
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib.colors import HexColor, Color
from reportlab.lib.utils import ImageReader
from IPython.display import display
from google.colab import drive, auth
import google.auth
from google.auth.transport.requests import Request
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive', force_remount=False)
WORK_DIR=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/V3_1/NB08_WWF_Lidar')
ROOT=WORK_DIR/'TMF_strip_descriptors'
G3=ROOT/'rgb_footprint_pixels'
G4=ROOT/'hansen_tmf_2014'
G5=ROOT/'integrated_ecological_gradient_2014'
G6=ROOT/'G6_125m_sampling_design'
OUT=G6/'FINAL_report'
PREV=OUT/'RGB_125m_examples'
for x in (OUT,PREV):x.mkdir(parents=True,exist_ok=True)

circle_file=G6/'WWF_G6_provisional_125m_circles.csv'
balance_file=G6/'WWF_G6_sampling_balance_and_shortfalls.csv'
global_file=G5/'WWF_G5_global_integrated_strata.csv'
coverage_file=G5/'WWF_G5_125m_circle_candidate_centre_counts.csv'
rgb_file=G6/'WWF_G6_rgb_coverage_screen.csv'
for p in (circle_file,balance_file,global_file,coverage_file):
    assert p.exists(), f'Missing prerequisite results: {p}'
circles=pd.read_csv(circle_file)
balance=pd.read_csv(balance_file)
global_stats=pd.read_csv(global_file)
coverage=pd.read_csv(coverage_file)
rgb=pd.read_csv(rgb_file) if rgb_file.exists() else None
assert circles.circle_id.is_unique and len(circles)>0
assert set(circles.stratum.unique()).issubset(set(range(1,10)))
assert len(global_stats)==10

LABELS={
1:'Young TMF regrowth (1–3 years)',
2:'Intermediate regrowth (4–7 years)',
3:'Longer TMF regrowth (8+ years)',
4:'Degraded forest without Hansen loss',
5:'Degraded forest with Hansen loss',
6:'Recent Hansen loss (2011–2014)',
7:'Historical Hansen loss (2001–2010)',
8:'Stable high-canopy forest reference',
9:'Stable lower-canopy forest reference',
10:'Other / ambiguous environments — not sampled'
}
# Optional development speed: render 1–2 classes at first, then use None for all 10.
REPORT_TEST_CLASSES=None
REPORT_RGB_CROP_WIDTH_M=170       # 125 m circle + ~22.5 m surroundings on each side
REPORT_CHIP_PIXELS=480
REPORT_REBUILD_RGB=False
REPORT_MAX_EXAMPLES_PER_CLASS=9
REPORT_PASS_STRICT=False          # False means include screened circles with clear labels.

byclass={}
for k in range(1,11):
    group=circles[circles.stratum==k].copy() if k!=10 else circles.iloc[:0].copy()
    # One representative per distinct WWF strip first, then remaining centres.
    group=group.sort_values(['purity_pct','circle_id'],ascending=[False,True])
    unique=group.drop_duplicates('plot_id')
    extras=group.loc[~group.circle_id.isin(unique.circle_id)]
    byclass[k]=pd.concat([unique,extras]).head(REPORT_MAX_EXAMPLES_PER_CLASS)
classes_to_render=list(range(1,11)) if REPORT_TEST_CLASSES is None else list(REPORT_TEST_CLASSES)
if rgb is not None:
    counts=rgb.rgb_quality_screen.value_counts()
    print('G.6B RGB screening results:',counts.to_dict())
    if REPORT_PASS_STRICT:
        good=set(rgb.loc[rgb.rgb_quality_screen=='PASS','circle_id'])
        for k in byclass:byclass[k]=byclass[k][byclass[k].circle_id.isin(good)]
print('Actual selected circles:',len(circles),'across',circles.plot_id.nunique(),'WWF strips')
display(balance[['stratum_code','target','n_selected','n_strips_selected',
                 'median_purity_pct','target_reached']].round(2))

# Restore access to WWF's private GCS bucket independently of the Drive mount.
# Re-use the current Colab Google identity; do not change bucket IAM or make it public.
# Explicitly supply an OAuth token to GDAL /vsigs, rather than relying on its
# external auth-code flow (which fails with 'Fetching OAuth2 access code...').
def gcs_auth_token():
    try:
        cred, _ = google.auth.default(
            scopes=['https://www.googleapis.com/auth/devstorage.read_only'])
        if not cred.valid or not cred.token:
            cred.refresh(Request())
        if not cred.token:
            raise RuntimeError('No OAuth access token obtained')
        return str(cred.token)
    except Exception as err:
        print('GCS OAuth not available; launching Colab Google authentication.')
        auth.authenticate_user()
        cred, _ = google.auth.default(
            scopes=['https://www.googleapis.com/auth/devstorage.read_only'])
        if not cred.valid or not cred.token:
            cred.refresh(Request())
        if not cred.token:
            raise RuntimeError('Unable to obtain GCS OAuth token') from err
        return str(cred.token)
GCS_ACCESS_TOKEN=gcs_auth_token()
print('Google Cloud read credentials available (token not displayed).')

# Cached real RGB previews, one per genuinely selected circle.
# PIL cropping uses bounded native GeoTIFF reads and 480x480 output.
def rgb_thumbnail(row):
    ident=str(row.circle_id)
    dest=PREV/(ident+'_WWF_RGB.png')
    if dest.exists() and not REPORT_REBUILD_RGB:
        return dest, None
    path='/vsigs/carbon-maps/001_Lidar_Rasters/RGB_mosaic/'+str(row.plot_id)+'_RGB.tif'
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR',
                       GS_OAUTH2_ACCESS_TOKEN=GCS_ACCESS_TOKEN):
        with rasterio.open(path) as src:
            assert src.count>=3, f'{row.plot_id}: not three-band RGB'
            x,y=Transformer.from_crs('EPSG:4326',src.crs,always_xy=True).transform(
                float(row.longitude),float(row.latitude))
            h=REPORT_RGB_CROP_WIDTH_M/2
            window=from_bounds(x-h,y-h,x+h,y+h,src.transform)
            raw=src.read(indexes=(1,2,3),window=window,boundless=True,fill_value=0,
                out_shape=(3,REPORT_CHIP_PIXELS,REPORT_CHIP_PIXELS),
                resampling=Resampling.bilinear)
    rgbarr=raw.astype(np.float32)
    good=np.isfinite(rgbarr).all(axis=0)&(~(rgbarr==0).all(axis=0))
    if good.sum()<200:
        raise ValueError(f'{ident}: thumbnail has almost no valid RGB coverage')
    if np.issubdtype(raw.dtype,np.integer) and raw.dtype.itemsize==1:
        rgb8=raw.transpose(1,2,0).astype(np.uint8)
    else:
        lo=np.percentile(rgbarr[:,good],1)
        hi=np.percentile(rgbarr[:,good],99)
        rgb8=np.clip((rgbarr-lo)/max(hi-lo,1e-6)*255,0,255).astype(np.uint8).transpose(1,2,0)
    rgb8[~good]=0
    image=Image.fromarray(rgb8,mode='RGB')
    draw=ImageDraw.Draw(image)
    R=(62.5/REPORT_RGB_CROP_WIDTH_M)*REPORT_CHIP_PIXELS
    M=REPORT_CHIP_PIXELS/2
    draw.ellipse((M-R,M-R,M+R,M+R),outline=(255,75,45),width=5)
    draw.line((M-7,M,M+7,M),fill=(255,255,255),width=2)
    draw.line((M,M-7,M,M+7),fill=(255,255,255),width=2)
    image.save(dest,optimize=True)
    return dest,None

image_map={}
failures=[]
for k in classes_to_render:
    if k==10:continue
    for row in byclass[k].itertuples(index=False):
        try:
            p,_=rgb_thumbnail(row)
            image_map[str(row.circle_id)]=p
        except Exception as error:
            failures.append(dict(circle_id=row.circle_id,plot_id=row.plot_id,
                                 error=str(error)[:250]))
print('RGB example images ready:',len(image_map),'| failed reads:',len(failures))
if failures:
    pd.DataFrame(failures).to_csv(OUT/'RGB_report_read_errors.csv',index=False)
    display(pd.DataFrame(failures).head())
if failures:
    raise RuntimeError(
        f'RGB atlas ABORTED: {len(failures)} of {len(image_map)+len(failures)} '
        'native WWF crops could not be read. No replacement PDF was produced. '
        'Check Colab GCS authentication, then rerun section 9.A. '
        'Previously cached good previews are retained.')
if len(image_map)!=sum(len(byclass[k]) for k in classes_to_render if k!=10):
    raise RuntimeError('Incomplete atlas image set: refusing to create empty-image PDF.')

# Sample accounting: actual selection, not target count.
s=circles.groupby('stratum').agg(
    actual_circles=('circle_id','count'),
    independent_strips=('plot_id','nunique'),
    utm_zones=('utm_zone','nunique'),
    median_purity=('purity_pct','median')).reindex(range(1,11),fill_value=0)
s['stratum']=s.index.map(LABELS)
targets=balance.set_index('stratum_code')['target'].to_dict()
s['target']=[int(targets.get(k,0)) for k in s.index]
s['visual_examples']=[len(byclass[k]) if k!=10 else 0 for k in s.index]
s.to_csv(OUT/'WWF_final_sampling_allocations.csv',index_label='stratum_code')

# -------- PDF (landscape A4, one 3x3 visual plate per ecological class) ------
W,H=landscape(A4)
pdf=OUT/'WWF_Musanga_2014_forest_history_and_sampling_RGB_atlas.pdf'
c=pdfcanvas.Canvas(str(pdf),pagesize=(W,H),pageCompression=1)
NAVY=HexColor('#173d36');GREY=HexColor('#57645e');ACCENT=HexColor('#ce593c')
def headline(title,subtitle=None):
    c.setFillColor(NAVY);c.rect(0,H-62,W,62,stroke=0,fill=1)
    c.setFillColorRGB(1,1,1)
    c.setFont('Helvetica-Bold',17);c.drawString(30,H-34,title[:85])
    if subtitle:
        c.setFont('Helvetica',9);c.drawString(31,H-50,subtitle[:120])
def paragraph(text,x,y,width=110,size=10,leading=14,bold=False):
    c.setFillColor(GREY if not bold else NAVY)
    c.setFont('Helvetica-Bold' if bold else 'Helvetica',size)
    for line in textwrap.wrap(str(text),width=width):
        c.drawString(x,y,line);y-=leading
    return y
def section(name,y):
    c.setFont('Helvetica-Bold',12);c.setFillColor(NAVY)
    c.drawString(31,y,name);return y-21
def footer(num):
    c.setStrokeColor(HexColor('#d4dfd8'));c.line(30,34,W-30,34)
    c.setFillColor(GREY);c.setFont('Helvetica',8)
    c.drawString(30,22,'WWF RGB / TMF–Hansen | exploratory ecological sampling | 2014 reference')
    c.drawRightString(W-30,22,f'Page {num}')
page=1
headline('WWF / Musanga — Forest history and sampling atlas',
         '216 WWF RGB survey strips | 10 cm RGB imagery | 30 m TMF × Hansen | reference 2014')
y=H-93
y=section('Objective',y)
y=paragraph('Assess which forest disturbance and recovery environments are represented by the WWF 2014–2015 RGB orthomosaics, and select independent 125 m circles for future exhaustive Musanga crown annotation.',31,y,115)
y-=8;y=section('Datasets and boundaries',y)
for t in [
    'WWF aerial orthomosaics: 10 cm native RGB, 216 strips, acquisition 2014–2015.',
    'JRC Tropical Moist Forest: annual 1990–2014 status and regrowth/degradation history.',
    'Hansen Global Forest Change: 2000 canopy cover, 2001–2014 forest loss, 2000–2012 gain.',
    'Analytical footprint = WWF survey polygon intersection with RGB raster bounds; true valid RGB extent is screened separately.'
]:
    y=paragraph('• '+t,31,y,115,size=9,leading=12);y-=4
y-=7;y=section('Completed findings',y)
pooled=float(global_stats.pixels.sum())
reg=global_stats[global_stats.code.isin([1,2,3])]
stable=float(global_stats.loc[global_stats.code==8,'pct_of_valid_pixels'].iloc[0])
y=paragraph(f'{pooled:,.0f} valid TMF–Hansen pixel observations. High-canopy undisturbed reference: {stable:.2f}%; all three regrowth-duration categories combined: {reg.pixels.sum()/pooled*100:.2f}%. Forest degradation is much better replicated than classified regrowth.',31,y,112,size=9)
y-=7;y=section('Sampling design and validation',y)
y=paragraph(f'G.6 target total = {sum(targets.values())} circles (chosen allocation, NOT an optimized sample size). Actual selected = {len(circles)} circles across {circles.plot_id.nunique()} WWF strips. Minimum selected centre spacing: 250 m; radius: 62.5 m; target-class purity: at least 70%.',31,y,115,size=9)
if rgb is not None:
    passed=int((rgb.rgb_quality_screen=='PASS').sum())
    y-=5;y=paragraph(f'Automated bounded RGB screening: {passed}/{len(rgb)} PASS. This is a preliminary image validity proxy, not human verification of crown interpretability.',31,y,115,size=9)
footer(page);c.showPage();page+=1

headline('Ecological strata — global representation and sampling','TMF regrowth duration is a classification-state duration, NOT measured stand age')
y=H-84
columns=[(32,30),(65,355),(426,68),(500,55),(568,65),(650,65)]
headers=['Code','Integrated ecological environment','Survey %','Target','Selected','WWF strips']
c.setFont('Helvetica-Bold',9);c.setFillColor(NAVY)
for (x,_),hd in zip(columns,headers):c.drawString(x,y,hd)
y-=14
for k in range(1,11):
    stat=global_stats[global_stats.code==k].iloc[0]
    row=s.loc[k]
    vals=[str(k),LABELS[k][:43],f'{stat.pct_of_valid_pixels:.2f}',
          str(int(row.target)),str(int(row.actual_circles)),str(int(row.independent_strips))]
    c.setFont('Helvetica',9);c.setFillColor(GREY)
    for (x,_),val in zip(columns,vals):c.drawString(x,y,val)
    y-=23
y-=13
y=section('Selection criteria and rationale',y)
details=[
    '1. Rare recovery strata (codes 1–3) first; limited to one circle per class per WWF strip.',
    '2. Degradation and historical-loss strata (codes 4–7) sampled across different strips/UTM zones.',
    '3. High-canopy controls (code 8) preferentially paired with a disturbed strip; lower-canopy controls (code 9) added.',
    '4. At least 70% target stratum in 125 m disk; centre >=62.5 m from survey-footprint boundary.',
    '5. Centre-to-centre separation >=250 m; maximum 5 total circles per strip and generally <=2 per class per strip.',
    '6. Candidate list sampled from 30 m eligible pixels, then selected greedily with a fixed seed; not random regional prevalence sampling.',
    '7. Code 10 represents heterogeneous/ambiguous environments and was deliberately excluded from the 114 target.'
]
for item in details:
    y=paragraph(item,31,y,113,size=8.7,leading=11);y-=4
footer(page);c.showPage();page+=1

# 3×3 grid pages: one per category, all source RGB panels individually cached.
from reportlab.lib.utils import ImageReader
for k in classes_to_render:
    subset=byclass[k]
    headline(f'Class {k} — {LABELS[k][:66]}',
        f'{len(circles[circles.stratum==k])} selected circles | {circles.loc[circles.stratum==k,"plot_id"].nunique()} WWF strips | up to 9 distinct RGB examples')
    left=29;gapx=12;gapy=10
    cellw=(W-2*left-2*gapx)/3
    top=H-76
    grid_h=(H-110-2*gapy)/3
    image_px=min(cellw-9,grid_h-27)
    rows=list(subset.itertuples(index=False))
    for j in range(9):
        rr=j//3;cc=j%3
        bx=left+cc*(cellw+gapx);top_y=top-rr*(grid_h+gapy)
        c.setStrokeColor(HexColor('#d1ddd5'));c.roundRect(bx,top_y-grid_h,cellw,grid_h,6,stroke=1,fill=0)
        if j<len(rows):
            q=rows[j];source_img=image_map.get(str(q.circle_id))
            if source_img is not None:
                c.drawImage(ImageReader(str(source_img)),bx+5,top_y-6-image_px,
                            width=image_px,height=image_px,preserveAspectRatio=True)
            else:
                c.setFillColor(HexColor('#f5e9e6'));c.rect(bx+5,top_y-6-image_px,image_px,image_px,fill=1,stroke=0)
                c.setFillColor(ACCENT);c.setFont('Helvetica',9)
                c.drawString(bx+12,top_y-35,'RGB crop unavailable')
            c.setFont('Helvetica-Bold',8);c.setFillColor(NAVY)
            c.drawString(bx+5,top_y-grid_h+17,f'{q.circle_id}  |  {q.plot_id}'[:45])
            c.setFont('Helvetica',7.5);c.setFillColor(GREY)
            c.drawString(bx+5,top_y-grid_h+6,f'Purity {q.purity_pct:.0f}%  ·  125 m circle')
        else:
            c.setFillColor(GREY);c.setFont('Helvetica',9)
            c.drawString(bx+16,top_y-grid_h/2,
                'Not sampled (code 10)' if k==10 else 'No additional selected circle')
    footer(page);c.showPage();page+=1

headline('Interpretation and remaining quality control')
y=H-95
for heading,body in [
    ('What is demonstrated','The WWF image sampling frame includes abundant mature-forest references and widespread forest degradation; TMF-detected regrowth is rare and concentrated. The selected circles provide targeted environmental contrasts but do not estimate regional prevalence.'),
    ('What is not established','Hansen loss is a stand-replacing disturbance indicator, not forest age. TMF regrowth-state duration is not true stand age. Neither TMF nor Hansen identifies Musanga. The 114 circles were chosen by predefined sampling targets and a greedy spatially constrained procedure.'),
    ('Required image validation','Inspect all selected 10 cm crops at native zoom for no-data gaps, blur, colour artefacts and clearly visible crowns before exhaustive annotation. The 114 automated RGB coverage passes are not a manual interpretation of image quality.'),
    ('Next scientific endpoint','Annotate individual Musanga crowns in the accepted circles, quantify per-circle cover, and then test occurrence/abundance across disturbance–recovery environments with site-aware uncertainty and appropriate sampling weights.')
]:
    y=section(heading,y)
    y=paragraph(body,31,y,111,size=9,leading=13);y-=12
footer(page);c.showPage()
c.save()
print('PDF REPORT:',pdf)
print('Pages:',page,'| size MB:',round(pdf.stat().st_size/1e6,2))
print('RGB preview images:',PREV)
print('Actual-sample table:',OUT/'WWF_final_sampling_allocations.csv')
print('EXPLICIT: code 10 has zero selected circles; class 1 may have fewer than 9 examples.')
print('No images were fabricated, duplicated, or sourced from other WWF strips.')
